# NB2 — Simulator Calibration

> **Goal:** Build the discrete-event simulator, calibrate the stochastic occupancy
> parameter θ_OCC against production telemetry, and validate on a held-out window.

> **Paper:** *Diagnosing ML-Driven Queue Systems in Production* — ICDM 2026

> **Note:** Data loading cells use stubs. Replace with your own loader matching the schema below.


## Table of Contents

### Part I — Data Loading
1. [Imports & Setup](#cell-1-imports)
2. [Load Operational Results](#cell-2-load)
3. [Filter QPlanner Calls](#cell-3-filter)
4. [Load Brains Scores](#cell-6-brains)
5. [Data Preparation — Simulation Inputs](#cell-7-dataprep)

### Part II — Simulation Engine
6. [Data Exploration — Visualise Simulation Inputs](#data-exploration)
7. [Simulator Core Classes](#simulator-core)
8. [Objective Function (FO)](#fo-scoring)
9. [Assignment Solver (Hungarian)](#assignment-solver)
10. [Two-Tier Solver](#two-tier-solver)
11. [Main Simulation Loop](#simulation-loop)

### Part III — Score Coverage
12. [Full Score Coverage via MLflow Re-scoring](#full-score-coverage)

### Part IV — Calibration
13. [Calibrated Parameters & Day Setup](#calibrated-parameters)
14. [OCC_SCALE Calibration](#occ-calibration)
15. [Churn Calibration (Parts 1–2)](#churn-calibration)

### Part V — Execution & Validation
16. [21-Day Simulation — Baseline](#run-simulation)
17. [Per-Day Comparison — Simulation vs Historical](#per-day-comparison)
18. [Validation Plots — Aggregated Distributions](#validation-plots)
19. [Churn Calibration Validation (Post-Simulation)](#churn-validation)
20. [Visualisation — Simulation Results (4 Panels)](#sim-results-viz)


<a id="cell-1-imports"></a>
## Cell 1: Imports & Environment Setup

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & ENVIRONMENT SETUP
# ============================================================
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import time
from scipy.optimize import linear_sum_assignment  # Hungarian algorithm


<a id="cell-2-load"></a>
## Cell 2: Load Operational Results from BigQuery

## Data Loading — Replace with your own loader

The original code loads production telemetry from BigQuery via an internal library.
Replace the cell below with your own data loader.

### Expected schema for `df_ops` (operational results)

| Column | Type | Description |
|--------|------|-------------|
| `call_id` | str | Unique call identifier |
| `call_start_ts` | datetime | Call arrival timestamp |
| `operator_id` | str | Assigned operator identifier |
| `wait_time_s` | float | Wait time in seconds at assignment |
| `outcome` | str | `RETAINED` or `CHURNED` |
| `qp_result` | str | `QPlanner-AgentAvailable`, `QPlanner-NoAgentReturned`, `QPlanner-AgentNotAvailable` |
| `assigned_ts` | datetime | Assignment timestamp (NaT if unserved) |

### Expected schema for `df_scores` (call-operator pairings / brains scores)

| Column | Type | Description |
|--------|------|-------------|
| `call_id` | str | Call identifier |
| `operator_id` | str | Operator identifier |
| `churn_ij` | float | Predicted churn probability P(churn | call_i, operator_j) |
| `churn_i` | float | Baseline churn probability P(churn | call_i, null operator) |
| `tick_ts` | datetime | Solver tick timestamp |


In [ ]:
# ============================================================
# DATA LOADING STUB — replace with your own loader
# ============================================================
# Load from CSV/Parquet files. Columns must match the schema
# described in the markdown cell above.
#
# Example:
#   df_ops    = pd.read_parquet('data/operational_results.parquet')
#   df_scores = pd.read_parquet('data/brains_scores.parquet')
#
# The production loader (removed for confidentiality) used:
#   from crm_qp_lib.data_reader import read_op_results, read_brains_data
#   df_ops    = read_op_results(start_date, end_date)
#   df_scores = read_brains_data(start_date, end_date)

DATA_PATH = 'data/'  # set to your data directory
df_ops    = pd.read_parquet(os.path.join(DATA_PATH, 'operational_results.parquet'))
df_scores = pd.read_parquet(os.path.join(DATA_PATH, 'brains_scores.parquet'))


<a id="cell-3-filter"></a>
## Cell 3: Filter to QPlanner-Handled Calls

In [ ]:
# =============================================================================
# CELL 3 — FILTER TO QPLANNER-HANDLED CALLS
# =============================================================================
# INPUT:  df_calls (18,862 rows — all calls in Jan 2026)
# DOES:   - Filters to rows where QUEUE_PLANNER_STATUS_DESC == 'On' (QP was active)
#         - Keeps only QPlanner outcome types: AgentAvailable, NoAgentReturned, AgentNotAvailable
#         - Excludes BAU (non-QPlanner) calls — those were handled by the default telephony system
# OUTPUT: df_calls_qplanner — only calls that went through QPlanner (~12,300 calls)
# =============================================================================

df_calls_qplanner = df_calls[
    (df_calls['QUEUE_PLANNER_STATUS_DESC'] == 'On') 
    & (df_calls['QUEUE_PLANNER_RESULT_DESC'].isin([
        'QPlanner-AgentAvailable',
        'QPlanner-NoAgentReturned',
        'QPlanner-AgentNotAvailable'
    ]))
]
print(f"QPlanner calls: {len(df_calls_qplanner):,} ({len(df_calls_qplanner)/len(df_calls)*100:.1f}% of total)")
df_calls_qplanner.shape

<a id="cell-6-brains"></a>
## Cell 6: Load Brains Scores (Call-Operator Pairings)

In [ ]:
# =============================================================================
# CONSTANTS — Outcome definitions (from Cell 3b in original notebook)
# =============================================================================
VALID_OUTCOMES = ["N RECUPERADO", "RECUPERADO COM ALTERACAO", "RECUPERADO SEM ALTERACAO"]

QP_ACCOUNTABLE_RESULTS = [
    'QPlanner-AgentAvailable',
    'QPlanner-NoAgentReturned',
    'QPlanner-AgentNotAvailable',
    'BAU',
]
print(f"✅ Constants loaded: VALID_OUTCOMES ({len(VALID_OUTCOMES)}), QP_ACCOUNTABLE_RESULTS ({len(QP_ACCOUNTABLE_RESULTS)})")

<a id="cell-7-dataprep"></a>
## Cell 7: Data Preparation — Build Simulation Inputs

In [ ]:
# =============================================================================
# CELL 7 — DATA PREPARATION: Build all simulation inputs
# =============================================================================
# INPUT:  df_calls_qplanner (12,302 QP calls), df_brains (1.56M brains rows)
#
# DOES (9 steps):
#   1. Parse timestamps & sort calls chronologically
#   2. Extract operator pool (unique operators + group)
#   3. Build call events table (one row per unique CALL_ID with arrival, churn_i, duration)
#   4. Build score_lookup dict: (call_id, operator) → churn_ij from FULL brains data
#      Also builds churn_i_lookup: call_id → churn_i
#   5. Compute simulation time window (start → end)
#   6. Define get_day_operators() helper — extracts day-specific operator pool + shifts
#   7. Compute real QPlanner tick cadence from brains request timestamps
#      (time between consecutive request_ids → DATA_TICK_MEDIAN ≈ 3s)
#   8. Compute real pipeline latency from data:
#      overhead = WAIT_TIME_IN_QUEUE (operational) - waiting_time (brains solver)
#      → DATA_PIPELINE_MEAN ≈ 14s, DATA_PIPELINE_STD ≈ 47s
#   9. Total wait-time distribution of unserved calls (INFORMATIONAL ONLY)
#      End-to-end wait (QP queue + FIFO) — NOT used as simulation input
#
# OUTPUT:
#   - df_operators: DataFrame with all unique operators (110)
#   - df_call_events: DataFrame with one row per call (12,302 calls)
#   - score_lookup: dict of (call_id, operator) → churn_ij (~100k+ pairs)
#   - churn_i_lookup: dict of call_id → churn_i
#   - global_median_churn_ij: fallback churn_ij for missing pairs
#   - get_day_operators(): function to extract day-specific ops + shifts
#   - DATA_TICK_MEDIAN, DATA_TICK_MEAN: data-derived tick cadence
#   - DATA_PIPELINE_MEAN, DATA_PIPELINE_STD: data-derived latency params
#   - DATA_MAX_WAIT_SAMPLES: total wait times of unserved calls (informational)
#   - df_calls_qplanner_ts: copy with parsed timestamps (used by later cells)
# =============================================================================

# --- 1. Parse timestamps and sort by arrival time ---
df_sim = df_calls_qplanner.copy()
df_sim['arrival_time'] = pd.to_datetime(df_sim['START_DATE_TIME'])
df_sim = df_sim.sort_values('arrival_time').reset_index(drop=True)

# --- 2. Extract the operator pool (unique operators with their group) ---
operator_cols = ['agent_username', 'agent_groupname']
df_operators = (
    df_sim[df_sim['agent_username'].notna()]
    .drop_duplicates(subset='agent_username')[operator_cols]
    .reset_index(drop=True)
)
print(f"📋 Operator pool: {len(df_operators)} unique operators")

# --- 3. Build call events (one row per unique call arriving at the queue) ---
# Each call needs: call_id, arrival_time, customer_id, churn_i, call_duration
# We pick the first occurrence of each call (some calls may have multiple operator pairings logged)
call_cols = [
    'CALL_ID', 'arrival_time', 'SA_COD', 'churn_i',
    'CALL_DURATION_SEC_QTY', 'WAIT_TIME_IN_QUEUE_SEC_QTY',
    'QUEUE_PLANNER_RESULT_DESC', 'agent_username'
]
df_call_events = (
    df_sim[call_cols]
    .drop_duplicates(subset='CALL_ID', keep='first')
    .reset_index(drop=True)
)
# Fill missing durations with median
median_duration = df_call_events['CALL_DURATION_SEC_QTY'].median()
df_call_events['call_duration'] = df_call_events['CALL_DURATION_SEC_QTY'].fillna(median_duration)
print(f"📞 Total unique calls to simulate: {len(df_call_events)}")
print(f"⏱️  Median call duration: {median_duration:.0f}s")

# --- 4. Build pairing score lookup from BRAINS DATA ---
# df_brains has (call_id, operator) pairings from t_raw_api_brains.
# Each row reflects an operator that was AVAILABLE or IMMINENTLY AVAILABLE
# at a specific historical tick (~8-10 operators per tick).
# Accumulated across all ticks for a call, this gives the "sparse" universe
# of operators the solver actually evaluated for each call.
# We map call_id from brains → CALL_ID used in df_call_events via matching.

# First, build a mapping from brains call_id to df_sim CALL_ID
# (brains uses call_id, df_sim uses CALL_ID — these should match)
brains_call_ids = set(df_brains['call_id'].unique())
sim_call_ids = set(df_call_events['CALL_ID'].unique())
matched_ids = brains_call_ids & sim_call_ids
print(f"🔗 Call ID matching: brains={len(brains_call_ids)}, sim={len(sim_call_ids)}, matched={len(matched_ids)}")
# Build score_lookup from df_brains (ALL operators ever scored across ALL ticks)
# NOTE: This sparse lookup is later replaced by full_score_lookup (Cell 12d)
df_brains_valid = df_brains[
    df_brains['call_id'].isin(matched_ids) & df_brains['churn_ij'].notna()
].copy()
score_lookup = dict(
    zip(
        zip(df_brains_valid['call_id'], df_brains_valid['agent_username']),
        df_brains_valid['churn_ij']
    )
)
churn_i_lookup = dict(
    zip(df_brains_valid['call_id'], df_brains_valid['churn_i'])
)
calls_with_scores = len(set(c for c, o in score_lookup.keys()))
ops_per_call = len(score_lookup) / calls_with_scores if calls_with_scores else 0
print(f"🎯 Pairing scores: {len(score_lookup):,} (call, operator) pairs")
print(f"   Calls with scores: {calls_with_scores:,} / {len(sim_call_ids)}")
print(f"   Avg operators per call: {ops_per_call:.1f}")

# For calls without brains scores (not in brains data), we'll rely on KAPPA gate
global_median_churn_ij = float(np.median(list(score_lookup.values())))
print(f"📊 Global median churn_ij: {global_median_churn_ij:.4f}")

# --- 5. Simulation time window ---
sim_start = df_call_events['arrival_time'].min()
sim_end = df_call_events['arrival_time'].max()
print(f"🕐 Simulation window: {sim_start} → {sim_end}")
print(f"   Duration: {(sim_end - sim_start).total_seconds() / 3600:.1f} hours")

# --- 6. Day-specific operator extraction & shift estimation ---
def get_day_operators(df_sim_data: pd.DataFrame, sim_day: str):
    """
    Extract operators who actually worked on a specific day + their shift windows.
    
    INPUT:  df_sim_data (QPlanner calls with timestamps), sim_day (YYYY-MM-DD string)
    OUTPUT: (df_day_operators, operator_shifts)
      - df_day_operators: DataFrame with agent_username, agent_groupname for that day
      - operator_shifts: dict mapping username → (shift_start, shift_end)
        Shifts estimated from first/last call ± 10min buffer
    """
    day_data = df_sim_data[
        pd.to_datetime(df_sim_data['START_DATE_TIME']).dt.date == pd.to_datetime(sim_day).date()
    ].copy()
    day_data['ts'] = pd.to_datetime(day_data['START_DATE_TIME'])

    day_ops = day_data[day_data['agent_username'].notna()]
    if day_ops.empty:
        return pd.DataFrame(columns=['agent_username', 'agent_groupname']), {}

    # Build shift windows: (first_call - buffer, last_call + avg_duration + buffer)
    shift_buffer = pd.Timedelta(minutes=10)
    median_dur = day_data['CALL_DURATION_SEC_QTY'].median()
    median_dur = median_dur if pd.notna(median_dur) else 1800

    operator_shifts = {}
    for username in day_ops['agent_username'].unique():
        op_calls = day_ops[day_ops['agent_username'] == username]
        shift_start = op_calls['ts'].min() - shift_buffer
        shift_end = op_calls['ts'].max() + pd.Timedelta(seconds=median_dur) + shift_buffer
        operator_shifts[username] = (shift_start, shift_end)

    df_day_operators = (
        day_ops.drop_duplicates(subset='agent_username')[['agent_username', 'agent_groupname']]
        .reset_index(drop=True)
    )
    return df_day_operators, operator_shifts

print(f"\n✅ Helper: get_day_operators() defined")

# --- 7. Compute real QPlanner tick cadence from brains request timestamps ---
# Each unique request_id in df_brains represents one solver invocation.
# The time between consecutive requests IS the real decision cadence.
df_brains_requests = (
    df_brains[['request_id', 'timestamp']]
    .drop_duplicates(subset='request_id')
    .sort_values('timestamp')
    .reset_index(drop=True)
)
df_brains_requests['timestamp'] = pd.to_datetime(df_brains_requests['timestamp'])
df_brains_requests['delta_s'] = df_brains_requests['timestamp'].diff().dt.total_seconds()

# Filter to reasonable cadence (< 5 min = within the same active window, not across gaps)
cadence = df_brains_requests['delta_s'].dropna()
cadence_within_window = cadence[cadence <= 300]

DATA_TICK_MEDIAN = float(cadence_within_window.median())
DATA_TICK_MEAN = float(cadence_within_window.mean())
DATA_TICK_P25 = float(cadence_within_window.quantile(0.25))
DATA_TICK_P75 = float(cadence_within_window.quantile(0.75))

print(f"⏱️  QPlanner request cadence (from {len(cadence_within_window):,} consecutive request pairs):")
print(f"   Median: {DATA_TICK_MEDIAN:.1f}s")
print(f"   Mean:   {DATA_TICK_MEAN:.1f}s")
print(f"   IQR:    [{DATA_TICK_P25:.1f}s, {DATA_TICK_P75:.1f}s]")
print(f"   → Using median ({DATA_TICK_MEDIAN:.0f}s) as tick_interval for simulation")

# --- 8. Compute real operational overhead (pipeline latency) from data ---
# For recommended calls in brains:
#   brains.waiting_time = wait at moment of solver decision (seconds)
#   WAIT_TIME_IN_QUEUE_SEC_QTY = total end-to-end wait in operational results (seconds)
#   overhead = total_wait - solver_wait = real pipeline latency (solver → actual assignment)
df_brains_rec = df_brains[df_brains['recommended'] == True].drop_duplicates(subset='call_id', keep='first')
df_overhead = df_brains_rec[['call_id', 'waiting_time']].rename(
    columns={'waiting_time': 'brains_wait_s'}
).merge(
    df_calls_qplanner[['CALL_ID', 'WAIT_TIME_IN_QUEUE_SEC_QTY']].drop_duplicates(subset='CALL_ID'),
    left_on='call_id', right_on='CALL_ID', how='inner'
)
df_overhead['overhead_s'] = df_overhead['WAIT_TIME_IN_QUEUE_SEC_QTY'] - df_overhead['brains_wait_s']

# Filter to reasonable range (0 to 600s — negative means data timing mismatch)
oh = df_overhead['overhead_s'].dropna()
oh_valid = oh[(oh >= 0) & (oh <= 600)]

DATA_PIPELINE_MEAN = float(oh_valid.mean())
DATA_PIPELINE_STD = float(oh_valid.std())
DATA_PIPELINE_MEDIAN = float(oh_valid.median())
DATA_PIPELINE_P25 = float(oh_valid.quantile(0.25))
DATA_PIPELINE_P75 = float(oh_valid.quantile(0.75))

print(f"\n⚙️  Operational overhead — solver decision → actual assignment (from {len(oh_valid):,} recommended calls):")
print(f"   Median: {DATA_PIPELINE_MEDIAN:.1f}s")
print(f"   Mean:   {DATA_PIPELINE_MEAN:.1f}s")
print(f"   Std:    {DATA_PIPELINE_STD:.1f}s")
print(f"   IQR:    [{DATA_PIPELINE_P25:.1f}s, {DATA_PIPELINE_P75:.1f}s]")
print(f"   Negative overhead count: {(oh < 0).sum()} (data timing mismatches, excluded)")
print(f"   → Using as data-derived pipeline_latency: μ={DATA_PIPELINE_MEAN:.0f}s, σ={DATA_PIPELINE_STD:.0f}s")

# --- 9. Total wait-time distribution of unserved calls (INFORMATIONAL) ---
# QPlanner holds calls for up to 180s (fixed window, agreed with operations).
# After 180s, the telephony system takes over via FIFO routing.
# The wait times below are the TOTAL end-to-end wait (QP queue + FIFO queue),
# NOT the QPlanner-specific timeout. They show how long the telephony system
# kept calls after QPlanner released them.
# NOTE: This distribution is NOT used as a simulation input.
_unserved_mask = df_call_events['QUEUE_PLANNER_RESULT_DESC'].isin([
    'QPlanner-NoAgentReturned', 'QPlanner-AgentNotAvailable'
])
DATA_MAX_WAIT_SAMPLES = df_call_events.loc[_unserved_mask, 'WAIT_TIME_IN_QUEUE_SEC_QTY'].dropna().values.astype(float)
DATA_MAX_WAIT_MEAN = float(np.mean(DATA_MAX_WAIT_SAMPLES))
DATA_MAX_WAIT_MEDIAN = float(np.median(DATA_MAX_WAIT_SAMPLES))
DATA_MAX_WAIT_P25 = float(np.percentile(DATA_MAX_WAIT_SAMPLES, 25))
DATA_MAX_WAIT_P75 = float(np.percentile(DATA_MAX_WAIT_SAMPLES, 75))
DATA_MAX_WAIT_P95 = float(np.percentile(DATA_MAX_WAIT_SAMPLES, 95))
DATA_MAX_WAIT_N = len(DATA_MAX_WAIT_SAMPLES)
print(f"\n⏳ Total wait-time of unserved calls (from {DATA_MAX_WAIT_N:,} calls — QP queue + FIFO, informational):")
print(f"   P25={DATA_MAX_WAIT_P25:.0f}s  Median={DATA_MAX_WAIT_MEDIAN:.0f}s  "
      f"P75={DATA_MAX_WAIT_P75:.0f}s  P95={DATA_MAX_WAIT_P95:.0f}s")
print(f"   Mean={DATA_MAX_WAIT_MEAN:.1f}s  Min={DATA_MAX_WAIT_SAMPLES.min():.0f}s  Max={DATA_MAX_WAIT_SAMPLES.max():.0f}s")
print(f"   → Stored DATA_MAX_WAIT_SAMPLES ({DATA_MAX_WAIT_N:,} values) — informational only, NOT used as simulation input")

# Keep a timestamped copy for later cells
df_calls_qplanner_ts = df_sim.copy()

---
# Part II — Simulation Engine


### Introduction

This section loads the brains scoring data and builds all simulation inputs (operator pool, call events, score lookups, tick cadence, pipeline latency).

<a id="data-exploration"></a>
## Cell 8: Data Exploration — Visualize Key Simulation Inputs

In [ ]:
# =============================================================================
# CELL 8 — DATA EXPLORATION: Visualize key simulation inputs
# =============================================================================
# INPUT:  cadence_within_window, oh_valid, df_call_events, df_brains_valid,
#         df_calls_qplanner, score_lookup, churn_i_lookup,
#         DATA_MAX_WAIT_SAMPLES (from Cell 7)
#
# DOES:   10-panel figure showing the critical data-derived distributions
#         that feed the simulator. Each panel maps to a simulation parameter:
#   1. QPlanner tick cadence   → tick_interval
#   2. Pipeline latency        → pipeline_latency (μ, σ)
#   3. Operators per day       → day-specific operator pool
#   4. Score coverage          → operators available per call
#   5. Call arrival pattern    → 30-min call volume (active hours)
#   6. Churn risk (churn_i)    → baseline churn per call
#   7. Churn_ij & reduction    → pairwise churn scores (core optimization signal)
#   8. Call duration           → operator busy time (pipeline_latency + call_duration)
#   9. Delivery failure rate   → fraction of QP recommendations that fail at telephony
#  10. FO score distribution   → objective function values driving assignments
#
# OUTPUT: 10-panel figure (no new variables — pure visualization)
#
# ─────────────────────────────────────────────────────────────────────────
# TERMINOLOGY NOTE  (applies to entire notebook)
# ─────────────────────────────────────────────────────────────────────────
# The data contains TWO independent outcome flags — don't confuse them:
#
#   ASSIGNED
#     QUEUE_PLANNER_RESULT_DESC = "QPlanner-AgentAvailable"
#     QPlanner recommended an operator and the call was routed.
#
#   UNSERVED  (what the simulator labels "expired")
#     QUEUE_PLANNER_RESULT_DESC ∈ {NoAgentReturned, AgentNotAvailable, ...}
#     QPlanner could NOT recommend an operator during its 180s window.
#     In the REAL telephony platform, these calls are then either:
#       • Deflected to callback (~55 s) — customer is offered callback,
#         hangs up, and returns later with a new CALL_ID.
#         QPlanner gets a second chance on the return call.
#       • Expired to BAU routing — telephony assigns an operator
#         via round-robin; QPlanner has no further say.
#     The QPlanner window is 180s (fixed, agreed with operations).
#     After 180s, the telephony system takes over via FIFO routing.
#     Observed total wait times (QP + FIFO) are longer (median ~247s),
#     but the simulation uses a fixed 180s QP window.
#
#   CUSTOMER ABANDONED  (FLG_ABANDONED = 1 in Infomart)
#     The customer proactively hung up before being served (~2.4% of calls).
#     This is completely independent of the QPlanner outcome above.
#     It is NOT modelled or counted in the simulator.
#
#   Throughout this notebook:
#     "unserved"  = QP could not match (umbrella term)
#     "expired"   = simulator label for calls removed after max_wait_time
#     "abandoned" is AVOIDED to prevent confusion with customer hang-ups.
# ─────────────────────────────────────────────────────────────────────────
# =============================================================================

fig = plt.figure(figsize=(18, 20))
fig.suptitle("Simulation Inputs — Key Distributions from Historical Data",
             fontsize=14, fontweight='bold', y=0.99)
gs = fig.add_gridspec(5, 2, hspace=0.50, wspace=0.35)

# --- 1. QPlanner tick cadence ---
ax = fig.add_subplot(gs[0, 0])
tick_data = cadence_within_window[cadence_within_window <= 30]  # zoom to ≤30s
ax.hist(tick_data, bins=50, color='#2196F3', alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(DATA_TICK_MEDIAN, color='#F44336', ls='--', lw=2, label=f'Median = {DATA_TICK_MEDIAN:.1f}s')
ax.axvline(DATA_TICK_MEAN, color='#FF9800', ls='--', lw=2, label=f'Mean = {DATA_TICK_MEAN:.1f}s')
ax.set_xlabel('Seconds between consecutive ticks')
ax.set_ylabel('Count')
ax.set_title('① QPlanner Tick Cadence\n(time between solver invocations)')
ax.legend(fontsize=8)

# --- 2. Pipeline latency (overhead) ---
ax = fig.add_subplot(gs[0, 1])
lat_data = oh_valid[oh_valid <= 120]  # zoom to ≤120s
ax.hist(lat_data, bins=60, color='#9C27B0', alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(DATA_PIPELINE_MEDIAN, color='#4CAF50', ls='--', lw=2,
           label=f'Median = {DATA_PIPELINE_MEDIAN:.1f}s')
ax.axvline(DATA_PIPELINE_MEAN, color='#FF9800', ls='--', lw=2,
           label=f'Mean = {DATA_PIPELINE_MEAN:.1f}s')
ax.set_xlabel('Overhead (seconds)')
ax.set_ylabel('Count')
ax.set_title('② Pipeline Latency\n(solver decision → actual assignment)')
ax.legend(fontsize=8)

# --- 3. Operators per day ---
ax = fig.add_subplot(gs[1, 0])
_df_tmp = df_calls_qplanner.copy()
_df_tmp['_date'] = pd.to_datetime(_df_tmp['START_DATE_TIME']).dt.date
ops_per_day = _df_tmp[_df_tmp['agent_username'].notna()].groupby('_date')['agent_username'].nunique()
calls_per_day_plot = _df_tmp.drop_duplicates(subset='CALL_ID').groupby(
    pd.to_datetime(_df_tmp.drop_duplicates(subset='CALL_ID')['START_DATE_TIME']).dt.date
).size()
ax2_twin = ax.twinx()
x_dates = range(len(ops_per_day))
ax.bar(x_dates, ops_per_day.values, color='#4CAF50', alpha=0.7, label='Operators')
ax2_twin.plot(x_dates, calls_per_day_plot.reindex(ops_per_day.index, fill_value=0).values,
              color='#F44336', marker='o', ms=4, lw=1.5, label='Calls')
ax.set_xlabel('Day index')
ax.set_ylabel('Operators (bars)', color='#4CAF50')
ax2_twin.set_ylabel('Calls (line)', color='#F44336')
ax.set_title('③ Operators & Calls per Day\n(day-specific pool size)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

# --- 4. Score coverage: operators per call ---
ax = fig.add_subplot(gs[1, 1])
ops_per_call_series = df_brains_valid.groupby('call_id')['agent_username'].nunique()
ax.hist(ops_per_call_series, bins=range(0, int(ops_per_call_series.max()) + 2),
        color='#FF9800', alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(ops_per_call_series.median(), color='#F44336', ls='--', lw=2,
           label=f'Median = {ops_per_call_series.median():.0f}')
ax.axvline(ops_per_call_series.mean(), color='#2196F3', ls='--', lw=2,
           label=f'Mean = {ops_per_call_series.mean():.1f}')
ax.set_xlabel('Unique operators scored per call (all ticks)')
ax.set_ylabel('Number of calls')
ax.set_title('④ Score Coverage\n(unique operators across all brains ticks)')
ax.legend(fontsize=8)

# --- 5. Call arrival pattern (30-min bins, trimmed to active range) ---
ax = fig.add_subplot(gs[2, 0])
_arrival_minutes = df_call_events['arrival_time'].dt.hour * 60 + df_call_events['arrival_time'].dt.minute
_bins_30 = np.arange(_arrival_minutes.min() // 30 * 30,
                     _arrival_minutes.max() // 30 * 30 + 31, 30)
_counts, _edges = np.histogram(_arrival_minutes, bins=_bins_30)
_centers = (_edges[:-1] + _edges[1:]) / 2
_labels = [f"{int(m // 60):02d}:{int(m % 60):02d}" for m in _centers]
ax.bar(range(len(_counts)), _counts, color='#2196F3', alpha=0.8,
       edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(_labels)))
ax.set_xticklabels(_labels, rotation=45, ha='right', fontsize=7)
ax.set_xlabel('Time of day (30-min bins)')
ax.set_ylabel('Total calls (all 21 days)')
ax.set_title('⑤ Call Arrival Pattern\n(30-min bins, active hours only)')

# --- 6. Churn risk (churn_i) distribution ---
ax = fig.add_subplot(gs[2, 1])
churn_vals = df_call_events['churn_i'].dropna()
ax.hist(churn_vals, bins=50, color='#F44336', alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(churn_vals.median(), color='#4CAF50', ls='--', lw=2,
           label=f'Median = {churn_vals.median():.3f}')
ax.axvline(churn_vals.mean(), color='#FF9800', ls='--', lw=2,
           label=f'Mean = {churn_vals.mean():.3f}')
ax.set_xlabel('churn_i (baseline churn probability)')
ax.set_ylabel('Number of calls')
ax.set_title('⑥ Churn Risk Distribution\n(customer baseline churn_i)')
ax.legend(fontsize=8)

# --- 7. Churn_ij & churn reduction distribution ---
# churn_ij = pairwise churn prob (call, operator) from score_lookup
# churn_reduction = churn_i - churn_ij → the optimization signal (G component of FO)
ax = fig.add_subplot(gs[3, 0])
_cij_values = np.array(list(score_lookup.values()))
_ci_for_pairs = np.array([churn_i_lookup.get(k[0], np.nan) for k in score_lookup.keys()])
_valid_mask = ~np.isnan(_ci_for_pairs)
_cij_valid = _cij_values[_valid_mask]
_ci_valid = _ci_for_pairs[_valid_mask]
_reductions = _ci_valid - _cij_valid

# Overlayed histograms: churn_ij and churn reduction
ax.hist(_cij_valid, bins=50, color='#E91E63', alpha=0.5, density=True,
        label=f'churn_ij (n={len(_cij_valid):,}, med={np.median(_cij_valid):.3f})')
ax.hist(_reductions, bins=50, color='#2196F3', alpha=0.5, density=True,
        label=f'Δ churn (i−ij) (med={np.median(_reductions):.4f})')
ax.axvline(0, color='#333', ls='-', lw=1, alpha=0.4)
ax.axvline(np.median(_reductions), color='#2196F3', ls='--', lw=2)
ax.set_xlabel('Probability')
ax.set_ylabel('Density')
ax.set_title('⑦ Churn_ij & Churn Reduction\n(pairwise score — core optimization signal)')
ax.legend(fontsize=7)
# Annotate key stats
_pct_positive = (_reductions > 0).mean() * 100
ax.text(0.97, 0.95, f'Δ>0: {_pct_positive:.1f}%\n'
        f'(QP can reduce churn)',
        transform=ax.transAxes, fontsize=8, va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

# --- 8. Call duration distribution ---
ax = fig.add_subplot(gs[3, 1])
_durations = df_call_events['call_duration'].dropna()
_dur_clipped = _durations[_durations <= 1800]  # zoom to ≤ 30min
_dur_bins = np.arange(0, 1801, 30)
ax.hist(_dur_clipped, bins=_dur_bins, color='#00BCD4', alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axvline(_durations.median(), color='#F44336', ls='--', lw=2,
           label=f'Median = {_durations.median():.0f}s')
ax.axvline(_durations.mean(), color='#FF9800', ls='--', lw=2,
           label=f'Mean = {_durations.mean():.0f}s')
ax.axvline(_durations.quantile(0.95), color='#9C27B0', ls=':', lw=2,
           label=f'P95 = {_durations.quantile(0.95):.0f}s')
ax.set_xlabel('Call duration (seconds)')
ax.set_ylabel('Number of calls')
ax.set_title('⑧ Call Duration\n(determines operator busy time)')
ax.legend(fontsize=8)
_pct_gt_1800 = (_durations > 1800).mean() * 100
if _pct_gt_1800 > 0:
    ax.text(0.97, 0.95, f'>{1800}s: {_pct_gt_1800:.1f}%',
            transform=ax.transAxes, fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

# --- 9. Delivery failure rate (recommended → delivered vs failed) ---
ax = fig.add_subplot(gs[4, 0])
_rec = df_calls_qplanner[df_calls_qplanner['recommended'] == True].drop_duplicates(subset='CALL_ID')
_n_rec = len(_rec)
_n_delivered = (_rec['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable').sum()
_n_failed = _n_rec - _n_delivered
_fail_rate = _n_failed / _n_rec * 100
_rec_tmp = _rec.copy()
_rec_tmp['_date'] = pd.to_datetime(_rec_tmp['START_DATE_TIME']).dt.date
_daily_rec = _rec_tmp.groupby('_date').size()
_daily_del = _rec_tmp[_rec_tmp['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable'].groupby('_date').size()
_daily_fail = _daily_rec - _daily_del.reindex(_daily_rec.index, fill_value=0)
_days = sorted(_daily_rec.index)
_x_d = range(len(_days))
ax.bar(_x_d, [_daily_del.get(d, 0) for d in _days], color='#4CAF50', alpha=0.8, label=f'Delivered ({_n_delivered:,})')
ax.bar(_x_d, [_daily_fail.get(d, 0) for d in _days],
       bottom=[_daily_del.get(d, 0) for d in _days],
       color='#f44336', alpha=0.8, label=f'Failed ({_n_failed:,}, {_fail_rate:.1f}%)')
ax.set_xlabel('Day index')
ax.set_ylabel('QP recommendations')
ax.set_title(f'⑨ Delivery Failure Rate\n(recommended → AgentNotAvailable = {_fail_rate:.1f}%)')
ax.legend(fontsize=8)

# --- 10. FO score distribution (from historical brains data) ---
# FO = β_G · G(churn_i, churn_ij) + β_P · P(wait_time) + κ
# Only pairs where FO > 0 get assigned — this is the KAPPA gate
# Column name in brains data: optimization_score
ax = fig.add_subplot(gs[4, 1])
_fo_col = 'optimization_score' if 'optimization_score' in df_brains_valid.columns else 'FO_VALUE'
_fo_scores = df_brains_valid[_fo_col].dropna() if _fo_col in df_brains_valid.columns else pd.Series(dtype=float)
if len(_fo_scores) > 0:
    _fo_clipped = _fo_scores[(_fo_scores >= -1) & (_fo_scores <= 1)]
    ax.hist(_fo_clipped, bins=80, color='#795548', alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='#F44336', ls='-', lw=2.5, label='KAPPA gate (FO=0)')
    ax.axvline(_fo_scores.median(), color='#FF9800', ls='--', lw=2,
               label=f'Median = {_fo_scores.median():.3f}')
    _pct_positive_fo = (_fo_scores > 0).mean() * 100
    ax.text(0.97, 0.95, f'FO>0: {_pct_positive_fo:.1f}%\n(assignable)',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'optimization_score not available\nin brains data',
            transform=ax.transAxes, fontsize=12, ha='center', va='center',
            bbox=dict(boxstyle='round', facecolor='#FFECB3'))
ax.set_xlabel('FO score (optimization_score)')
ax.set_ylabel('Count')
ax.set_title('⑩ Objective Function (FO) Distribution\n(FO > 0 passes KAPPA gate → assignment)')

plt.show()

# Print summary table
print("\n📋 Simulation Input Summary")
print("=" * 70)
print(f"  Tick cadence:      median {DATA_TICK_MEDIAN:.1f}s, mean {DATA_TICK_MEAN:.1f}s")
print(f"  Pipeline latency:  median {DATA_PIPELINE_MEDIAN:.1f}s, mean {DATA_PIPELINE_MEAN:.1f}s (σ={DATA_PIPELINE_STD:.1f}s)")
print(f"  Operators/day:     {ops_per_day.min()}–{ops_per_day.max()} (median {ops_per_day.median():.0f})")
print(f"  Calls/day:         {calls_per_day_plot.min()}–{calls_per_day_plot.max()} (median {calls_per_day_plot.median():.0f})")
print(f"  Ops/call (scores): {ops_per_call_series.min()}–{ops_per_call_series.max()} (median {ops_per_call_series.median():.0f}, mean {ops_per_call_series.mean():.1f})")
print(f"  Churn_i range:     {churn_vals.min():.3f}–{churn_vals.max():.3f} (median {churn_vals.median():.3f})")
print(f"  Churn_ij:          {len(_cij_valid):,} pairs, median {np.median(_cij_valid):.3f}")
print(f"  Churn reduction:   median {np.median(_reductions):.4f} ({_pct_positive:.1f}% positive — QP can reduce churn)")
print(f"  Call window:       {int(_arrival_minutes.min() // 60):02d}:{int(_arrival_minutes.min() % 60):02d}–{int(_arrival_minutes.max() // 60):02d}:{int(_arrival_minutes.max() % 60):02d} (peak bin {_labels[np.argmax(_counts)]})")
print(f"  Call duration:     median {_durations.median():.0f}s, mean {_durations.mean():.0f}s, P95 {_durations.quantile(0.95):.0f}s")
print(f"  Delivery failure:  {_n_failed:,}/{_n_rec:,} ({_fail_rate:.1f}%) recommended calls failed at telephony")
print(f"  QP window:         180s (fixed, agreed with operations — then telephony FIFO takes over)")

<a id="simulator-core"></a>
## Cell 9: Simulator Core — State Management Classes

In [ ]:
# =============================================================================
# CELL 9 — SIMULATOR CORE: State management classes
# =============================================================================
# INPUT:  None (defines data structures used by later cells)
#
# DOES:   Defines the core domain objects for the simulation:
#   - QueuedCall:       A call waiting in the queue (call_id, churn_i, arrival_time, duration)
#   - Operator:         An operator (username, group, status=available|busy, busy_until)
#   - Assignment:       Record of a completed assignment (call→operator with scores and wait)
#   - SimulationState:  The full state tracker — manages queue, operators, assignments.
#       Key methods:
#         get_available_operators()   → list of operators with status="available"
#         release_finished_calls()    → frees operators whose busy_until has passed
#         enqueue_calls(new_calls)    → adds calls to queue
#         remove_expired(to_fifo)     → removes calls exceeding max_wait_time (180s)
#                                       to_fifo=True → move to FIFO queue (telephony)
#         remove_fifo_abandoned()     → removes FIFO calls exceeding fifo_max_wait
#         execute_assignment(call, op, churn_ij, score, pipeline_latency)
#           → marks operator busy, removes call from queue, records Assignment
#           → waiting_time = time_in_queue + pipeline_latency
#           → operator busy for: pipeline_latency + call_duration
#         route_fifo(available_ops, score_lookup, pipeline_latency)
#           → assigns FIFO queue calls (oldest first) to available operators
#           → models telephony BAU routing for calls QPlanner couldn't serve
#           → operators get busy → realistic occupancy feedback loop
#   - detect_active_windows(): scans call timestamps, finds gaps > 5min
#       → returns list of (start, end) tuples when QPlanner was active
#       → used by the simulation to pause during QPlanner-inactive periods
#
# OUTPUT: Classes and functions available for cells 10-13
# =============================================================================
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta


@dataclass
class QueuedCall:
    """A call waiting in the queue."""
    call_id: str
    customer_id: str
    arrival_time: datetime
    churn_i: float              # baseline churn prob (no agent)
    call_duration: float = 0    # expected call duration (seconds)


@dataclass
class Operator:
    """An operator that can handle calls."""
    username: str
    group: str
    status: str = "available"   # available | busy
    busy_until: Optional[datetime] = None
    current_call_id: Optional[str] = None


@dataclass
class Assignment:
    """Record of a call-operator assignment decision."""
    tick: int
    timestamp: datetime
    call_id: str
    customer_id: str
    operator: str
    waiting_time: float         # seconds the call waited in queue
    churn_i: float              # baseline churn
    churn_ij: float             # churn with this operator
    churn_reduction: float      # churn_i - churn_ij
    optimization_score: float   # the FO score for this pair


@dataclass
class SimulationState:
    """Tracks the full state of the simulation."""
    current_time: datetime = None
    tick: int = 0
    queue: List[QueuedCall] = field(default_factory=list)
    operators: Dict[str, Operator] = field(default_factory=dict)
    assignments: List[Assignment] = field(default_factory=list)
    unserved_calls: List[dict] = field(default_factory=list)
    fifo_queue: List[QueuedCall] = field(default_factory=list)      # calls awaiting FIFO telephony routing
    fifo_assignments: List[Assignment] = field(default_factory=list) # FIFO-routed assignments
    failed_deliveries: List[dict] = field(default_factory=list)  # recommended but not delivered (AgentNotAvailable)
    max_wait_time: float = 180.0  # max seconds before call expires from QP queue

    def get_available_operators(self) -> List[Operator]:
        return [op for op in self.operators.values() if op.status == "available"]

    def release_finished_calls(self):
        """Release operators whose calls have finished."""
        released = 0
        for op in self.operators.values():
            if op.status == "busy" and op.busy_until and self.current_time >= op.busy_until:
                op.status = "available"
                op.busy_until = None
                op.current_call_id = None
                released += 1
        return released

    def enqueue_calls(self, new_calls: List[QueuedCall]):
        """Add new calls to the queue."""
        self.queue.extend(new_calls)

    def remove_expired(self, to_fifo=True):
        """Remove calls that exceeded max wait time (expired from QP queue).

        After max_wait_time (180s, agreed with operations), QPlanner releases
        the call and the telephony system takes over via FIFO routing.

        Args:
            to_fifo: If True, move expired calls to FIFO queue for telephony
                     routing instead of marking them as permanently unserved.
        """
        expired = []
        remaining = []
        for call in self.queue:
            wait = (self.current_time - call.arrival_time).total_seconds()
            if wait > self.max_wait_time:
                expired.append(call)
            else:
                remaining.append(call)
        self.queue = remaining
        if to_fifo:
            self.fifo_queue.extend(expired)
        else:
            for c in expired:
                self.unserved_calls.append({
                    'call_id': c.call_id,
                    'reason': 'expired',
                    'wait_time': (self.current_time - c.arrival_time).total_seconds()
                })
        return len(expired)

    def remove_fifo_abandoned(self, fifo_max_wait: float = 600.0) -> int:
        """Remove calls from FIFO queue that exceed max total wait time.

        Models customer abandonment: customers who waited > fifo_max_wait
        (measured from original arrival_time) hang up and become truly unserved.

        Args:
            fifo_max_wait: Max total wait from original arrival (QP queue + FIFO queue).
                Default 600s (10 min) — customers unlikely to wait longer.

        Returns:
            Number of calls abandoned.
        """
        abandoned = []
        remaining = []
        for call in self.fifo_queue:
            total_wait = (self.current_time - call.arrival_time).total_seconds()
            if total_wait > fifo_max_wait:
                abandoned.append(call)
            else:
                remaining.append(call)
        self.fifo_queue = remaining
        for c in abandoned:
            self.unserved_calls.append({
                'call_id': c.call_id,
                'reason': 'fifo_abandoned',
                'wait_time': (self.current_time - c.arrival_time).total_seconds()
            })
        return len(abandoned)

    def route_fifo(self, available_ops: List[Operator],
                   score_lookup: dict = None,
                   pipeline_latency: float = 2.0,
                   block_operators: bool = False) -> int:
        """
        Route FIFO queue calls to available operators (oldest call first).
        Models telephony BAU routing for calls that QPlanner couldn't serve.

        When block_operators=False (default), FIFO assignments are recorded for
        churn outcome estimation but operators are NOT marked busy. The occupancy
        model already captures telephony BAU utilization implicitly; blocking
        operators here would double-count and starve QPlanner.

        When block_operators=True, operators are marked busy for the full call
        duration (pipeline_latency + call_duration). Use only if the occupancy
        model (occ_scale) has been reduced to compensate.

        Args:
            available_ops: List of currently available Operator objects.
            score_lookup: Optional dict (call_id, operator) → churn_ij.
                If available, uses actual churn_ij; otherwise falls back to churn_i.
            pipeline_latency: Seconds of telephony routing overhead (much
                smaller than QPlanner pipeline — no ML solver, just ACD routing).
            block_operators: If True, mark operators as busy after FIFO assignment.

        Returns:
            Number of FIFO assignments made this tick.
        """
        if not self.fifo_queue or not available_ops:
            return 0

        # Sort FIFO queue by arrival time (oldest first — true FIFO)
        self.fifo_queue.sort(key=lambda c: c.arrival_time)

        assigned = 0
        remaining_ops = list(available_ops)
        remaining_calls = []

        for call in self.fifo_queue:
            if not remaining_ops:
                remaining_calls.append(call)
                continue

            # Assign to first available operator (no optimization)
            op = remaining_ops.pop(0)

            # Look up churn_ij if available; else use churn_i (no optimization benefit)
            churn_ij = score_lookup.get((call.call_id, op.username)) if score_lookup else None
            if churn_ij is None:
                churn_ij = call.churn_i

            wait_time = (self.current_time - call.arrival_time).total_seconds() + pipeline_latency

            if block_operators:
                # Full blocking: operator busy for routing + call duration
                # WARNING: this double-counts with the occupancy model, starving QP
                op.status = "busy"
                op.busy_until = self.current_time + timedelta(
                    seconds=pipeline_latency + call.call_duration
                )
                op.current_call_id = call.call_id
            # else: non-blocking — operator stays available
            # The occupancy model already captures telephony BAU utilization.
            # FIFO records the assignment for churn outcome estimation only.

            # Record FIFO assignment (optimization_score=0 — no optimization)
            self.fifo_assignments.append(Assignment(
                tick=self.tick,
                timestamp=self.current_time,
                call_id=call.call_id,
                customer_id=call.customer_id,
                operator=op.username,
                waiting_time=wait_time,
                churn_i=call.churn_i,
                churn_ij=churn_ij,
                churn_reduction=call.churn_i - churn_ij,
                optimization_score=0.0,
            ))
            assigned += 1

        self.fifo_queue = remaining_calls
        return assigned

    def execute_assignment(self, call: QueuedCall, operator: Operator,
                           churn_ij: float, score: float,
                           pipeline_latency: float = 0.0,
                           acw_time: float = 0.0):
        """Assign a call to an operator and update state.

        pipeline_latency: seconds of overhead between solver decision and operator pickup.
            Models the production pipeline chain (API orchestration → ACD routing → pickup).
            Added to both recorded waiting_time and operator busy duration.
        acw_time: after-call work time in seconds. Operators stay busy after the call
            ends for wrap-up work (CRM notes, system updates). Added to busy_until
            but NOT to customer waiting_time.
        """
        wait_time = (self.current_time - call.arrival_time).total_seconds() + pipeline_latency
        # Update operator state (busy for routing overhead + call duration + ACW)
        operator.status = "busy"
        operator.busy_until = self.current_time + timedelta(seconds=pipeline_latency + call.call_duration + acw_time)
        operator.current_call_id = call.call_id
        # Remove call from queue
        self.queue = [c for c in self.queue if c.call_id != call.call_id]
        # Record assignment
        self.assignments.append(Assignment(
            tick=self.tick,
            timestamp=self.current_time,
            call_id=call.call_id,
            customer_id=call.customer_id,
            operator=operator.username,
            waiting_time=wait_time,
            churn_i=call.churn_i,
            churn_ij=churn_ij,
            churn_reduction=call.churn_i - churn_ij,
            optimization_score=score,
        ))


def detect_active_windows(df_qp_calls: pd.DataFrame, gap_threshold_min: float = 5.0) -> List[Tuple[datetime, datetime]]:
    """
    Detect QPlanner active windows from call timestamps.
    A gap > gap_threshold_min minutes means QPlanner was inactive.
    Returns list of (window_start, window_end) tuples.
    """
    if df_qp_calls.empty:
        return []

    ts = pd.to_datetime(df_qp_calls['START_DATE_TIME']).sort_values().reset_index(drop=True)
    threshold = pd.Timedelta(minutes=gap_threshold_min)

    windows = []
    window_start = ts.iloc[0]
    prev = ts.iloc[0]

    for t in ts.iloc[1:]:
        if t - prev > threshold:
            windows.append((window_start, prev))
            window_start = t
        prev = t

    windows.append((window_start, prev))

    return windows


def analyze_simulation(state: SimulationState, title: str = "Simulation") -> pd.DataFrame:
    """Convert simulation results to a DataFrame and show summary stats."""
    if not state.assignments:
        print("⚠️ No assignments to analyze")
        return pd.DataFrame()

    df_results = pd.DataFrame([
        {
            'tick': a.tick,
            'timestamp': a.timestamp,
            'call_id': a.call_id,
            'operator': a.operator,
            'waiting_time': a.waiting_time,
            'churn_i': a.churn_i,
            'churn_ij': a.churn_ij,
            'churn_reduction': a.churn_reduction,
            'optimization_score': a.optimization_score,
        }
        for a in state.assignments
    ])

    print(f"{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  QP Assignments: {len(df_results)}")
    print(f"  Unserved:       {len(state.unserved_calls)}")
    print(f"  Service rate:   {len(df_results) / (len(df_results) + len(state.unserved_calls)) * 100:.1f}%")
    print()
    print(f"  Waiting Time (seconds):")
    print(f"    Mean:   {df_results['waiting_time'].mean():.1f}")
    print(f"    Median: {df_results['waiting_time'].median():.1f}")
    print(f"    P95:    {df_results['waiting_time'].quantile(0.95):.1f}")
    print(f"    Max:    {df_results['waiting_time'].max():.1f}")
    print()
    print(f"  Churn Reduction:")
    print(f"    Mean:   {df_results['churn_reduction'].mean():.4f}")
    print(f"    Median: {df_results['churn_reduction'].median():.4f}")
    print(f"    Total:  {df_results['churn_reduction'].sum():.2f}")

    if state.fifo_assignments:
        fifo_waits = [a.waiting_time for a in state.fifo_assignments]
        total_served = len(df_results) + len(state.fifo_assignments)
        print()
        print(f"  FIFO Telephony Routing:")
        print(f"    FIFO assigned:  {len(state.fifo_assignments)}")
        print(f"    Total served:   {total_served} (QP: {len(df_results)} + FIFO: {len(state.fifo_assignments)})")
        print(f"    Overall svc rate: {total_served / (total_served + len(state.unserved_calls)) * 100:.1f}%")
        print(f"    Avg wait:       {np.mean(fifo_waits):.1f}s (median: {np.median(fifo_waits):.1f}s)")

    print(f"{'='*60}")

    return df_results


print("✅ Simulator classes defined: QueuedCall, Operator, Assignment, SimulationState, detect_active_windows, analyze_simulation")
print("   SimulationState now includes: fifo_queue, fifo_assignments, failed_deliveries, route_fifo(), remove_fifo_abandoned()")
print("   QPlanner max_wait_time = 180s (fixed, agreed with operations)")

<a id="fo-scoring"></a>
## Cell 10: Scoring Function (FO)

In [ ]:
# =============================================================================
# CELL 10 — SCORING FUNCTION: Mirrors the production mip-v3 objective function
# =============================================================================
# INPUT:  None (defines the FO formula + production hyperparameters)
#
# DOES:   Implements the production objective function FO:
#           FO = β_G · G + β_P · P + κ
#
#   G component (churn reduction gain):
#     g_value = (churn_i - churn_ij) / (|churn_i| + |churn_ij|)   ← normalized churn reduction
#     c = sigmoid(K · churn_i - τ)                                 ← high-risk customer weighting
#     G = (0.5 + g_value) · c
#     → G is higher when the operator reduces churn more AND the customer is high-risk
#
#   P component (waiting-time priority):
#     clamped_wait = min(wait, TE_MAX)   ← only upper-clamped (production clips to TE_MAX only)
#     norm_wait = (clamped_wait - TE_MIN) / (TE_MAX - TE_MIN)
#     P = (e^(α · norm_wait) - 1) / (e^α - 1)
#     → P can be negative when wait < TE_MIN (norm_wait < 0), penalizing very early calls
#
#   Combined: FO = 0.35·G + 0.65·P + (-0.30414)
#     → Assignment only when FO > 0 (KAPPA gate)
#     → Early calls (wait<~100s) naturally have FO ≤ 0 because P is too small
#
#   Discount-priority mode (USE_DISCOUNT_WEIGHTING=True):
#     Multiplies c = sigmoid(K·churn_i - τ) × discount_i
#     where discount_i = normalized operator-spread for the call.
#     → Keeps churn-risk signal (high-risk customers still prioritised)
#     → But down-weights calls where ALL operators give similar churn_ij
#     → Net effect: QP focuses on calls that are BOTH high-risk AND actionable
#
#   Direct-churn oracle mode (USE_DIRECT_CHURN=True):
#     Bypasses the FO formula entirely. score = 1 - churn_ij.
#     → Hungarian minimises churn_ij directly (no gate, no waiting-time influence)
#     → Theoretical ceiling for per-tick matching quality
#     → If THIS can't move NR%, the churn model is the bottleneck
#
#   Two-tier routing mode (USE_TWO_TIER=True):
#     Structural change to break zero-sum:
#     → Tier 1 (high churn_i + high operator spread): wait for the RIGHT
#       valuable operator; only assign when reduction ≥ threshold
#     → Tier 2 (everything else): assign immediately to non-valuable operators
#     → Reserves valuable operators for Tier 1 when T1 calls are waiting
#     → TIER1_STRICT_RESERVE=True: valuable ops NEVER serve T2 (always held for future T1)
#     → TIER1_STRICT_RESERVE=False (default): when no T1 waiting, valuable ops can serve T2
#     → USE_SLATE_LIFT=True: gate baseline = max(cij) in the tick's operator slate
#       instead of the biased churn_i artifact. Always non-negative.
#       Use TIER1_LIFT_FRAC to set fraction of spread that must be captured.
#
# OUTPUT:
#   - FOConfig dataclass: production hyperparameters (TE_MIN=50, TE_MAX=180, etc.)
#   - compute_score(churn_i, churn_ij, waiting_time, config, discount_i) → float
#   - Prints P component at key wait times to show when KAPPA gate opens
# =============================================================================

# This is THE function you modify to experiment with different FO configurations.
#
# Production source of truth:
#   transformations/clupa/qplanner-ds/experiments/cockpit_scheduler/experiment_registry/
#   All live experiments (v0.4.0_BAU, v1.4.1, v1.4.2) use identical brains FO parameters.

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


@dataclass
class FOConfig:
    """Objective function hyperparameters — PRODUCTION DEFAULTS from mip-v3-blacklist.
    MODIFY THESE to experiment with alternative FO configurations."""
    # Waiting time bounds (seconds) — "55 before callback + 15 callback + 110 after callback"
    TE_MIN: float = 50.0
    TE_MAX: float = 180.0
    # Exponential steepness for waiting-time priority
    ALFA: float = 3.0
    # Churn sigmoid parameters
    CHURN_K: float = 20.0
    CHURN_TAU: float = 5.0
    # Component weights
    BETA_G: float = 0.35    # Weight of churn-reduction gain (production)
    BETA_P: float = 0.65    # Weight of waiting-time priority (production)
    KAPPA: float = -0.30414 # Constant offset (production value) — acts as minimum score gate
                            # Assignment only happens when FO > 0, i.e. β_G·G + β_P·P > |KAPPA|
    # Discount-priority mode: weight G by risk × operator-spread
    USE_DISCOUNT_WEIGHTING: bool = False  # When True, c = sigmoid(churn_i) × discount_i
                                          # focuses on calls that are both high-risk AND actionable
    # Direct churn oracle mode: bypass FO entirely, score = 1 - churn_ij
    USE_DIRECT_CHURN: bool = False  # When True, Hungarian minimises raw churn_ij (theoretical ceiling)

    # ── Two-tier routing mode ──────────────────────────────────────
    # Structural change to break the zero-sum constraint.
    # Tier 1 (salvageable): high churn_i + high operator spread → wait for RIGHT operator
    # Tier 2 (routine): everything else → assign immediately to non-valuable operators
    USE_TWO_TIER: bool = False
    TIER1_CALL_IDS: set = None       # pre-computed set of Tier 1 call IDs (churn_i > thresh AND best_reduction > thresh)
    TIER1_VALUABLE_OPS: set = None   # operators whose mean cij on Tier 1 calls is below median (good at saving)
    TIER1_MIN_REDUCTION: float = 0.05  # minimum (churn_i - churn_ij) for a Tier 1 assignment to proceed
    TIER1_STRICT_RESERVE: bool = False  # When True, valuable ops NEVER serve T2 (always held for future T1)
                                        # When False, valuable ops serve T2 when no T1 calls are waiting
    # ── Slate-based lift mode ─────────────────────────────────────
    # Instead of reduction = churn_i - churn_ij (biased artifact baseline),
    # use reduction = max(churn_ij in slate) - churn_ij (worst-op-in-slate baseline).
    # Eliminates negative operator contributions. Always ≥ 0.
    # Gate: "operator must capture ≥ TIER1_LIFT_FRAC of available spread".
    # e.g., TIER1_LIFT_FRAC=0.50 → operator must be in the better half of the slate.
    USE_SLATE_LIFT: bool = False
    TIER1_LIFT_FRAC: float = 0.50   # fraction of (max_cij - min_cij) that lift must reach
                                     # 0.50 = "at least 50% of available quality range"


def compute_score(
    churn_i: float,
    churn_ij: float,
    waiting_time: float,
    config: FOConfig,
    discount_i: float = 0.0,  # normalized operator spread for this call [0,1]
) -> float:
    """
    Compute the optimization score for a (call, operator) pair.

    This is the function you swap out to test different FO formulations.
    Returns a score where HIGHER = better assignment.

    Modes:
      - Default: FO = β_G·G + β_P·P + κ  (production formula)
      - USE_DISCOUNT_WEIGHTING: G weights c = sigmoid(churn_i) × discount_i
      - USE_DIRECT_CHURN: bypasses FO entirely → score = 1 - churn_ij
        (theoretical ceiling — Hungarian directly minimises churn_ij)
    """
    # --- Oracle mode: bypass FO, directly optimise churn_ij ---
    if config.USE_DIRECT_CHURN:
        return 1.0 - churn_ij  # always > 0 (no gate), lower churn_ij = higher score

    # --- G component: churn reduction gain ---
    denom = abs(churn_i) + abs(churn_ij)
    g_value = (churn_i - churn_ij) / denom if denom > 1e-9 else 0.0
    c_sig = sigmoid(config.CHURN_K * churn_i - config.CHURN_TAU)
    if config.USE_DISCOUNT_WEIGHTING:
        c = c_sig * discount_i  # risk × actionability
    else:
        c = c_sig
    G = (0.5 + g_value) * c

    # Production only clips to TE_MAX (no lower bound).
    # When wait < TE_MIN, P goes negative → penalizes very early assignments.
    clamped_wait = min(waiting_time, config.TE_MAX)
    te_range = config.TE_MAX - config.TE_MIN
    norm_wait = (clamped_wait - config.TE_MIN) / te_range if te_range > 0 else 0.0
    exp_alfa = np.exp(config.ALFA)
    P = (np.exp(config.ALFA * norm_wait) - 1.0) / (exp_alfa - 1.0) if exp_alfa > 1 else norm_wait

    # --- Combined FO ---
    score = config.BETA_G * G + config.BETA_P * P + config.KAPPA
    return score


# Quick sanity check with production defaults
cfg = FOConfig()
test_score = compute_score(churn_i=0.7, churn_ij=0.3, waiting_time=60.0, config=cfg)
print(f"✅ FOConfig (production defaults):")
print(f"   TE_MIN={cfg.TE_MIN}, TE_MAX={cfg.TE_MAX}, ALFA={cfg.ALFA}")
print(f"   BETA_G={cfg.BETA_G}, BETA_P={cfg.BETA_P}, KAPPA={cfg.KAPPA}")
print(f"   CHURN_K={cfg.CHURN_K}, CHURN_TAU={cfg.CHURN_TAU}")
print(f"   USE_DISCOUNT_WEIGHTING={cfg.USE_DISCOUNT_WEIGHTING}")
print(f"   USE_DIRECT_CHURN={cfg.USE_DIRECT_CHURN}")
print(f"   USE_TWO_TIER={cfg.USE_TWO_TIER}")
print(f"   TIER1_STRICT_RESERVE={cfg.TIER1_STRICT_RESERVE}")
print(f"   USE_SLATE_LIFT={cfg.USE_SLATE_LIFT}")
print(f"   TIER1_LIFT_FRAC={cfg.TIER1_LIFT_FRAC}")
print(f"   Test score (churn_i=0.7, churn_ij=0.3, wait=60s): {test_score:.4f}")

# Oracle mode sanity check
cfg_oracle = FOConfig(USE_DIRECT_CHURN=True)
test_oracle = compute_score(churn_i=0.7, churn_ij=0.3, waiting_time=60.0, config=cfg_oracle)
print(f"   Oracle score (same inputs): {test_oracle:.4f}  (= 1.0 - 0.3)")

for w in [30, 50, 100, 120, 140, 160, 180]:
    cw = min(w, cfg.TE_MAX)  # production only clips upper bound
    nw = (cw - cfg.TE_MIN) / (cfg.TE_MAX - cfg.TE_MIN)
    p = (np.exp(cfg.ALFA * nw) - 1) / (np.exp(cfg.ALFA) - 1)
    p_weighted = cfg.BETA_P * p
    g_needed = (abs(cfg.KAPPA) - p_weighted) / cfg.BETA_G if cfg.BETA_G > 0 else 0
    print(f"   wait={w:>3}s → P={p:.3f}, β_P·P={p_weighted:.3f}, G needed for FO>0: {max(0, g_needed):.3f}")


<a id="assignment-solver"></a>
## Cell 11: Assignment Solver (Hungarian Algorithm)

In [ ]:
# =============================================================================
# CELL 11 — ASSIGNMENT SOLVER: Optimal matching via Hungarian algorithm
# =============================================================================
# INPUT (per tick):
#   - queue: list of QueuedCall objects currently waiting
#   - available_ops: list of Operator objects currently available
#   - score_lookup: dict (call_id, operator) → churn_ij (from cell 7)
#   - churn_i_lookup: dict call_id → churn_i (from cell 7)
#   - current_time: datetime (for computing waiting_time)
#   - fo_config: FOConfig with production parameters
#   - fallback_churn_ij: optional fallback for missing pairs
#   - call_discounts: pre-computed {call_id: normalized_spread} for discount-priority
#
# DOES:
#   1. Builds an (n_calls × n_ops) cost matrix
#   2. For each (call, operator) pair:
#      - Looks up churn_ij from score_lookup (or uses fallback)
#      - Computes FO score via compute_score()
#      - Sets cost = -score (Hungarian minimizes, we want to maximize)
#      - Infeasible pairs (no churn_ij) get cost = 1e6 (effectively infinite)
#   3. Solves with scipy.optimize.linear_sum_assignment (Hungarian algorithm)
#      → Guarantees 1-to-1 matching: each call ≤ 1 operator, each op ≤ 1 call
#   4. Filters results: only keeps pairs where score > 0 (KAPPA gate)
#
# OUTPUT: List of (QueuedCall, Operator, churn_ij, score) tuples to assign
# =============================================================================

from scipy.optimize import linear_sum_assignment


def solve_assignments(
    queue: List[QueuedCall],
    available_ops: List[Operator],
    score_lookup: dict,
    churn_i_lookup: dict,
    current_time: datetime,
    fo_config: FOConfig,
    min_score_threshold: float = 0.0,
    fallback_churn_ij: float = None,  # fallback churn_ij for missing (call, operator) pairs
    call_discounts: dict = None,  # pre-computed {call_id: normalized_spread} for discount-priority
) -> List[Tuple[QueuedCall, Operator, float, float]]:
    """
    Solve the optimal call-operator assignment.

    Returns list of (call, operator, churn_ij, score) tuples.
    Only returns pairs where score > min_score_threshold.

    If fallback_churn_ij is set, pairs without a pre-computed score
    will use this value instead of being marked infeasible. This
    dramatically increases score coverage (matching real smartpairing
    behavior where ALL available operators get scores).

    If call_discounts is set and fo_config.USE_DISCOUNT_WEIGHTING is True,
    passes per-call discount_i to compute_score for operator-spread weighting.
    """
    if not queue or not available_ops:
        return []

    n_calls = len(queue)
    n_ops = len(available_ops)

    # Build cost matrix (we negate scores because linear_sum_assignment minimizes)
    LARGE_COST = 1e6  # penalty for infeasible pairs (no score available)
    cost_matrix = np.full((n_calls, n_ops), LARGE_COST)
    score_matrix = np.zeros((n_calls, n_ops))
    churn_ij_matrix = np.full((n_calls, n_ops), np.nan)

    for i, call in enumerate(queue):
        wait = (current_time - call.arrival_time).total_seconds()
        _discount = call_discounts.get(call.call_id, 0.0) if call_discounts else 0.0
        for j, op in enumerate(available_ops):
            # Look up pre-computed churn_ij for this (call, operator) pair
            churn_ij = score_lookup.get((call.call_id, op.username))
            if churn_ij is None and fallback_churn_ij is not None:
                # Use fallback: mimic smartpairing computing a "neutral" score
                churn_ij = fallback_churn_ij
            if churn_ij is not None:
                score = compute_score(call.churn_i, churn_ij, wait, fo_config, discount_i=_discount)
                score_matrix[i, j] = score
                churn_ij_matrix[i, j] = churn_ij
                cost_matrix[i, j] = -score  # negate for minimization

    # Solve assignment (handles rectangular matrices)
    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    # Filter: only keep assignments with valid scores above threshold
    results = []
    for i, j in zip(row_idx, col_idx):
        if cost_matrix[i, j] < LARGE_COST and score_matrix[i, j] > min_score_threshold:
            results.append((
                queue[i],
                available_ops[j],
                churn_ij_matrix[i, j],
                score_matrix[i, j],
            ))

    return results


print("✅ Assignment solver defined (Hungarian algorithm via scipy, with fallback + discount-priority support)")

<a id="two-tier-solver"></a>
## Cell 11b: Two-Tier Solver (`solve_tiered`)

In [ ]:
# =============================================================================
# CELL 12 — TWO-TIER SOLVER: Structural change to break zero-sum
# =============================================================================
# INPUT:  queue, available_ops, score_lookup, churn_i_lookup, fo_config
#
# DOES:   Implements two-tier routing:
#   Tier 1 (salvageable) = pre-computed high-risk + high operator-spread calls
#     → Solved FIRST with valuable operators using oracle scoring (1-churn_ij)
#     → Min-reduction gate: only assign if reduction ≥ threshold
#       - Default: reduction = churn_i - churn_ij (baseline is churn_i artifact)
#       - USE_SLATE_LIFT: reduction = max_cij_in_slate - churn_ij (baseline is worst op)
#         with threshold = TIER1_LIFT_FRAC × (max_cij - min_cij)
#     → If no good enough operator available → call WAITS (up to 180s naturally)
#
#   Tier 2 (routine) = everything else
#     → Solved SECOND with remaining operators
#     → Uses same oracle scoring but NO min-reduction gate
#     → Operator pool depends on strict-reserve mode:
#       (a) Default: if T1 calls still waiting → reserve valuable ops, T2 gets only non-valuable
#                    if no T1 waiting → T2 gets ALL remaining ops (including unused valuable)
#       (b) Strict (TIER1_STRICT_RESERVE=True): T2 NEVER gets valuable ops, even if idle
#
# OUTPUT: list of (call, operator, churn_ij, score) pairs
# =============================================================================

def solve_tiered(
    queue: list,
    available_ops: list,
    score_lookup: dict,
    churn_i_lookup: dict,
    current_time: float,
    fo_config: FOConfig,
    fallback_churn_ij: float = 0.5,
    call_discounts: dict = None,
) -> list:
    """Two-tier solver: split queue by call value class, solve T1 first with valuable ops."""
    t1_ids = fo_config.TIER1_CALL_IDS or set()
    val_ops_set = fo_config.TIER1_VALUABLE_OPS or set()
    min_red = fo_config.TIER1_MIN_REDUCTION
    _use_slate = fo_config.USE_SLATE_LIFT
    _lift_frac = fo_config.TIER1_LIFT_FRAC

    # --- Split queue by tier ---
    t1_queue = [c for c in queue if c.call_id in t1_ids]
    t2_queue = [c for c in queue if c.call_id not in t1_ids]

    # --- Split operators by value class ---
    valuable_ops = [op for op in available_ops if op.username in val_ops_set]
    other_ops = [op for op in available_ops if op.username not in val_ops_set]

    all_pairs = []
    used_ops = set()

    # Oracle config for both tiers (bypass FO, minimize churn_ij)
    _oracle_cfg = FOConfig(USE_DIRECT_CHURN=True)

    # ── Pre-compute slate stats for slate-based lift ─────────────
    # For each T1 call, find max_cij and min_cij across available valuable operators
    _slate_max = {}
    _slate_min = {}
    if _use_slate and t1_queue and valuable_ops:
        _val_usernames = [op.username for op in valuable_ops]
        for call in t1_queue:
            cij_vals = []
            for op_name in _val_usernames:
                key = (call.call_id, op_name)
                cij = score_lookup.get(key)
                if cij is not None:
                    cij_vals.append(cij)
            if cij_vals:
                _slate_max[call.call_id] = max(cij_vals)
                _slate_min[call.call_id] = min(cij_vals)
            else:
                _slate_max[call.call_id] = fallback_churn_ij
                _slate_min[call.call_id] = fallback_churn_ij

    # ── Step 1: Tier 1 × valuable operators ──────────────────────
    if t1_queue and valuable_ops:
        t1_pairs = solve_assignments(
            queue=t1_queue,
            available_ops=valuable_ops,
            score_lookup=score_lookup,
            churn_i_lookup=churn_i_lookup,
            current_time=current_time,
            fo_config=_oracle_cfg,
            fallback_churn_ij=fallback_churn_ij,
            call_discounts=call_discounts,
        )
        # Apply min-reduction gate: only assign if reduction is meaningful
        for call, op, cij, score in t1_pairs:
            if _use_slate:
                # Slate-based lift: reduction relative to worst op in slate
                max_cij = _slate_max.get(call.call_id, fallback_churn_ij)
                min_cij = _slate_min.get(call.call_id, fallback_churn_ij)
                spread = max_cij - min_cij
                reduction = max_cij - cij
                # Gate: operator must capture ≥ LIFT_FRAC of the available spread
                threshold = _lift_frac * spread if spread > 1e-6 else 0.0
                passes_gate = reduction >= threshold
            else:
                # Original: reduction relative to churn_i artifact
                reduction = call.churn_i - cij
                passes_gate = reduction >= min_red

            if passes_gate:
                all_pairs.append((call, op, cij, score))
                used_ops.add(op.username)
            # else: call stays in queue → will wait for a better operator next tick

    # ── Step 2: Tier 2 × remaining operators ─────────────────────
    # Determine which operators are available for Tier 2
    t1_assigned_ids = {c.call_id for c, _, _, _ in all_pairs if c.call_id in t1_ids}
    t1_still_waiting = len(t1_queue) - len(t1_assigned_ids)

    remaining_ops = [op for op in available_ops if op.username not in used_ops]

    if t1_still_waiting > 0 or fo_config.TIER1_STRICT_RESERVE:
        # Valuable ops reserved for Tier 1:
        #   - Default mode: only when T1 calls are still waiting
        #   - Strict mode: ALWAYS (valuable ops NEVER serve T2)
        t2_ops = [op for op in remaining_ops if op.username not in val_ops_set]
    else:
        # No Tier 1 calls waiting AND not strict → Tier 2 can use ALL remaining operators
        t2_ops = remaining_ops

    if t2_queue and t2_ops:
        t2_pairs = solve_assignments(
            queue=t2_queue,
            available_ops=t2_ops,
            score_lookup=score_lookup,
            churn_i_lookup=churn_i_lookup,
            current_time=current_time,
            fo_config=_oracle_cfg,
            fallback_churn_ij=fallback_churn_ij,
            call_discounts=call_discounts,
        )
        all_pairs.extend(t2_pairs)

    return all_pairs


<a id="simulation-loop"></a>
## Cell 12: Main Simulation Loop

In [ ]:
# =============================================================================
# CELL 12 — MAIN SIMULATION LOOP (gap-aware, with FIFO telephony routing)
# =============================================================================
# INPUT:
#   - df_call_events: one row per call with arrival_time, churn_i, duration (from cell 7)
#   - df_operators: operator pool for the day (from get_day_operators)
#   - score_lookup: (call_id, operator) → churn_ij (from cell 7)
#   - churn_i_lookup: call_id → churn_i (from cell 7)
#   - fo_config: FOConfig with production parameters (from cell 10)
#   - active_windows: QPlanner on/off periods (from detect_active_windows)
#   - operator_shifts: per-operator shift windows (from get_day_operators)
#   - hourly_occupancy: hour → occupancy fraction (computed per-day in cell 12)
#   - Calibrated parameters: tick_interval, pipeline_latency, nqp_mean_task_s
#
# DOES (main tick loop):
#   Each tick (every 3s of simulated time):
#     0. CHECK GAP: if not in active window, block 70% of operators and skip
#        On gap→active transition, move queued calls to FIFO (telephony takes over)
#     1. RELEASE: free operators whose (pipeline_latency + call_duration + ACW) elapsed
#     2. ENQUEUE: add new calls that arrived since last tick
#     3. CALLBACK DEFLECTION: calls waiting ≥ 55s get one-time callback offer
#     4. EXPIRE: remove calls waiting > max_wait_time (180s) → move to FIFO queue
#     4b. FIFO ABANDON: remove FIFO calls where total wait > fifo_max_wait
#
#   ┌─────────────────────────────────────────────────────────────────────────┐
#   │ OPERATOR AVAILABILITY PIPELINE (6-stage filter, each tick)             │
#   │                                                                       │
#   │ An operator must pass ALL stages to reach the solver:                 │
#   │                                                                       │
#   │  Stage 1: release_finished_calls()                                    │
#   │    → Free operators whose call + ACW ended (busy_until ≤ now).        │
#   │    → Operator goes status="available".                                │
#   │                                                                       │
#   │  Stage 2: get_available_operators()                                   │
#   │    → Filter out status="busy" operators (handling QP or FIFO calls).  │
#   │    → These never reach any occupancy check.                           │
#   │                                                                       │
#   │  Stage 3: Shift filter                                                │
#   │    → Remove operators outside their shift window (start ≤ now ≤ end). │
#   │                                                                       │
#   │  Stage 4: Occupancy check — ongoing non-QP task?                      │
#   │    → Check nqp_busy_until[op] > current_time.                         │
#   │    → If YES: operator still doing admin/breaks/cross-team work from   │
#   │      a previous tick → skip (no dice roll, deterministic).            │
#   │                                                                       │
#   │  Stage 5: Occupancy check — start NEW non-QP task?                    │
#   │    → Roll random() < p_start (≈0.15% per tick).                       │
#   │    → If YES: assign exponential-duration task (~20min mean),          │
#   │      record nqp_busy_until[op] = now + duration → skip.              │
#   │    → If NO: operator is truly available.                              │
#   │                                                                       │
#   │  Stage 6: Solver (Hungarian algorithm)                                │
#   │    → Only operators surviving all 5 filters enter the cost matrix.    │
#   │    → KAPPA gate (FO > 0) further restricts actual assignments.        │
#   │                                                                       │
#   │ KEY: Stages 4-5 use a persistent nqp_busy_until dict that carries     │
#   │ state across ticks. Non-QP tasks last ~20min (hundreds of ticks),     │
#   │ producing bursty correlated unavailability, not memoryless coin flips. │
#   │ CALIBRATED_OCC_SCALE controls the effective occupancy rate that       │
#   │ drives p_start, calibrated so sim QP served ≈ historical (~5,407).   │
#   └─────────────────────────────────────────────────────────────────────────┘
#
#     7. EXECUTE QP assignments (mark operators busy for latency + duration + ACW)
#     8. FIFO ROUTE (PRIORITY 2 — Telephony): remaining operators serve FIFO queue
#        → oldest call first, no ML optimization, ~2s routing latency
#        → only fifo_routing_fraction of leftover operators offered (ACD pacing)
#        → non-blocking by default (churn estimation only, no double-count with occ model)
#     9. ADVANCE CLOCK by tick_interval seconds
#
# OUTPUT: SimulationState containing:
#   - state.assignments: QPlanner-optimized assignments
#   - state.fifo_assignments: telephony FIFO-routed assignments
#   - state.unserved_calls: abandoned/end-of-day calls (truly lost)
#   - state.operators: final operator states
#   - state.score_coverage: dict with solver-level coverage stats
# =============================================================================

def _is_in_active_window(t: datetime, windows: List[Tuple[datetime, datetime]]) -> bool:
    """Check if timestamp t falls within any QPlanner active window."""
    for ws, we in windows:
        if ws <= t <= we:
            return True
    return False


def run_simulation(
    df_call_events: pd.DataFrame,
    df_operators: pd.DataFrame,
    score_lookup: dict,
    churn_i_lookup: dict,
    fo_config: FOConfig = FOConfig(),
    tick_interval: int = 5,         # seconds between decisions
    max_wait_time: float = 180.0,   # seconds before call expires from QP queue (fixed, agreed with operations)
    sim_day: str = None,            # simulate a single day (YYYY-MM-DD), or None for all
    active_windows: List[Tuple[datetime, datetime]] = None,  # QPlanner active windows
    gap_busy_fraction: float = 0.7, # fraction of operators busy during gaps (other strategy)
    base_occupancy: float = 0.0,    # fraction of operators unavailable per tick (non-QP work)
    hourly_occupancy: Dict[int, float] = None,  # hour → occupancy override (replaces base_occupancy)
    operator_shifts: Dict[str, Tuple[datetime, datetime]] = None,  # per-operator shift windows
    fallback_churn_ij: float = None,  # fallback churn_ij for missing scores (enables full coverage)
    pipeline_latency: float = 0.0,    # seconds of overhead: API chain + ACD routing + operator pickup
    pipeline_latency_std: float = 0.0,  # std of pipeline latency (lognormal). 0 = fixed latency.
    nqp_mean_task_s: float = 1200.0,   # mean duration of non-QP tasks (seconds). Controls occupancy persistence. (~20 min; matches CALIBRATED_NQP_TASK_S)
    fifo_latency: float = 2.0,        # telephony FIFO routing latency (much faster than QP pipeline)
    fifo_max_wait: float = 600.0,     # max total wait (arrival→abandon) for FIFO calls (seconds)
    fifo_routing_fraction: float = 1.0, # fraction of leftover operators offered to FIFO each tick (0-1). <1 = ACD pacing buffer
    fifo_block_operators: bool = False,  # if True, FIFO marks operators busy for call_duration (double-counts with occ model!)
    enable_fifo: bool = True,          # enable FIFO telephony routing for expired/gap-cleared calls
    delivery_failure_rate: float = 0.0,  # probability of delivery failure per assignment (production ~14.2%)
    callback_deflection_time: float = 0.0,  # seconds of wait before telephony offers callback (0 = disabled)
    callback_deflection_rate: float = 0.0,  # probability [0,1] of accepting callback offer at deflection_time
    acw_mean_s: float = 0.0,  # mean after-call work time (seconds). Operators stay busy after call ends. 0 = disabled.
    verbose: bool = True,
) -> SimulationState:
    """
    Run the QPlanner simulator (gap-aware, with FIFO telephony routing).

    The KAPPA gate is modeled naturally: every tick, ALL queued calls are scored
    against available operators. The solver computes FO for each (call, operator)
    pair, and only pairs with FO > 0 are assigned. With KAPPA = -0.30414, this
    means calls need sufficient P (waiting-time priority) to overcome |KAPPA|.

    FIFO telephony routing: calls that QPlanner couldn't serve (expired after
    max_wait_time=180s or cleared during gaps) enter a FIFO queue. After
    QPlanner has first pick of available operators each tick, remaining
    operators serve FIFO calls oldest-first. This creates a realistic occupancy
    feedback loop: FIFO calls consume operator time → fewer operators available
    for QPlanner → more realistic simulation.

    Customer abandonment: FIFO calls exceeding fifo_max_wait (default 600s = 10min
    from original arrival) are abandoned — customers unlikely to wait longer.

    Two-tier mode (fo_config.USE_TWO_TIER=True): replaces the standard solver
    with solve_tiered(), which splits calls into Tier 1 (salvageable) and
    Tier 2 (routine), reserves valuable operators for Tier 1, and applies
    a minimum-reduction gate. See solve_tiered() docstring for details.
    """
    # --- Filter to simulation day if specified ---
    events = df_call_events.copy()
    if sim_day:
        events = events[events['arrival_time'].dt.date == pd.to_datetime(sim_day).date()]
        if events.empty:
            print(f"⚠️ No calls found for {sim_day}")
            return SimulationState()
    events = events.sort_values('arrival_time').reset_index(drop=True)

    # --- Initialize state ---
    state = SimulationState(max_wait_time=max_wait_time)
    state.current_time = events['arrival_time'].iloc[0]

    # Initialize operators (all start as available)
    for _, row in df_operators.iterrows():
        state.operators[row['agent_username']] = Operator(
            username=row['agent_username'],
            group=row['agent_groupname'],
        )

    # --- Initialize non-QP task tracking (task-based occupancy model) ---
    nqp_busy_until = {}
    nqp_rng = np.random.RandomState(42)
    first_hour = events['arrival_time'].iloc[0].hour
    initial_occ = hourly_occupancy.get(first_hour, base_occupancy) if hourly_occupancy else base_occupancy
    for op_name in state.operators:
        if nqp_rng.random() < initial_occ:
            remaining = nqp_rng.exponential(nqp_mean_task_s / 2)
            nqp_busy_until[op_name] = state.current_time + timedelta(seconds=remaining)

    sim_end = events['arrival_time'].iloc[-1] + timedelta(seconds=max_wait_time)
    next_call_idx = 0
    total_calls = len(events)
    log_every = max(1, int(60 / tick_interval))
    was_in_gap = False

    # Score coverage tracking
    _total_solver_pairs = 0
    _total_solver_hits = 0
    _tick_ops_log = []     # (n_ops, n_queue) per solver-active tick
    _callback_deflections = 0  # counter for callback-deflected calls
    _callback_offered = set()  # track call_ids already offered callback (one-time offer)

    # Pre-compute per-call operator discount (spread) for discount-priority mode
    _call_discounts = None
    if fo_config.USE_DISCOUNT_WEIGHTING:
        from collections import defaultdict
        _cij_by_call = defaultdict(list)
        for (cid, _op), cij_val in score_lookup.items():
            _cij_by_call[cid].append(cij_val)
        _spreads = {cid: max(vals) - min(vals) for cid, vals in _cij_by_call.items() if len(vals) >= 2}
        _max_spread = max(_spreads.values()) if _spreads else 1.0
        _call_discounts = {cid: sp / _max_spread for cid, sp in _spreads.items()}
        if verbose:
            print(f"   Discount-priority: {len(_call_discounts):,} calls scored, "
                  f"max spread={_max_spread:.4f}, median={np.median(list(_spreads.values())):.4f}")

    # Determine solver mode
    _use_tiered = fo_config.USE_TWO_TIER and fo_config.TIER1_CALL_IDS

    if verbose:
        gap_mode = "ON" if active_windows else "OFF"
        fifo_mode = "ON" if enable_fifo else "OFF"
        print(f"🚀 Starting simulation: {len(events)} calls, {len(state.operators)} operators")
        print(f"   FO config: β_G={fo_config.BETA_G}, β_P={fo_config.BETA_P}, κ={fo_config.KAPPA}")
        if _use_tiered:
            _n_t1 = len(fo_config.TIER1_CALL_IDS)
            _n_vops = len(fo_config.TIER1_VALUABLE_OPS) if fo_config.TIER1_VALUABLE_OPS else 0
            print(f"   ★ TWO-TIER MODE: {_n_t1:,} Tier 1 calls, {_n_vops} valuable ops, "
                  f"min_reduction={fo_config.TIER1_MIN_REDUCTION:.3f}")
        print(f"   Tick interval: {tick_interval}s, Max wait: {max_wait_time}s (fixed QP window)")
        print(f"   Gap handling: {gap_mode}" + (f" (busy_fraction={gap_busy_fraction})" if active_windows else ""))
        if acw_mean_s > 0:
            print(f"   After-call work (ACW): mean={acw_mean_s:.0f}s (exponential, delays operator re-availability)")
        if hourly_occupancy:
            print(f"   Hourly occupancy: dynamic ({len(hourly_occupancy)} hours, range {min(hourly_occupancy.values()):.0%}-{max(hourly_occupancy.values()):.0%})")
        else:
            print(f"   Base occupancy: {base_occupancy:.0%} (static)")
        print()
        state.tick += 1

    # --- Main loop ---
    while state.current_time <= sim_end:
        state.tick += 1

        # --- GAP HANDLING: Check if we're in a QPlanner active window ---
        in_active_window = True
        if active_windows:
            in_active_window = _is_in_active_window(state.current_time, active_windows)

        if active_windows and not in_active_window:
            if not was_in_gap and verbose:
                print(f"  ⏸️  Entering gap at {state.current_time.strftime('%H:%M:%S')} — operators serving other strategy")
            available = state.get_available_operators()
            n_to_block = int(len(available) * gap_busy_fraction)
            for op in available[:n_to_block]:
                op.status = "busy"
                op.busy_until = state.current_time + timedelta(seconds=tick_interval * 2)
            was_in_gap = True
            state.release_finished_calls()

            # During gaps, FIFO routing still happens (telephony doesn't stop)
            if enable_fifo:
                state.remove_fifo_abandoned(fifo_max_wait=fifo_max_wait)
                if state.fifo_queue:
                    gap_avail = state.get_available_operators()
                    if operator_shifts and gap_avail:
                        gap_avail = [
                            op for op in gap_avail
                            if op.username in operator_shifts
                            and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
                        ]
                    if fifo_routing_fraction < 1.0 and gap_avail:
                        n_gap_fifo = max(1, int(len(gap_avail) * fifo_routing_fraction))
                        gap_avail = gap_avail[:n_gap_fifo]
                    if gap_avail:
                        state.route_fifo(gap_avail, score_lookup, fifo_latency, block_operators=fifo_block_operators)
            state.current_time += timedelta(seconds=tick_interval)
            continue

        # Transition: gap → active window
        if was_in_gap and in_active_window:
            if verbose:
                print(f"  ▶️  Resuming QPlanner at {state.current_time.strftime('%H:%M:%S')}")
            for op in state.operators.values():
                if op.status == "busy" and op.busy_until and op.busy_until <= state.current_time + timedelta(seconds=tick_interval):
                    op.status = "available"
                    op.busy_until = None
                    op.current_call_id = None
            # Gap-cleared QP queue calls → move to FIFO (telephony takes over)
            if state.queue:
                if enable_fifo:
                    state.fifo_queue.extend(state.queue)
                else:
                    for c in state.queue:
                        state.unserved_calls.append({
                            'call_id': c.call_id,
                            'reason': 'gap_cleared',
                            'wait_time': (state.current_time - c.arrival_time).total_seconds()
                        })
                state.queue = []
            was_in_gap = False

        # --- NORMAL SIMULATION LOGIC ---
        # 1. Release operators whose calls have finished
        released = state.release_finished_calls()

        # 2. Enqueue new calls that arrived since last tick
        new_calls = []
        while next_call_idx < total_calls:
            row = events.iloc[next_call_idx]
            if row['arrival_time'] <= state.current_time:
                churn_i = churn_i_lookup.get(row['CALL_ID'], row.get('churn_i', 0.5))
                new_calls.append(QueuedCall(
                    call_id=row['CALL_ID'],
                    customer_id=str(row['SA_COD']),
                    arrival_time=row['arrival_time'],
                    churn_i=float(churn_i),
                    call_duration=float(row['call_duration']),
                ))
                next_call_idx += 1
            else:
                break
        state.enqueue_calls(new_calls)

        #     ONE-TIME offer: when a call first reaches callback_deflection_time,
        #     it gets a single random check with callback_deflection_rate probability.
        #     probability of accepting the callback and leaving the QP queue.
        #     This models the real telephony IVR that offers "press 1 for callback"
        #     and explains why many unserved calls leave at ~55s in historical data.
        if callback_deflection_time > 0 and callback_deflection_rate > 0 and state.queue:
            deflected = []
            remaining = []
            for c in state.queue:
                wait = (state.current_time - c.arrival_time).total_seconds()
                # Only offer callback ONCE per call, when wait first crosses threshold
                if wait >= callback_deflection_time and c.call_id not in _callback_offered:
                    _callback_offered.add(c.call_id)
                    if nqp_rng.random() < callback_deflection_rate:
                        deflected.append(c)
                        continue
                remaining.append(c)
            if deflected:
                _callback_deflections += len(deflected)
                for c in deflected:
                    # Deflected calls go to FIFO (telephony callback = will be served later)
                    if enable_fifo:
                        state.fifo_queue.append(c)
                    else:
                        state.unserved_calls.append({
                            'call_id': c.call_id,
                            'reason': 'callback_deflection',
                            'wait_time': (state.current_time - c.arrival_time).total_seconds()
                        })
                state.queue = remaining
        # 3. Remove expired QP calls → move to FIFO queue (telephony takes over)
        expired = state.remove_expired(to_fifo=enable_fifo)

        # 3b. Remove FIFO calls where customer abandoned (total wait > fifo_max_wait)
        if enable_fifo:
            state.remove_fifo_abandoned(fifo_max_wait=fifo_max_wait)

        # 4. PRIORITY 1 — QPlanner: solve optimal assignments
        available_ops = state.get_available_operators()

        # Apply operator shift constraints (only operators currently on shift)
        if operator_shifts and available_ops:
            available_ops = [
                op for op in available_ops
                if op.username in operator_shifts
                and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
            ]

        # Apply occupancy: task-based non-QP work model
        current_hour = state.current_time.hour
        effective_occ = hourly_occupancy.get(current_hour, base_occupancy) if hourly_occupancy else base_occupancy
        if effective_occ > 0 and available_ops:
            mean_avail_s = nqp_mean_task_s * (1 - effective_occ) / max(effective_occ, 0.01)
            truly_available = []
            for op in available_ops:
                nqp_end = nqp_busy_until.get(op.username)
                if nqp_end and nqp_end > state.current_time:
                    continue
                p_start = min(0.95, tick_interval / max(mean_avail_s, 1.0))

                if nqp_rng.random() < p_start:
                    task_dur = max(float(tick_interval), nqp_rng.exponential(nqp_mean_task_s))
                    nqp_busy_until[op.username] = state.current_time + timedelta(seconds=task_dur)
                    continue

                truly_available.append(op)
            available_ops = truly_available

        # Track score coverage before solver call
        if state.queue and available_ops:
            _tick_pairs = len(state.queue) * len(available_ops)
            _tick_hits = sum(
                1 for c in state.queue for op in available_ops
                if (c.call_id, op.username) in score_lookup
            )
            _total_solver_pairs += _tick_pairs
            _total_solver_hits += _tick_hits
            _tick_ops_log.append((len(available_ops), len(state.queue)))

        # QPlanner solver — two modes:
        #   Standard: KAPPA gate naturally rejects calls where FO ≤ 0
        #   Two-tier: split queue by call value, reserve operators for Tier 1
        if state.queue and available_ops:
            if _use_tiered:
                pairs = solve_tiered(
                    queue=state.queue,
                    available_ops=available_ops,
                    score_lookup=score_lookup,
                    churn_i_lookup=churn_i_lookup,
                    current_time=state.current_time,
                    fo_config=fo_config,
                    fallback_churn_ij=fallback_churn_ij,
                    call_discounts=_call_discounts,
                )
            else:
                pairs = solve_assignments(
                    queue=state.queue,
                    available_ops=available_ops,
                    score_lookup=score_lookup,
                    churn_i_lookup=churn_i_lookup,
                    current_time=state.current_time,
                    fo_config=fo_config,
                    fallback_churn_ij=fallback_churn_ij,
                    call_discounts=_call_discounts,
                )
            # With delivery_failure_rate > 0: each assignment may fail
            # (operator went busy during pipeline latency → AgentNotAvailable)
            # Failed calls → FIFO queue (same as production behavior)
            for call, op, churn_ij, score in pairs:
                if pipeline_latency_std > 0 and pipeline_latency > 0:
                    mu = pipeline_latency
                    sigma = pipeline_latency_std
                    sigma2_ln = np.log(1 + (sigma / mu) ** 2)
                    mu_ln = np.log(mu) - sigma2_ln / 2
                    lat_sample = max(5.0, nqp_rng.lognormal(mu_ln, np.sqrt(sigma2_ln)))
                else:
                    lat_sample = pipeline_latency
                # Delivery failure check
                if delivery_failure_rate > 0 and nqp_rng.random() < delivery_failure_rate:
                    # Record failed delivery
                    state.failed_deliveries.append({
                        'call_id': call.call_id,
                        'operator': op.username,
                        'reason': 'delivery_failure',
                        'wait_time': (state.current_time - call.arrival_time).total_seconds(),
                        'score': score,
                    })
                    # Mark operator as busy for pipeline_latency period:
                    # In production, delivery failed because the operator WAS busy
                    # (AgentNotAvailable). Model this by blocking the operator briefly.
                    _fail_busy_s = max(lat_sample, nqp_rng.exponential(nqp_mean_task_s * 0.3))
                    op.status = "busy"
                    op.busy_until = state.current_time + timedelta(seconds=_fail_busy_s)
                    # Move failed call to FIFO queue (telephony takes over)
                    if enable_fifo:
                        state.fifo_queue.append(call)
                    else:
                        state.unserved_calls.append({
                            'call_id': call.call_id,
                            'reason': 'delivery_failure',
                            'wait_time': (state.current_time - call.arrival_time).total_seconds()
                        })
                    # Remove call from QP queue (it's been handled, even if failed)
                    state.queue = [c for c in state.queue if c.call_id != call.call_id]
                    continue
                # Apply after-call work (ACW): operator stays busy after call ends
                acw_sample = nqp_rng.exponential(acw_mean_s) if acw_mean_s > 0 else 0.0
                state.execute_assignment(call, op, churn_ij, score,
                                         pipeline_latency=lat_sample,
                                         acw_time=acw_sample)

        # 6. PRIORITY 2 — FIFO telephony routing: remaining operators serve FIFO queue
        #    fifo_routing_fraction < 1.0 models ACD pacing: not all operators instantly
        #    get a telephony call, leaving a buffer for QP's next tick.
        if enable_fifo and state.fifo_queue:
            fifo_avail = state.get_available_operators()
            # Apply shift constraints to FIFO operators too
            if operator_shifts and fifo_avail:
                fifo_avail = [
                    op for op in fifo_avail
                    if op.username in operator_shifts
                    and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
                ]
            # ACD pacing: only a fraction of available operators serve FIFO each tick
            if fifo_routing_fraction < 1.0 and fifo_avail:
                n_fifo_ops = max(1, int(len(fifo_avail) * fifo_routing_fraction))
                fifo_avail = fifo_avail[:n_fifo_ops]
            if fifo_avail:
                state.route_fifo(fifo_avail, score_lookup, fifo_latency, block_operators=fifo_block_operators)

        # Log progress
        if verbose and state.tick % log_every == 0:
            n_busy = sum(1 for op in state.operators.values() if op.status == "busy")
            fifo_info = f" | fifo_q={len(state.fifo_queue):>3} | fifo_done={len(state.fifo_assignments):>4}" if enable_fifo else ""
            print(
                f"  tick={state.tick:>5} | time={state.current_time.strftime('%H:%M:%S')} | "
                f"queue={len(state.queue):>3} | assigned={len(state.assignments):>5} | "
                f"busy_ops={n_busy:>3}/{len(state.operators)}{fifo_info}"
            )
        state.current_time += timedelta(seconds=tick_interval)

        # Early exit if all calls processed and both queues empty
        if next_call_idx >= total_calls and not state.queue and not state.fifo_queue:
            break

    # Any calls still in FIFO queue at end → truly unserved
    for c in state.fifo_queue:
        state.unserved_calls.append({
            'call_id': c.call_id,
            'reason': 'fifo_end_of_day',
            'wait_time': (state.current_time - c.arrival_time).total_seconds()
        })
    state.fifo_queue = []

    # Store score coverage stats on state for aggregation
    state.score_coverage = {
        'total_pairs': _total_solver_pairs,
        'lookup_hits': _total_solver_hits,
        'fallback_used': _total_solver_pairs - _total_solver_hits,
        'coverage_pct': _total_solver_hits / _total_solver_pairs * 100 if _total_solver_pairs > 0 else 0,
    }
    state.tick_ops_log = _tick_ops_log  # (n_available_ops, n_queued_calls) per solver tick
    state.callback_deflections = _callback_deflections  # total calls deflected by callback offer

    if verbose:
        print(f"\n✅ Simulation complete!")
        print(f"   QP assignments:   {len(state.assignments)}")
        print(f"   Failed deliveries: {len(state.failed_deliveries)}")
        print(f"   Callback deflect.: {_callback_deflections}")
        print(f"   FIFO assignments: {len(state.fifo_assignments)}")
        print(f"   Total served:     {len(state.assignments) + len(state.fifo_assignments)}")
        print(f"   Truly unserved:   {len(state.unserved_calls)}")
        if state.unserved_calls:
            reasons = {}
            for u in state.unserved_calls:
                r = u.get('reason', 'unknown')
                reasons[r] = reasons.get(r, 0) + 1
            for r, cnt in sorted(reasons.items()):
                print(f"     ↳ {r}: {cnt}")
        if state.assignments:
            waits = [a.waiting_time for a in state.assignments]
            reductions = [a.churn_reduction for a in state.assignments]
            print(f"   Median wait:      {np.median(waits):.1f}s")
            print(f"   Mean Δchurn:      {np.mean(reductions):.4f}")
        if state.fifo_assignments:
            fifo_waits = [a.waiting_time for a in state.fifo_assignments]
            print(f"   FIFO avg wait: {np.mean(fifo_waits):.1f}s (median: {np.median(fifo_waits):.1f}s)")
        if _total_solver_pairs > 0:
            _fb = _total_solver_pairs - _total_solver_hits
            print(f"   Score coverage: {_total_solver_hits:,}/{_total_solver_pairs:,} "
                  f"({_total_solver_hits/_total_solver_pairs*100:.1f}%) | "
                  f"fallback: {_fb:,} pairs ({_fb/_total_solver_pairs*100:.1f}%)")
    return state

<a id="full-score-coverage"></a>
## Full Score Coverage: Panel + Model Re-scoring

The sparse `score_lookup` from brains historical data only covers ~13% of (call, operator) pairs — the solver only scored operators that happened to be available at each historic tick. In production, smartpairing creates a **cross join** of ALL queued calls × ALL available operators and runs `model.predict()` on every pair.

This section:
1. Loads **production feature panels** from BigQuery (customer + operator features at mid-January)
2. Loads the **smartpairing MLflow model** (`smartpairing_ebm_vanilla@champion`)
3. Scores **ALL** (call, operator) pairs per day → `full_score_lookup` (100% coverage)
4. Validates distributions against the original sparse lookup
5. **Overrides** `score_lookup` so the simulation uses full coverage
6. Computes a **fallback** (median churn_ij) for any remaining unscored pairs

In [ ]:
# =============================================================================
# CELL 12b — LOAD PRODUCTION PANELS + SCORE ALL PAIRS VIA MLFLOW MODEL
# =============================================================================
# INPUT:
#   - df_call_events (12,302 calls), df_operators (~110 operators)
#   - score_lookup (sparse, ~102k pairs from brains history)
#   - churn_i_lookup (from cell 7)
#   - _day_setups OR df_calls_qplanner (for identifying per-day operators)
#
# DOES:
#   1. Queries BQ production panels (customer + operator features) for mid-January
#   2. Loads smartpairing_ebm_vanilla@champion model from MLflow
#   3. For EACH simulation day, builds a cross join of (calls × operators)
#   4. Enriches with panel features and runs model.predict()
#   5. Builds full_score_lookup (781k+ entries, 100% coverage)
#   6. Builds full_churn_i_lookup (ghost operator, all NaN op features)
#
# OUTPUT:
#   - full_score_lookup: dict (call_id, operator) → churn_ij (all pairs)
#   - full_churn_i_lookup: dict call_id → churn_i (no-operator baseline)
# =============================================================================
import os
from tqdm.auto import tqdm
from google.cloud import bigquery as _bq
# from crm_qp_lib.data_reader import impersonate_credentials  # internal — removed

_bq_creds = impersonate_credentials()
_bq_client = _bq.Client(credentials=_bq_creds)

# ─── 1. LOAD CUSTOMER PANEL ─────────────────────────────────────────────────
_PANEL_TABLE_CUST = "t_customers_panel"
_PANEL_TABLE_OPS  = "t_operators_panel_metrics"

# Find the latest valid partition date in January 2026
_partition_query = f"""
SELECT MAX(SAFE.PARSE_DATE('%Y%m%d', partition_id)) as latest_partition
FROM `YOUR_GCP_PROJECT.internal.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = '{_PANEL_TABLE_CUST}'
  AND REGEXP_CONTAINS(partition_id, r'^[0-9]{{8}}$')
  AND SAFE.PARSE_DATE('%Y%m%d', partition_id) BETWEEN '2026-01-01' AND '2026-01-31'
"""
_part_df = _bq_client.query(_partition_query).to_dataframe()
_panel_date = str(_part_df['latest_partition'].iloc[0])
print(f"📅 Panel date: {_panel_date}")

# Load customer panel
_panel_cust_query = f"""
SELECT * FROM `YOUR_GCP_PROJECT.internal.{_PANEL_TABLE_CUST}`
WHERE AVAILABLE_TIME = '{_panel_date}'
"""
_df_panel_cust = _bq_client.query(_panel_cust_query).to_dataframe()
_df_panel_cust = _df_panel_cust.rename(columns={'SA_COD': 'customer_value_sa_cod'})
_cust_ids = set(df_call_events['SA_COD'].astype(str))
_panel_cust_ids = set(_df_panel_cust['customer_value_sa_cod'].astype(str))
_cust_coverage = len(_cust_ids & _panel_cust_ids) / len(_cust_ids) * 100

# Load operator panel
_panel_ops_query = f"""
SELECT * FROM `YOUR_GCP_PROJECT.internal.{_PANEL_TABLE_OPS}`
WHERE AVAILABLE_TIME = '{_panel_date}'
"""
_df_panel_ops = _bq_client.query(_panel_ops_query).to_dataframe()
_df_panel_ops = _df_panel_ops.rename(
    columns={'HIST_USER_COD': 'wrgo_ad_user_name'}
)
_df_panel_ops['wrgo_ad_user_name'] = _df_panel_ops['wrgo_ad_user_name'].str.lower()
_op_ids = set(df_operators['agent_username'].str.lower())
_panel_op_ids = set(_df_panel_ops['wrgo_ad_user_name'].str.lower())
_ops_coverage = len(_op_ids & _panel_op_ids) / len(_op_ids) * 100

print(f"📋 Customer panel: {len(_df_panel_cust):,} rows ({_cust_coverage:.1f}% of sim customers)")
print(f"📋 Operator panel: {len(_df_panel_ops):,} rows ({_ops_coverage:.1f}% of sim operators)")

# ─── 2. LOAD MLFLOW MODEL ───────────────────────────────────────────────────
# from crm_qp_lib.inference.mlflow import MLflowAuthenticator  # internal — removed
import mlflow as _unauthenticated_mlflow

# Clear cached auth so we get a fresh token for the right audience
if "CLUPA_LAST_MLFLOW_AUTH_TIME" in os.environ:
    del os.environ["CLUPA_LAST_MLFLOW_AUTH_TIME"]

_GCP_PROJECT = "YOUR_GCP_PROJECT"
_MLFLOW_URI  = "https://YOUR_MLFLOW_SERVER"

_auth_mlflow = MLflowAuthenticator(_GCP_PROJECT).authenticate_mlflow(_unauthenticated_mlflow)
_auth_mlflow.set_tracking_uri(_MLFLOW_URI)
os.environ["MLFLOW_TRACKING_URI"] = _MLFLOW_URI

_run_id = _auth_mlflow.MlflowClient().get_model_version_by_alias(
    "smartpairing_ebm_vanilla", "champion"
).run_id
_sp_model = _auth_mlflow.pyfunc.load_model(f"runs:/{_run_id}/pipeline")
_feats = list(
    _sp_model.unwrap_python_model().model.named_steps["pre_processing"]
    .steps[-1][1].column_order
)
print(f"🤖 Model loaded: smartpairing_ebm_vanilla@champion (run_id={_run_id[:12]}…)")
print(f"   Features ({len(_feats)}): {_feats[:5]}…")

# ─── 3. BUILD FEATURE COLUMN MAPPING (case-insensitive) ─────────────────────
_cust_feats = [f for f in _feats if not f.startswith('ohp_')]
_ops_feats  = [f for f in _feats if f.startswith('ohp_')]

# BQ columns are ALL UPPERCASE; model features have lowercase prefix + UPPERCASE suffix
_cust_col_map = {}
_bq_cust_cols_upper = {c.upper(): c for c in _df_panel_cust.columns}
for feat in _cust_feats:
    if feat.upper() in _bq_cust_cols_upper:
        _cust_col_map[_bq_cust_cols_upper[feat.upper()]] = feat

_ops_col_map = {}
_bq_ops_cols_upper = {c.upper(): c for c in _df_panel_ops.columns}
for feat in _ops_feats:
    if feat.upper() in _bq_ops_cols_upper:
        _ops_col_map[_bq_ops_cols_upper[feat.upper()]] = feat

print(f"   Customer features matched: {len(_cust_col_map)}/{len(_cust_feats)}")
print(f"   Operator features matched: {len(_ops_col_map)}/{len(_ops_feats)}")
assert len(_cust_col_map) == len(_cust_feats), f"Missing customer features!"
assert len(_ops_col_map) == len(_ops_feats), f"Missing operator features!"

# Build panel subsets with renamed columns
_panel_c = _df_panel_cust[['customer_value_sa_cod'] + list(_cust_col_map.keys())].copy()
_panel_c = _panel_c.rename(columns=_cust_col_map)
_panel_c['customer_value_sa_cod'] = _panel_c['customer_value_sa_cod'].astype(str)

_panel_o = _df_panel_ops[['wrgo_ad_user_name'] + list(_ops_col_map.keys())].copy()
_panel_o = _panel_o.rename(columns=_ops_col_map)
# Keep original case — must match agent_username from get_day_operators

# ─── 4. SCORE ALL PAIRS PER DAY ─────────────────────────────────────────────
# Pre-compute per-day setups (same as cell 13 will do)
_tmp_qp_ts = df_calls_qplanner.copy()
_tmp_qp_ts['_date'] = pd.to_datetime(_tmp_qp_ts['START_DATE_TIME']).dt.date
_tmp_days = sorted(_tmp_qp_ts['_date'].unique())

full_score_lookup = {}
full_churn_i_lookup = {}

for day in tqdm(_tmp_days, desc="Scoring all pairs"):
    day_str = str(day)
    # Day's calls
    _day_events = df_call_events[df_call_events['arrival_time'].dt.date == day]
    if _day_events.empty:
        continue
    # Day's operators
    _day_ops_df, _ = get_day_operators(df_calls_qplanner, day_str)
    if _day_ops_df.empty:
        continue

    # Cross join — preserve original agent_username case (same as simulation uses)
    _calls = _day_events[['CALL_ID', 'SA_COD']].copy()
    _calls['SA_COD'] = _calls['SA_COD'].astype(str)
    _calls['_key'] = 1
    _ops = _day_ops_df[['agent_username']].copy()
    _ops['_key'] = 1
    _cross = _calls.merge(_ops, on='_key').drop(columns='_key')
    _cross = _cross.rename(columns={'SA_COD': 'customer_value_sa_cod',
                                     'agent_username': 'wrgo_ad_user_name'})

    # Enrich with panel features
    _cross = _cross.merge(_panel_c, on='customer_value_sa_cod', how='left')
    _cross = _cross.merge(_panel_o, on='wrgo_ad_user_name', how='left')

    # Predict churn_ij
    _vals = _sp_model.predict(_cross[_feats])
    _keys = list(zip(_cross['CALL_ID'], _cross['wrgo_ad_user_name']))
    full_score_lookup.update(dict(zip(_keys, _vals)))

    # churn_i (ghost operator — all NaN operator features)
    _ci_df = _day_events[['CALL_ID', 'SA_COD']].drop_duplicates(subset='CALL_ID').copy()
    _ci_df['SA_COD'] = _ci_df['SA_COD'].astype(str)
    _ci_df = _ci_df.rename(columns={'SA_COD': 'customer_value_sa_cod'})
    _ci_df = _ci_df.merge(_panel_c, on='customer_value_sa_cod', how='left')
    for f in _ops_feats:
        _ci_df[f] = np.nan
    _ci_vals = _sp_model.predict(_ci_df[_feats])
    full_churn_i_lookup.update(dict(zip(_ci_df['CALL_ID'], _ci_vals)))

# ─── 5. REPORT ──────────────────────────────────────────────────────────────
_n_needed = sum(
    len(df_call_events[df_call_events['arrival_time'].dt.date == d]) *
    len(get_day_operators(df_calls_qplanner, str(d))[0])
    for d in _tmp_days
    if not df_call_events[df_call_events['arrival_time'].dt.date == d].empty
    and not get_day_operators(df_calls_qplanner, str(d))[0].empty
)
print(f"\n✅ Full score lookup built:")
print(f"   Pairs scored:    {len(full_score_lookup):,} / {_n_needed:,} ({len(full_score_lookup)/_n_needed*100:.1f}%)")
print(f"   churn_i entries: {len(full_churn_i_lookup):,}")
print(f"   Original sparse: {len(score_lookup):,} ({len(score_lookup)/len(full_score_lookup)*100:.1f}% of full)")

# Sample comparison
_common = set(score_lookup.keys()) & set(full_score_lookup.keys())
if _common:
    _sample_k = list(_common)[:5]
    print(f"\n   Sample comparison (orig → full):")
    for k in _sample_k:
        print(f"     {k[0][:20]}… × {k[1][:20]}…: {score_lookup[k]:.4f} → {full_score_lookup[k]:.4f}")

In [ ]:
# =============================================================================
# CELL 12c — SANITY CHECK: Score Distribution (sparse vs full)
# =============================================================================
import matplotlib.pyplot as plt

_orig_vals = np.array(list(score_lookup.values()))
_full_vals = np.array(list(full_score_lookup.values()))
_ci_orig   = np.array(list(churn_i_lookup.values()))
_ci_full   = np.array(list(full_churn_i_lookup.values()))

# Find overlapping keys for direct comparison
_common_keys = set(score_lookup.keys()) & set(full_score_lookup.keys())
_orig_common = np.array([score_lookup[k] for k in _common_keys])
_full_common = np.array([full_score_lookup[k] for k in _common_keys])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Panel 1: churn_ij distributions ---
ax = axes[0, 0]
_bins = np.linspace(0, 1, 60)
ax.hist(_orig_vals, bins=_bins, alpha=0.6, label=f'Original (n={len(_orig_vals):,})', color='steelblue', density=True)
ax.hist(_full_vals, bins=_bins, alpha=0.6, label=f'Full (n={len(_full_vals):,})', color='coral', density=True)
ax.set_xlabel('churn_ij'); ax.set_ylabel('Density')
ax.set_title('churn_ij Distribution')
ax.axvline(_orig_vals.mean(), color='steelblue', ls='--', lw=1)
ax.axvline(_full_vals.mean(), color='coral', ls='--', lw=1)
ax.legend(fontsize=8)

# --- Panel 2: churn_i distributions ---
ax = axes[0, 1]
ax.hist(_ci_orig, bins=_bins, alpha=0.6, label=f'Original churn_i (n={len(_ci_orig):,})', color='steelblue', density=True)
ax.hist(_ci_full, bins=_bins, alpha=0.6, label=f'Full churn_i (n={len(_ci_full):,})', color='coral', density=True)
ax.set_xlabel('churn_i'); ax.set_ylabel('Density')
ax.set_title('churn_i Distribution (baseline, no operator)')
ax.legend(fontsize=8)

# --- Panel 3: scatter of overlapping pairs ---
ax = axes[1, 0]
_sample_n = min(10000, len(_orig_common))
_idx = np.random.choice(len(_orig_common), _sample_n, replace=False)
ax.scatter(_orig_common[_idx], _full_common[_idx], alpha=0.15, s=4, color='purple')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('Original churn_ij (production)')
ax.set_ylabel('Full churn_ij (panel re-scored)')
ax.set_title(f'Overlapping pairs (n={len(_common_keys):,}, showing {_sample_n:,})')
_corr = np.corrcoef(_orig_common, _full_common)[0, 1]
_mae = np.abs(_orig_common - _full_common).mean()
ax.text(0.05, 0.92, f'r = {_corr:.4f}\nMAE = {_mae:.4f}', transform=ax.transAxes,
        fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Panel 4: summary stats table ---
ax = axes[1, 1]; ax.axis('off')
_stats = [
    ['Metric', 'Original', 'Full', 'Delta'],
    ['Count', f'{len(_orig_vals):,}', f'{len(_full_vals):,}', f'+{len(_full_vals)-len(_orig_vals):,}'],
    ['Coverage', f'{len(_orig_vals)/len(_full_vals)*100:.1f}%', '100.0%', ''],
    ['Mean churn_ij', f'{_orig_vals.mean():.4f}', f'{_full_vals.mean():.4f}', f'{_full_vals.mean()-_orig_vals.mean():+.4f}'],
    ['Median', f'{np.median(_orig_vals):.4f}', f'{np.median(_full_vals):.4f}', f'{np.median(_full_vals)-np.median(_orig_vals):+.4f}'],
    ['Std', f'{_orig_vals.std():.4f}', f'{_full_vals.std():.4f}', ''],
    ['Overlap corr', f'{_corr:.4f}', '', ''], ['Overlap MAE', f'{_mae:.4f}', '', ''],
]
_table = ax.table(cellText=_stats, loc='center', cellLoc='center')
_table.auto_set_font_size(False); _table.set_fontsize(9); _table.scale(1, 1.5)
for (row, col), cell in _table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#4472C4')
        cell.set_text_props(color='white', fontweight='bold')
ax.set_title('Score Distribution Summary', fontsize=11, fontweight='bold')

plt.suptitle('Score Lookup: Original (sparse) vs Full (panel re-scored)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Verdict
print(f"\nVerdict: r = {_corr:.4f}, MAE = {_mae:.4f}")
if _corr > 0.85:
    print(f"  Good match — safe to use full lookup for simulation")
else:
    print(f"  ⚠️ Low correlation — investigate feature mismatches")

In [ ]:
# =============================================================================
# CELL 12d — OVERRIDE: Use full score lookup for simulation
# =============================================================================
# STRATEGY:
#   1. Replace score_lookup with full_score_lookup (100% of known pairs)
#   2. Merge churn_i_lookup: keep original brains values (day-accurate),
#      fill gaps from panel re-scoring
#   3. Compute fallback_churn_ij = median of full scores for ANY remaining
#      missing pairs at simulation time (customers/operators not in panel)
#
# FALLBACK CRITERIA:
#   Pairs missing from score_lookup at simulation time are rare edge cases:
#     - Customers not in the BQ panel (~6%)
#     - Operators not in the BQ panel (~21.8%)
#   For these, we assign the MEDIAN churn_ij from the full lookup.
#   Rationale: median is a "neutral" estimate — the pair is treated as
#   having average compatibility. This is conservative (no false signal)
#   and much better than the previous approach of marking them infeasible
#   (cost=1e6), which prevented the solver from considering those operators.
# =============================================================================

# Preserve sparse lookup for comparison in later diagnostic cells
sparse_score_lookup = score_lookup.copy()
sparse_churn_i_lookup = churn_i_lookup.copy()

# Override score_lookup with full coverage
score_lookup = full_score_lookup.copy()

# Merge churn_i: keep original (more accurate), add new entries from panel
for k, v in full_churn_i_lookup.items():
    if k not in churn_i_lookup:
        churn_i_lookup[k] = v

# Compute fallback for remaining missing pairs
fallback_churn_ij = float(np.median(list(score_lookup.values())))

# ── Per-day coverage report (matches what the simulation actually evaluates) ──
# The simulation evaluates calls × operators PER DAY, not globally.
# So coverage should be measured per-day, not all-calls × all-operators.
_tmp_qp_cov = df_calls_qplanner.copy()
_tmp_qp_cov['_date'] = pd.to_datetime(_tmp_qp_cov['START_DATE_TIME']).dt.date
_cov_days = sorted(_tmp_qp_cov['_date'].unique())

_total_day_pairs = 0
_total_day_hits = 0

print("=" * 80)
print("  SCORE LOOKUP OVERRIDE — Full Coverage Active")
print("=" * 80)
print(f"  score_lookup:      {len(score_lookup):,} entries  (was {len(sparse_score_lookup):,})")
print(f"  churn_i_lookup:    {len(churn_i_lookup):,} entries")
print(f"  fallback_churn_ij: {fallback_churn_ij:.4f}  (median of full lookup)")
print()
print(f"  {'Day':<14} {'Calls':>7} {'Ops':>5} {'Pairs':>10} {'Hits':>10} {'Miss':>7} {'Cov%':>7}")
print(f"  {'─'*54}")
for _d in _cov_days:
    _day_events = df_call_events[df_call_events['arrival_time'].dt.date == _d]
    _day_ops_df, _ = get_day_operators(df_calls_qplanner, str(_d))
    if _day_events.empty or _day_ops_df.empty:
        continue
    _day_calls = set(_day_events['CALL_ID'])
    _day_ops = set(_day_ops_df['agent_username'])
    _n_pairs = len(_day_calls) * len(_day_ops)
    _n_hits = sum(1 for c in _day_calls for o in _day_ops if (c, o) in score_lookup)
    _n_miss = _n_pairs - _n_hits
    _total_day_pairs += _n_pairs
    _total_day_hits += _n_hits
    _pct = _n_hits / _n_pairs * 100 if _n_pairs > 0 else 0
    print(f"  {str(_d):<14} {len(_day_calls):>7,} {len(_day_ops):>5} {_n_pairs:>10,} {_n_hits:>10,} {_n_miss:>7,} {_pct:>6.1f}%")

_total_miss = _total_day_pairs - _total_day_hits
_total_pct = _total_day_hits / _total_day_pairs * 100 if _total_day_pairs > 0 else 0
print(f"  {'─'*54}")
print(f"  {'TOTAL':<14} {'':>7} {'':>5} {_total_day_pairs:>10,} {_total_day_hits:>10,} {_total_miss:>7,} {_total_pct:>6.1f}%")
print()
print(f"  → At simulation time: {_total_miss:,} pairs will use fallback ({_total_day_pairs - _total_day_hits} / {_total_day_pairs:,})")
print(f"  Fallback criteria: median churn_ij = {fallback_churn_ij:.4f}")
print(f"  → Treats missing pairs as 'average compatibility' (neutral, conservative)")
print(f"  → Much better than cost=1e6 (infeasible), which excluded operators entirely")
print()
print(f"✅ score_lookup now uses FULL panel-rescored data")
print(f"   All downstream cells (simulation, experiments, diagnostics) will use it")

<a id="calibrated-parameters"></a>
## Cell 13a: Calibrated Parameters & Day Setup

In [ ]:
# =============================================================================
# CELL 13a — CALIBRATED PARAMETERS & DAY SETUP
# =============================================================================
# INPUT:
#   - DATA_TICK_MEDIAN, DATA_PIPELINE_MEAN, DATA_PIPELINE_STD (from cell 7)
#   - df_calls_qplanner, df_call_events (from cells 2-3)
#
# DOES:
#   1. Defines all calibrated simulation parameters (data-derived constants)
#   2. Identifies QPlanner days and the busiest day (see calls_per_day output)
#
# OUTPUT:
#   - CALIBRATED_* constants (used by calibration sweeps and simulation)
#   - df_calls_qplanner_ts, qp_days, busiest_day, calls_per_day
#
# NOTE: CALIBRATED_OCC_SCALE and CALIBRATED_CHURN_BIAS are derived by the
#   calibration cells that follow. Set _RUN_*_CALIBRATION = True to re-derive.
# =============================================================================

# ─── CALIBRATED PARAMETERS (all data-derived) ───────────────────────────────
#
# These parameters are derived from historical data and calibration procedures.
# Two parameters require special calibration runs (OCC_SCALE, CHURN_BIAS):
#
# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CALIBRATED_OCC_SCALE (occupancy scaling factor)                            │
# │ ─────────────────────────────────────────────────────────────────────────── │
# │ Derived from: OCC_SCALE sweep (see calibration cell below).                │
# │ Method: Coarse grid [0.5–4.0] + bisection refinement to find the value    │
# │   where simulated QP served calls ≈ historical QP served calls.           │
# │ Result: see CALIBRATED_OCC_SCALE below (update after running calibration).│
# │ Interpretation: Raw occupancy (1 - active_ops/total_ops) underestimates   │
# │   true operator busyness. Scaling by 1.27× corrects for tasks not visible │
# │   in the call data (admin, breaks, cross-team work).                      │
# │ Sensitivity: Sharp transition between 1.0 (+15%) and 1.5 (-15%).          │
# │ Re-calibrate when: operator pool or shift patterns change significantly.  │
# └─────────────────────────────────────────────────────────────────────────────┘
#
# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CALIBRATED_CHURN_BIAS (EBM model calibration correction)                   │
# │ ─────────────────────────────────────────────────────────────────────────── │
# │ Derived from: Churn Calibration Diagnostic (see calibration cell below).   │
# │ Method: Compare EBM model P(churn) predictions (full_score_lookup) vs      │
# │   actual RESOL_MOT_DSC outcomes on historical QP-assigned pairs.          │
# │ Result: see CALIBRATED_CHURN_BIAS below (update after running calibration)│
# │ Correction: Additive — subtract bias from all churn_ij values.            │
# │   Applied in validation (Cell 36) as: churn_ij_corrected = churn_ij - bias│
# │ Interpretation: The EBM model (smartpairing_ebm_vanilla@champion) has a   │
# │   systematic upward bias, likely from training data distribution shift.    │
# │ Brier score: reported by calibration cell output.                         │
# │ Re-calibrate when: EBM model is retrained or champion version changes.    │
# └─────────────────────────────────────────────────────────────────────────────┘
# ─────────────────────────────────────────────────────────────────────────────

# --- Calibration run guards (set True to re-run calibration sweeps) ---
_RUN_OCC_CALIBRATION = False     # ← Set True to re-run OCC_SCALE sweep (~20 min)
_RUN_CHURN_CALIBRATION = False   # ← Set True to re-run churn calibration

# --- Timing & latency (from data distributions) ---
CALIBRATED_TICK = int(round(DATA_TICK_MEDIAN))
CALIBRATED_LATENCY = int(round(DATA_PIPELINE_MEAN))
CALIBRATED_LATENCY_STD = int(round(DATA_PIPELINE_STD))
CALIBRATED_NQP_TASK_S = 1200

# --- Operator availability calibration (from OCC_SCALE sweep) ---
CALIBRATED_OCC_SCALE = 1.2656  # sweep result: sim QP served ≈ historical 5,407 (gap=-0.2%)

# --- Churn model calibration (from churn calibration diagnostic) ---
CALIBRATED_CHURN_BIAS = 0.0288  # additive bias: model overestimates by +2.88pp (pred=32.10%, actual=29.22%)

# --- FIFO routing ---
CALIBRATED_FIFO_LATENCY = 2.0  # telephony FIFO routing: ~2s (no ML, just ACD)
CALIBRATED_FIFO_BLOCK = False  # FIFO is non-blocking: churn estimation only, no operator hold

# --- Call flow adjustments (from historical data) ---
CALIBRATED_DELIVERY_FAILURE_RATE = 0.142  # from 890/6297 historical failed deliveries (AgentNotAvailable)
CALIBRATED_CALLBACK_DEFLECTION_TIME = 55.0  # seconds — telephony offers callback at ~55s wait
CALIBRATED_CALLBACK_DEFLECTION_RATE = 0.12   # fraction of queued calls that accept callback at deflection time
CALIBRATED_ACW_MEAN_S = 45.0  # mean after-call work time (seconds). Exponentially distributed. Typical: 30-60s.

print(f"📐 Calibrated parameters (data-derived):")
print(f"   tick_interval = {CALIBRATED_TICK}s (median request cadence)")
print(f"   pipeline_latency μ = {CALIBRATED_LATENCY}s, σ = {CALIBRATED_LATENCY_STD}s (from real overhead)")
print(f"   occ_scale = {CALIBRATED_OCC_SCALE} (from sweep → sim QP served ≈ historical)")
print(f"   churn_bias = {CALIBRATED_CHURN_BIAS} (EBM model overestimates by +{CALIBRATED_CHURN_BIAS*100:.2f}pp)")
print(f"   FIFO: latency={CALIBRATED_FIFO_LATENCY}s, block_operators={CALIBRATED_FIFO_BLOCK}")
print(f"   delivery_failure_rate = {CALIBRATED_DELIVERY_FAILURE_RATE:.1%} (from historical AgentNotAvailable)")
print(f"   acw_mean = {CALIBRATED_ACW_MEAN_S:.0f}s (after-call work — delays operator re-availability)")
print(f"   max_wait_time = 180s (fixed QP window, agreed with operations)\n")

# --- Identify all QPlanner days ---
df_calls_qplanner_ts = df_calls_qplanner.copy()
df_calls_qplanner_ts['_date'] = pd.to_datetime(df_calls_qplanner_ts['START_DATE_TIME']).dt.date
qp_days = sorted(df_calls_qplanner_ts['_date'].unique())
calls_per_day = df_call_events.groupby(df_call_events['arrival_time'].dt.date).size()
busiest_day = str(calls_per_day.idxmax())
print(f"📅 Found {len(qp_days)} QPlanner days: {qp_days[0]} → {qp_days[-1]}")
print(f"   Busiest day: {busiest_day} ({calls_per_day.max()} calls)")

<a id="occ-calibration"></a>
## Calibration Procedures — Deriving simulation parameters (skipped by default)

The cells below derive `CALIBRATED_OCC_SCALE` and `CALIBRATED_CHURN_BIAS` from data.
These are **one-time calibration procedures** — the extracted parameter values are hardcoded in Cell 13a above.

**Execution order:** These cells run their own internal simulations (OCC_SCALE sweep) or compare
model predictions against historical outcomes (churn calibration). They only need the setup from
Cell 13a (parameters + `qp_days`), not the main simulation outputs.

**When to re-run:**
- OCC_SCALE sweep: when operator pool size or shift patterns change significantly
- Churn calibration: when the EBM model (`smartpairing_ebm_vanilla@champion`) is retrained

Set `_RUN_CALIBRATION = True` in each cell to re-execute.

In [ ]:
# =============================================================================
# OCC_SCALE SWEEP — Find occupancy scale that calibrates sim to historical
# =============================================================================
# ⚠️ CALIBRATION CELL — Run once to derive CALIBRATED_OCC_SCALE.
#   Set _RUN_OCC_CALIBRATION = True to re-run (~20 min).
#   Update CALIBRATED_OCC_SCALE in Cell 13a with the result.
# =============================================================================
# Rationale: FO scores and churn_ij are nearly identical between sparse/full
# lookups, so the gap in QP served calls is NOT from scoring differences —
# it comes from operators being available too often in simulation.
# This sweep finds the OCC_SCALE value where sim QP served ≈ historical.
#
# Approach: Coarse grid sweep [0.5–4.0], then bisection refinement.
# Each run ~84s → 10 points ≈ 14 min, + 3-4 bisection steps ≈ 5 min more.
#
# Output: BEST_OCC_SCALE, df_occ_sweep (used by visualization cell below).
# =============================================================================

_RUN_OCC_CALIBRATION = False  # ← Set True to re-run OCC_SCALE sweep

if not _RUN_OCC_CALIBRATION:
    print("⏭️  OCC_SCALE sweep skipped (already calibrated)")
    print(f"   CALIBRATED_OCC_SCALE = {CALIBRATED_OCC_SCALE}")
    print("   Set _RUN_OCC_CALIBRATION = True and re-run this cell to recalibrate.")
    BEST_OCC_SCALE = CALIBRATED_OCC_SCALE
    df_occ_sweep = None

# ─── Calibration code below only runs when _RUN_OCC_CALIBRATION = True ───
if _RUN_OCC_CALIBRATION:
    import time as _time

    HIST_QP_SERVED = ...    # ← set to your historical QP served count (from Cell 14 comparison table)
    TOLERANCE_PCT = 1.0     # accept if within ±1% of target

    # --- Phase 1: Coarse grid sweep ---
    occ_scale_grid = [0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00, 2.50, 3.00, 4.00]
    sweep_results = []

    print(f"{'='*80}")
    print(f"🔍 OCC_SCALE SWEEP — Target: {HIST_QP_SERVED:,} QP served (historical)")
    print(f"   Testing {len(occ_scale_grid)} values: {occ_scale_grid}")
    print(f"{'='*80}\n")

    for idx, occ_scale_test in enumerate(occ_scale_grid):
        t0 = _time.time()
        day_states_sweep = {}

        for day in qp_days:
            day_str = str(day)
            df_qp_day = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == day]
            day_active_windows_sw = detect_active_windows(df_qp_day, gap_threshold_min=5.0)
            df_day_ops_sw, op_shifts_sw = get_day_operators(df_calls_qplanner, day_str)
            if df_day_ops_sw.empty:
                continue
            day_data_occ_sw = df_qp_day.copy()
            day_data_occ_sw['hour'] = pd.to_datetime(day_data_occ_sw['START_DATE_TIME']).dt.hour
            day_ops_hr = day_data_occ_sw.groupby('hour')['agent_username'].nunique()
            day_pool_sw = len(df_day_ops_sw)
            day_occ_raw_sw = {h: 1 - (n / day_pool_sw) for h, n in day_ops_hr.items()}
            day_occupancy_sw = {h: min(0.99, occ * occ_scale_test) for h, occ in day_occ_raw_sw.items()}

            state_sw = run_simulation(
                df_call_events=df_call_events,
                df_operators=df_day_ops_sw,
                score_lookup=full_score_lookup,
                churn_i_lookup=churn_i_lookup,
                fo_config=baseline_config,
                tick_interval=CALIBRATED_TICK,
                max_wait_time=180,
                sim_day=day_str,
                active_windows=day_active_windows_sw,
                gap_busy_fraction=0.7,
                hourly_occupancy=day_occupancy_sw,
                operator_shifts=op_shifts_sw,
                fallback_churn_ij=None,
                pipeline_latency=CALIBRATED_LATENCY,
                pipeline_latency_std=CALIBRATED_LATENCY_STD,
                nqp_mean_task_s=CALIBRATED_NQP_TASK_S,
                fifo_latency=CALIBRATED_FIFO_LATENCY,
                fifo_block_operators=CALIBRATED_FIFO_BLOCK,
                enable_fifo=True,
                delivery_failure_rate=CALIBRATED_DELIVERY_FAILURE_RATE,
                callback_deflection_time=CALIBRATED_CALLBACK_DEFLECTION_TIME,
                callback_deflection_rate=CALIBRATED_CALLBACK_DEFLECTION_RATE,
                acw_mean_s=CALIBRATED_ACW_MEAN_S,
                verbose=False,
            )
            day_states_sweep[day_str] = state_sw

        total_qp_sw = sum(len(s.assignments) for s in day_states_sweep.values())
        total_recs_sw = total_qp_sw + sum(len(s.failed_deliveries) for s in day_states_sweep.values())
        total_fifo_sw = sum(len(s.fifo_assignments) for s in day_states_sweep.values())
        total_unserved_sw = sum(len(s.unserved_calls) for s in day_states_sweep.values())
        total_fd_sw = sum(len(s.failed_deliveries) for s in day_states_sweep.values())
        total_cb_sw = sum(getattr(s, 'callback_deflections', 0) for s in day_states_sweep.values())

        all_waits_sw = [a.waiting_time for s in day_states_sweep.values() for a in s.assignments]
        avg_wait_sw = np.mean(all_waits_sw) if all_waits_sw else 0
        med_wait_sw = np.median(all_waits_sw) if all_waits_sw else 0
        p95_wait_sw = np.percentile(all_waits_sw, 95) if all_waits_sw else 0
        sla_sw = (sum(1 for w in all_waits_sw if w <= 180) / len(all_waits_sw) * 100) if all_waits_sw else 0

        elapsed = _time.time() - t0
        gap_pct = (total_qp_sw - HIST_QP_SERVED) / HIST_QP_SERVED * 100
        hit = "✅" if abs(gap_pct) <= TOLERANCE_PCT else ("🔽" if gap_pct < 0 else "🔼")

        sweep_results.append({
            'occ_scale': occ_scale_test,
            'qp_served': total_qp_sw,
            'qp_recs': total_recs_sw,
            'fifo': total_fifo_sw,
            'unserved': total_unserved_sw,
            'failed_del': total_fd_sw,
            'callbacks': total_cb_sw,
            'avg_wait': avg_wait_sw,
            'med_wait': med_wait_sw,
            'p95_wait': p95_wait_sw,
            'sla_180': sla_sw,
            'gap_pct': gap_pct,
            'elapsed_s': elapsed,
        })

        print(f"  [{idx+1:>2}/{len(occ_scale_grid)}] OCC_SCALE={occ_scale_test:.2f} → "
              f"QP={total_qp_sw:>5,} (gap={gap_pct:>+6.1f}%) {hit}  "
              f"recs={total_recs_sw:>5,} fail={total_fd_sw:>4} cb={total_cb_sw:>4}  "
              f"wait={avg_wait_sw:>5.0f}s SLA={sla_sw:>5.1f}%  [{elapsed:.0f}s]")

    # --- Phase 2: Bisection refinement ---
    print(f"\n{'─'*80}")
    print(f"📐 Phase 2: Bisection refinement")

    df_sweep = pd.DataFrame(sweep_results)
    above = df_sweep[df_sweep['gap_pct'] > 0]
    below = df_sweep[df_sweep['gap_pct'] <= 0]

    if above.empty or below.empty:
        if above.empty:
            print(f"   ⚠️  All values BELOW target — need lower OCC_SCALE range")
            best_idx = df_sweep['gap_pct'].abs().idxmin()
        else:
            print(f"   ⚠️  All values ABOVE target — need higher OCC_SCALE range")
            best_idx = df_sweep['gap_pct'].abs().idxmin()
        best_row = df_sweep.loc[best_idx]
        print(f"   Best so far: OCC_SCALE={best_row['occ_scale']:.2f} → QP={best_row['qp_served']:,.0f} (gap={best_row['gap_pct']:+.1f}%)")
    else:
        above_sorted = above.sort_values('occ_scale')
        below_sorted = below.sort_values('occ_scale')
        lo_scale = above_sorted['occ_scale'].iloc[-1]
        hi_scale = below_sorted['occ_scale'].iloc[0]

        print(f"   Bracket: [{lo_scale:.2f}, {hi_scale:.2f}]")
        print(f"   Refining with bisection (max 5 iterations)...\n")

        for bisect_iter in range(5):
            mid_scale = (lo_scale + hi_scale) / 2.0
            t0 = _time.time()
            day_states_bi = {}

            for day in qp_days:
                day_str = str(day)
                df_qp_day = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == day]
                day_active_windows_bi = detect_active_windows(df_qp_day, gap_threshold_min=5.0)
                df_day_ops_bi, op_shifts_bi = get_day_operators(df_calls_qplanner, day_str)
                if df_day_ops_bi.empty:
                    continue
                day_data_occ_bi = df_qp_day.copy()
                day_data_occ_bi['hour'] = pd.to_datetime(day_data_occ_bi['START_DATE_TIME']).dt.hour
                day_ops_hr_bi = day_data_occ_bi.groupby('hour')['agent_username'].nunique()
                day_pool_bi = len(df_day_ops_bi)
                day_occ_raw_bi = {h: 1 - (n / day_pool_bi) for h, n in day_ops_hr_bi.items()}
                day_occupancy_bi = {h: min(0.99, occ * mid_scale) for h, occ in day_occ_raw_bi.items()}

                state_bi = run_simulation(
                    df_call_events=df_call_events,
                    df_operators=df_day_ops_bi,
                    score_lookup=full_score_lookup,
                    churn_i_lookup=churn_i_lookup,
                    fo_config=baseline_config,
                    tick_interval=CALIBRATED_TICK,
                    max_wait_time=180,
                    sim_day=day_str,
                    active_windows=day_active_windows_bi,
                    gap_busy_fraction=0.7,
                    hourly_occupancy=day_occupancy_bi,
                    operator_shifts=op_shifts_bi,
                    fallback_churn_ij=None,
                    pipeline_latency=CALIBRATED_LATENCY,
                    pipeline_latency_std=CALIBRATED_LATENCY_STD,
                    nqp_mean_task_s=CALIBRATED_NQP_TASK_S,
                    fifo_latency=CALIBRATED_FIFO_LATENCY,
                    fifo_block_operators=CALIBRATED_FIFO_BLOCK,
                    enable_fifo=True,
                    delivery_failure_rate=CALIBRATED_DELIVERY_FAILURE_RATE,
                    callback_deflection_time=CALIBRATED_CALLBACK_DEFLECTION_TIME,
                    callback_deflection_rate=CALIBRATED_CALLBACK_DEFLECTION_RATE,
                    acw_mean_s=CALIBRATED_ACW_MEAN_S,
                    verbose=False,
                )
                day_states_bi[day_str] = state_bi

            total_qp_bi = sum(len(s.assignments) for s in day_states_bi.values())
            total_recs_bi = total_qp_bi + sum(len(s.failed_deliveries) for s in day_states_bi.values())
            total_fd_bi = sum(len(s.failed_deliveries) for s in day_states_bi.values())
            total_cb_bi = sum(getattr(s, 'callback_deflections', 0) for s in day_states_bi.values())
            all_waits_bi = [a.waiting_time for s in day_states_bi.values() for a in s.assignments]
            avg_wait_bi = np.mean(all_waits_bi) if all_waits_bi else 0
            sla_bi = (sum(1 for w in all_waits_bi if w <= 180) / len(all_waits_bi) * 100) if all_waits_bi else 0

            elapsed = _time.time() - t0
            gap_pct_bi = (total_qp_bi - HIST_QP_SERVED) / HIST_QP_SERVED * 100
            hit_bi = "✅" if abs(gap_pct_bi) <= TOLERANCE_PCT else ("🔽" if gap_pct_bi < 0 else "🔼")

            sweep_results.append({
                'occ_scale': mid_scale,
                'qp_served': total_qp_bi,
                'qp_recs': total_recs_bi,
                'fifo': sum(len(s.fifo_assignments) for s in day_states_bi.values()),
                'unserved': sum(len(s.unserved_calls) for s in day_states_bi.values()),
                'failed_del': total_fd_bi,
                'callbacks': total_cb_bi,
                'avg_wait': avg_wait_bi,
                'med_wait': np.median(all_waits_bi) if all_waits_bi else 0,
                'p95_wait': np.percentile(all_waits_bi, 95) if all_waits_bi else 0,
                'sla_180': sla_bi,
                'gap_pct': gap_pct_bi,
                'elapsed_s': elapsed,
            })

            print(f"  Bisect {bisect_iter+1}: OCC_SCALE={mid_scale:.4f} → "
                  f"QP={total_qp_bi:>5,} (gap={gap_pct_bi:>+5.1f}%) {hit_bi}  "
                  f"recs={total_recs_bi:>5,} wait={avg_wait_bi:>5.0f}s SLA={sla_bi:>5.1f}%  [{elapsed:.0f}s]")

            if abs(gap_pct_bi) <= TOLERANCE_PCT:
                print(f"\n   🎯 Converged! OCC_SCALE={mid_scale:.4f} within ±{TOLERANCE_PCT}% of target")
                break

            if gap_pct_bi > 0:
                lo_scale = mid_scale
            else:
                hi_scale = mid_scale

    # --- Summary ---
    df_sweep = pd.DataFrame(sweep_results).sort_values('occ_scale').reset_index(drop=True)
    best_idx = df_sweep['gap_pct'].abs().idxmin()
    best = df_sweep.loc[best_idx]

    print(f"\n{'='*80}")
    print(f"📊 OCC_SCALE SWEEP SUMMARY")
    print(f"{'='*80}")
    print(f"{'OCC_SCALE':>10} {'QP Served':>10} {'Gap %':>8} {'Recs':>7} {'FailDel':>8} {'CB':>5} "
          f"{'AvgWait':>8} {'SLA%':>6}")
    for _, r in df_sweep.iterrows():
        mark = " ←★" if r['occ_scale'] == best['occ_scale'] else ""
        print(f"{r['occ_scale']:>10.4f} {r['qp_served']:>10,.0f} {r['gap_pct']:>+7.1f}% "
              f"{r['qp_recs']:>7,.0f} {r['failed_del']:>8,.0f} {r['callbacks']:>5,.0f} "
              f"{r['avg_wait']:>7.0f}s {r['sla_180']:>5.1f}%{mark}")

    print(f"\n🎯 BEST FIT: OCC_SCALE = {best['occ_scale']:.4f}")
    print(f"   QP Served: {best['qp_served']:,.0f} vs historical {HIST_QP_SERVED:,} (gap={best['gap_pct']:+.1f}%)")
    print(f"   Avg Wait: {best['avg_wait']:.0f}s, SLA ≤180s: {best['sla_180']:.1f}%")
    print(f"   → Update CALIBRATED_OCC_SCALE = {best['occ_scale']:.4f} in Cell 13a")
    print(f"{'='*80}")

    BEST_OCC_SCALE = best['occ_scale']
    df_occ_sweep = df_sweep.copy()

In [ ]:
# =============================================================================
# OCC_SCALE SWEEP — Visualization
# =============================================================================
# ⚠️ Depends on OCC_SCALE sweep cell above. Only runs if sweep was executed.
# =============================================================================

if df_occ_sweep is None:
    print("⏭️  OCC_SCALE visualization skipped (sweep not executed)")
    print(f"   Using pre-calibrated value: CALIBRATED_OCC_SCALE = {CALIBRATED_OCC_SCALE}")
else:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- Panel 1: QP Served vs OCC_SCALE ---
    ax1 = axes[0]
    ax1.plot(df_occ_sweep['occ_scale'], df_occ_sweep['qp_served'], 'o-', color='steelblue', linewidth=2, markersize=6)
    ax1.axhline(y=HIST_QP_SERVED, color='red', linestyle='--', linewidth=1.5, label=f'Historical ({HIST_QP_SERVED:,})')
    ax1.axhline(y=HIST_QP_SERVED * 1.01, color='red', linestyle=':', alpha=0.3)
    ax1.axhline(y=HIST_QP_SERVED * 0.99, color='red', linestyle=':', alpha=0.3)
    ax1.axvline(x=BEST_OCC_SCALE, color='green', linestyle='--', alpha=0.7, label=f'Best fit ({BEST_OCC_SCALE:.3f})')
    ax1.fill_between(df_occ_sweep['occ_scale'], HIST_QP_SERVED * 0.99, HIST_QP_SERVED * 1.01,
                     alpha=0.1, color='red', label='±1% tolerance')
    ax1.set_xlabel('OCC_SCALE')
    ax1.set_ylabel('QP Served Calls')
    ax1.set_title('QP Served vs OCC_SCALE')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # --- Panel 2: Gap % vs OCC_SCALE ---
    ax2 = axes[1]
    ax2.plot(df_occ_sweep['occ_scale'], df_occ_sweep['gap_pct'], 'o-', color='darkorange', linewidth=2, markersize=6)
    ax2.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Zero gap')
    ax2.fill_between(df_occ_sweep['occ_scale'], -TOLERANCE_PCT, TOLERANCE_PCT,
                     alpha=0.1, color='green', label=f'±{TOLERANCE_PCT}% tolerance')
    ax2.axvline(x=BEST_OCC_SCALE, color='green', linestyle='--', alpha=0.7)
    ax2.set_xlabel('OCC_SCALE')
    ax2.set_ylabel('Gap vs Historical (%)')
    ax2.set_title('Simulation Gap vs OCC_SCALE')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # --- Panel 3: Wait time & SLA vs OCC_SCALE ---
    ax3 = axes[2]
    ax3_twin = ax3.twinx()
    ax3.plot(df_occ_sweep['occ_scale'], df_occ_sweep['avg_wait'], 's-', color='purple', linewidth=2, markersize=5, label='Avg Wait (s)')
    ax3.plot(df_occ_sweep['occ_scale'], df_occ_sweep['p95_wait'], '^-', color='darkviolet', linewidth=1.5, markersize=5, alpha=0.7, label='P95 Wait (s)')
    ax3_twin.plot(df_occ_sweep['occ_scale'], df_occ_sweep['sla_180'], 'D-', color='teal', linewidth=2, markersize=5, label='SLA ≤180s (%)')
    ax3.axvline(x=BEST_OCC_SCALE, color='green', linestyle='--', alpha=0.7)
    ax3.set_xlabel('OCC_SCALE')
    ax3.set_ylabel('Wait Time (s)', color='purple')
    ax3_twin.set_ylabel('SLA ≤180s (%)', color='teal')
    ax3.set_title('Wait Time & SLA vs OCC_SCALE')
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax3_twin.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center left')
    ax3.grid(True, alpha=0.3)

    fig.suptitle(f'OCC_SCALE Calibration Sweep — Best fit: {BEST_OCC_SCALE:.4f}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\n🎯 Recommended update: CALIBRATED_OCC_SCALE = {BEST_OCC_SCALE:.4f}")

<a id="churn-calibration"></a>
### Churn Model Calibration — Diagnosing EBM model bias

Compares EBM model P(churn) predictions against actual `RESOL_MOT_DSC` outcomes.
Result: `CALIBRATED_CHURN_BIAS = 0.0288` (model overestimates by +2.88pp).
Set `_RUN_CHURN_CALIBRATION = True` to re-run.

> **Note:** Gap decomposition (Part 3) and Isotonic regression validation are in a separate cell after the simulation, since they require `all_day_states`.

In [ ]:
# =============================================================================
# CHURN MODEL CALIBRATION — Derivation (Parts 1-2)
# =============================================================================
# ⚠️ CALIBRATION CELL — Run once to derive CALIBRATED_CHURN_BIAS.
#   Set _RUN_CHURN_CALIBRATION = True to re-run.
#   Update CALIBRATED_CHURN_BIAS in Cell 13a with the result.
# =============================================================================
# Derives the churn bias by comparing EBM model predictions vs actual outcomes
# on historical QP-assigned pairs. No simulation outputs needed.
#
# NOTE: Gap decomposition (Part 3) and Isotonic regression validation are in
#   a separate cell AFTER the simulation, since they need all_day_states.
# =============================================================================

_RUN_CHURN_CALIBRATION = False  # ← Set True to re-run churn calibration

if not _RUN_CHURN_CALIBRATION:
    print("⏭️  Churn calibration diagnostic skipped (already calibrated)")
    print(f"   CALIBRATED_CHURN_BIAS = {CALIBRATED_CHURN_BIAS} (+{CALIBRATED_CHURN_BIAS*100:.2f}pp)")
    print("   Set _RUN_CHURN_CALIBRATION = True and re-run this cell to recalibrate.")

# ─── Calibration code below only runs when _RUN_CHURN_CALIBRATION = True ───
if _RUN_CHURN_CALIBRATION:
    import matplotlib.pyplot as plt
    from sklearn.calibration import calibration_curve
    from sklearn.metrics import brier_score_loss

    # ═══════════════════════════════════════════════════════════════════════════
    # PART 1: Model calibration on HISTORICAL QP-assigned pairs
    # ═══════════════════════════════════════════════════════════════════════════
    _hist_assigned = df_calls_qplanner[
        df_calls_qplanner['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable'
    ].drop_duplicates(subset='CALL_ID')
    _hist_assigned = _hist_assigned[_hist_assigned['RESOL_MOT_DSC'].isin(VALID_OUTCOMES)].copy()

    _calib_records = []
    for _, row in _hist_assigned.iterrows():
        cid = row['CALL_ID']
        op = row['agent_username']
        actual = 1 if row['RESOL_MOT_DSC'] == 'N RECUPERADO' else 0
        pred_full = full_score_lookup.get((cid, op))
        pred_sparse = score_lookup.get((cid, op))
        chi = churn_i_lookup.get(cid)
        if pred_full is not None:
            _calib_records.append({
                'call_id': cid, 'operator': op, 'actual': actual,
                'pred_full': pred_full, 'pred_sparse': pred_sparse, 'churn_i': chi,
            })

    _df_calib = pd.DataFrame(_calib_records)
    print(f"{'='*80}")
    print(f"📊 CHURN MODEL CALIBRATION — Historical QP-assigned pairs")
    print(f"{'='*80}")
    print(f"   Pairs with full_score_lookup match: {len(_df_calib):,}")
    print(f"   Pairs with sparse match:            {_df_calib['pred_sparse'].notna().sum():,}")

    _actual_rate = _df_calib['actual'].mean()
    _pred_rate = _df_calib['pred_full'].mean()
    _pred_sparse_rate = _df_calib.loc[_df_calib['pred_sparse'].notna(), 'pred_sparse'].mean()
    _bias = _pred_rate - _actual_rate

    print(f"\n   📐 Overall calibration:")
    print(f"      Actual churn rate:        {_actual_rate:.4f} ({_actual_rate*100:.2f}%)")
    print(f"      Model mean P(churn):      {_pred_rate:.4f} ({_pred_rate*100:.2f}%)")
    print(f"      Sparse mean P(churn):     {_pred_sparse_rate:.4f} ({_pred_sparse_rate*100:.2f}%)")
    print(f"      Bias (model - actual):    {_bias:+.4f} ({_bias*100:+.2f} pp)")

    _brier = brier_score_loss(_df_calib['actual'], _df_calib['pred_full'])
    print(f"      Brier score (full):       {_brier:.4f}")
    if _df_calib['pred_sparse'].notna().sum() > 100:
        _brier_s = brier_score_loss(
            _df_calib.loc[_df_calib['pred_sparse'].notna(), 'actual'],
            _df_calib.loc[_df_calib['pred_sparse'].notna(), 'pred_sparse']
        )
        print(f"      Brier score (sparse):     {_brier_s:.4f}")

    # ═══════════════════════════════════════════════════════════════════════════
    # PART 2: Calibration curve (reliability diagram)
    # ═══════════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax1 = axes[0]
    prob_true, prob_pred = calibration_curve(_df_calib['actual'], _df_calib['pred_full'],
                                              n_bins=15, strategy='quantile')
    ax1.plot(prob_pred, prob_true, 'o-', color='steelblue', linewidth=2, markersize=6,
             label=f'Full model (bias={_bias*100:+.1f}pp)')
    ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')

    _sparse_mask = _df_calib['pred_sparse'].notna()
    if _sparse_mask.sum() > 100:
        prob_true_s, prob_pred_s = calibration_curve(
            _df_calib.loc[_sparse_mask, 'actual'],
            _df_calib.loc[_sparse_mask, 'pred_sparse'],
            n_bins=15, strategy='quantile'
        )
        ax1.plot(prob_pred_s, prob_true_s, 's-', color='coral', linewidth=2, markersize=5,
                 alpha=0.7, label='Sparse (brains)')

    ax1.set_xlabel('Mean Predicted P(churn)')
    ax1.set_ylabel('Fraction of Actual Churns')
    ax1.set_title('Calibration Curve (Reliability Diagram)')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    _df_calib['decile'] = pd.qcut(_df_calib['pred_full'], q=10, labels=False, duplicates='drop')
    _decile_stats = _df_calib.groupby('decile').agg(
        mean_pred=('pred_full', 'mean'), actual_rate=('actual', 'mean'), count=('actual', 'count')
    ).reset_index()
    ax2.bar(_decile_stats['decile'], _decile_stats['actual_rate'], alpha=0.6,
            color='steelblue', label='Actual churn rate')
    ax2.plot(_decile_stats['decile'], _decile_stats['mean_pred'], 'ro-',
             linewidth=2, markersize=6, label='Mean predicted')
    ax2.set_xlabel('Prediction Decile')
    ax2.set_ylabel('Churn Rate')
    ax2.set_title('Predicted vs Actual by Decile')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    _df_calib_day = _df_calib.copy()
    _df_calib_day['day'] = _hist_assigned.set_index('CALL_ID').loc[
        _df_calib_day['call_id'], 'START_DATE_TIME'
    ].values
    _df_calib_day['day'] = pd.to_datetime(_df_calib_day['day']).dt.strftime('%m-%d')
    _day_bias = _df_calib_day.groupby('day').agg(
        actual_rate=('actual', 'mean'), pred_rate=('pred_full', 'mean'), n=('actual', 'count')
    ).reset_index()
    _day_bias['bias_pp'] = (_day_bias['pred_rate'] - _day_bias['actual_rate']) * 100

    ax3 = axes[2]
    colors = ['steelblue' if b >= 0 else 'coral' for b in _day_bias['bias_pp']]
    ax3.bar(range(len(_day_bias)), _day_bias['bias_pp'], color=colors, alpha=0.7)
    ax3.axhline(y=0, color='black', linewidth=0.5)
    ax3.axhline(y=_bias*100, color='red', linestyle='--', linewidth=1.5,
                label=f'Overall bias: {_bias*100:+.1f}pp')
    ax3.set_xticks(range(len(_day_bias)))
    ax3.set_xticklabels(_day_bias['day'], rotation=45, fontsize=7)
    ax3.set_xlabel('Day')
    ax3.set_ylabel('Bias (pp) = Predicted − Actual')
    ax3.set_title('Model Bias by Day')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)

    fig.suptitle('Churn Model Calibration — EBM (full_score_lookup) vs Actual Outcomes',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ═══════════════════════════════════════════════════════════════════════════
    # Correction factor summary
    # ═══════════════════════════════════════════════════════════════════════════
    _additive_correction = _bias
    _mult_ratio = _actual_rate / _pred_rate if _pred_rate > 0 else 1.0

    print(f"\n   📐 Calibration corrections:")
    print(f"      Additive: churn_ij → churn_ij - {_additive_correction:.4f}")
    print(f"      Multiplicative: churn_ij → churn_ij × {_mult_ratio:.4f}")
    print(f"\n   → Update CALIBRATED_CHURN_BIAS = {_additive_correction:.4f} in Cell 13a")
    print(f"{'='*80}")

<a id="run-simulation"></a>
## Cell 13b: Run 21-Day Simulation

In [ ]:
# =============================================================================
# CELL 13b — RUN 21-DAY SIMULATION with FIFO telephony routing
# =============================================================================
# INPUT:
#   - CALIBRATED_* constants (from Cell 13a)
#   - df_call_events, df_calls_qplanner_ts, qp_days, busiest_day (from Cell 13a)
#   - score_lookup, churn_i_lookup, full_score_lookup (from cells 7 + 12b)
#   - FOConfig (from cell 10), run_simulation (from cell 12)
#   - detect_active_windows, get_day_operators (from cells 7, 9)
#   - fallback_churn_ij (from override cell)
#
# DOES:
#   For EACH of the 21 QPlanner days:
#     a. Detect active windows (when QPlanner was on vs gaps)
#     b. Get day-specific operators + shift windows
#     c. Build hourly occupancy from real operator activity data
#     d. Call run_simulation() with max_wait_time=180s (fixed QP window) and FIFO enabled
#   Then aggregates results across all days.
#
# OUTPUT:
#   - all_day_states: dict day_str → SimulationState (21 entries)
#   - all_assignments: flat list of all QP Assignment objects across days
#   - all_fifo_assignments: flat list of all FIFO Assignment objects across days
#   - state_baseline: SimulationState for the busiest day
#   - hourly_occupancy, operator_shifts (busiest day)
# =============================================================================

# --- Run simulation for EACH day ---
baseline_config = FOConfig()
all_day_states = {}  # day_str → SimulationState

for day in qp_days:
    day_str = str(day)

    # Detect active windows for this day
    df_qp_day = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == day]
    day_active_windows = detect_active_windows(df_qp_day, gap_threshold_min=5.0)

    # Get day-specific operators & shift windows
    df_day_ops, op_shifts = get_day_operators(df_calls_qplanner, day_str)
    if df_day_ops.empty:
        print(f"  ⏭️  {day_str}: no operators, skipping")
        continue

    # Build hourly occupancy for this day
    day_data_occ = df_qp_day.copy()
    day_data_occ['hour'] = pd.to_datetime(day_data_occ['START_DATE_TIME']).dt.hour
    day_ops_per_hour = day_data_occ.groupby('hour')['agent_username'].nunique()
    day_pool = len(df_day_ops)
    day_occ_raw = {h: 1 - (n / day_pool) for h, n in day_ops_per_hour.items()}
    day_occupancy = {h: min(0.99, occ * CALIBRATED_OCC_SCALE) for h, occ in day_occ_raw.items()}

    # Run simulation with FIFO enabled (verbose=False for multi-day)
    state = run_simulation(
        df_call_events=df_call_events,
        df_operators=df_day_ops,
        score_lookup=full_score_lookup,
        churn_i_lookup=churn_i_lookup,
        fo_config=baseline_config,
        tick_interval=CALIBRATED_TICK,
        max_wait_time=180,
        sim_day=day_str,
        active_windows=day_active_windows,
        gap_busy_fraction=0.7,
        hourly_occupancy=day_occupancy,
        operator_shifts=op_shifts,
        fallback_churn_ij=fallback_churn_ij,
        pipeline_latency=CALIBRATED_LATENCY,
        pipeline_latency_std=CALIBRATED_LATENCY_STD,
        nqp_mean_task_s=CALIBRATED_NQP_TASK_S,
        fifo_latency=CALIBRATED_FIFO_LATENCY,
        fifo_block_operators=CALIBRATED_FIFO_BLOCK,
        enable_fifo=True,
        delivery_failure_rate=CALIBRATED_DELIVERY_FAILURE_RATE,
        callback_deflection_time=CALIBRATED_CALLBACK_DEFLECTION_TIME,
        callback_deflection_rate=CALIBRATED_CALLBACK_DEFLECTION_RATE,
        acw_mean_s=CALIBRATED_ACW_MEAN_S,
        verbose=False,
    )

    all_day_states[day_str] = state
    n_qp = len(state.assignments)
    n_fifo = len(state.fifo_assignments)
    n_u = len(state.unserved_calls)
    n_fd = len(state.failed_deliveries)
    n_cb = getattr(state, 'callback_deflections', 0)
    waits = [a.waiting_time for a in state.assignments]
    avg_w = np.mean(waits) if waits else 0
    day_mark = " ★" if day_str == busiest_day else ""
    cov = state.score_coverage
    cov_pct = cov['coverage_pct']
    print(f"  ✅ {day_str}: QP={n_qp:>3}, FIFO={n_fifo:>4}, unserved={n_u:>2}, "
          f"fail_del={n_fd:>2}, callback={n_cb:>2}, ops={day_pool:>3}, "
          f"windows={len(day_active_windows)}, avg_wait={avg_w:>5.0f}s, "
          f"score_cov={cov_pct:.1f}%{day_mark}")

# --- Aggregate all days ---
all_assignments = []
all_fifo_assignments = []
for day_str, state in all_day_states.items():
    all_assignments.extend(state.assignments)
    all_fifo_assignments.extend(state.fifo_assignments)

total_unserved = sum(len(s.unserved_calls) for s in all_day_states.values())
total_failed_deliveries = sum(len(s.failed_deliveries) for s in all_day_states.values())
total_callback_deflections = sum(getattr(s, 'callback_deflections', 0) for s in all_day_states.values())
total_qp = len(all_assignments)
total_fifo = len(all_fifo_assignments)
total_calls = total_qp + total_fifo + total_unserved

print(f"\n{'='*70}")
print(f"📊 TOTAL across {len(all_day_states)} days:")
print(f"   QP assigned (served):     {total_qp:,}  ← comparable to historical")
print(f"   FIFO routed (counterfact.): {total_fifo:,}  ← non-blocking churn estimation only")
print(f"   Truly unserved:           {total_unserved:,}")
print(f"   Failed deliveries:        {total_failed_deliveries:,}  ({total_failed_deliveries/(total_qp+total_failed_deliveries)*100:.1f}% of QP recs)")

print(f"   Callback deflections:     {total_callback_deflections:,}  (left queue at ~{CALIBRATED_CALLBACK_DEFLECTION_TIME:.0f}s)")
print(f"   Total calls entering QP:  {total_calls:,}")
print(f"   QP service rate:          {total_qp / total_calls * 100:.1f}%  (QP only — apples-to-apples with historical)")

if all_assignments:
    all_sim_waits = [a.waiting_time for a in all_assignments]
    print(f"\n   QPlanner (optimized):")
    print(f"     Avg wait:  {np.mean(all_sim_waits):.1f}s (median: {np.median(all_sim_waits):.1f}s)")
    print(f"     P95 wait:  {np.percentile(all_sim_waits, 95):.1f}s")
    sla = sum(1 for w in all_sim_waits if w <= 180) / len(all_sim_waits) * 100
    print(f"     SLA ≤180s: {sla:.1f}%")

if all_fifo_assignments:
    fifo_waits = [a.waiting_time for a in all_fifo_assignments]
    fifo_reductions = [a.churn_reduction for a in all_fifo_assignments]
    print(f"\n   FIFO telephony (counterfactual — non-blocking, churn estimation only):")
    print(f"     Avg wait:  {np.mean(fifo_waits):.1f}s (median: {np.median(fifo_waits):.1f}s)")
    print(f"     P95 wait:  {np.percentile(fifo_waits, 95):.1f}s")
    print(f"     Avg churn reduction: {np.mean(fifo_reductions):.4f}")
    print(f"     ⚠️  Non-blocking: operators NOT held busy — these are NOT real served calls.")
    print(f"        Used solely for churn outcome estimation of overflow calls.")

# Score coverage aggregate (per-tick evaluation, summed across all days)
_agg_pairs = sum(s.score_coverage['total_pairs'] for s in all_day_states.values())
_agg_hits = sum(s.score_coverage['lookup_hits'] for s in all_day_states.values())
_agg_pct = _agg_hits / _agg_pairs * 100 if _agg_pairs > 0 else 0
_agg_fb = _agg_pairs - _agg_hits
print(f"\n   Score lookup coverage (solver-time):")
print(f"     Pairs evaluated: {_agg_pairs:,} (across all ticks × all days)")
print(f"     Lookup hits:     {_agg_hits:,} ({_agg_pct:.1f}%)")
print(f"     Fallback used:   {_agg_fb:,} ({100-_agg_pct:.1f}%) → churn_ij = {fallback_churn_ij:.4f}")
print(f"{'='*70}")

# Export busiest-day scoped variables needed by experiment cells (17-18)
df_qp_busiest = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == pd.to_datetime(busiest_day).date()]
df_day_operators, operator_shifts = get_day_operators(df_calls_qplanner, busiest_day)
day_active_windows = detect_active_windows(df_qp_busiest, gap_threshold_min=5.0)

# Rebuild hourly occupancy for busiest day
day_data_for_occ = df_qp_busiest.copy()
day_data_for_occ['hour'] = pd.to_datetime(day_data_for_occ['START_DATE_TIME']).dt.hour
ops_per_hour = day_data_for_occ.groupby('hour')['agent_username'].nunique()
total_pool = len(df_day_operators)
hourly_occ_raw = {h: 1 - (n / total_pool) for h, n in ops_per_hour.items()}
hourly_occupancy = {h: min(0.99, occ * CALIBRATED_OCC_SCALE) for h, occ in hourly_occ_raw.items()}

# Store busiest day state + variables for backward compatibility with analysis/experiment cells
state_baseline = all_day_states.get(busiest_day)

<a id="per-day-comparison"></a>
## Cell 14: Comparison — Simulation vs Historical (Per-Day)

In [ ]:
# =============================================================================
# CELL 14 — COMPARISON: Per-day + aggregated Simulation vs Historical
# =============================================================================
# INPUT:
#   - all_day_states: dict of day_str → SimulationState (21 days, from cell 13)
#   - df_calls_qplanner: historical calls with recommended flag and wait times
#   - all_assignments: flat list of all Assignment objects (from cell 13)
#   - busiest_day, CALIBRATED_* parameters
#
# DOES:
#   1. For each day, extracts historical ground truth (recommended==True calls)
#   2. Computes per-day metrics: assigned count, avg/median/P95 wait, SLA ≤180s
#   3. Builds a comparison table (sim vs hist) with percentage deltas
#   4. Computes aggregated totals across all 21 days
#   5. Prints formatted per-day table + totals + calibrated parameters
#
# OUTPUT:
#   - df_comparison: DataFrame with per-day sim vs hist metrics
#   - Aggregated variables: agg_hist_*, agg_sim_* (n, avg, med, p95, sla)
#   - hist_all_paired, hist_all_waits, sim_all_waits_arr (for downstream cells)
# =============================================================================

# --- Build per-day comparison ---
comparison_rows = []

for day_str, state in sorted(all_day_states.items()):
    # Historical ground truth for this day
    day_date = pd.to_datetime(day_str).date()
    hist_day = df_calls_qplanner[
        pd.to_datetime(df_calls_qplanner['START_DATE_TIME']).dt.date == day_date
    ]
    hist_paired = hist_day[hist_day['recommended'] == True].drop_duplicates(subset='CALL_ID')

    if hist_paired.empty or not state.assignments:
        continue

    h_n = len(hist_paired)
    h_avg = hist_paired['WAIT_TIME_IN_QUEUE_SEC_QTY'].mean()
    h_med = hist_paired['WAIT_TIME_IN_QUEUE_SEC_QTY'].median()
    h_p95 = hist_paired['WAIT_TIME_IN_QUEUE_SEC_QTY'].quantile(0.95)
    h_waits = hist_paired['WAIT_TIME_IN_QUEUE_SEC_QTY'].values
    h_sla = (h_waits <= 180).sum() / len(h_waits) * 100

    s_waits = [a.waiting_time for a in state.assignments]
    s_n = len(s_waits)
    s_rec = s_n + len(state.failed_deliveries)  # total recommendations = assigned + failed
    s_avg = np.mean(s_waits)
    s_med = np.median(s_waits)
    s_p95 = np.percentile(s_waits, 95)
    s_sla = sum(1 for w in s_waits if w <= 180) / len(s_waits) * 100

    # Unserved counts
    hist_day_total = len(hist_day)
    hist_day_unserved = hist_day_total - h_n  # total - assigned = unserved
    sim_day_unserved = len(state.unserved_calls)

    comparison_rows.append({
        'day': day_str,
        'hist_n': h_n, 'sim_n': s_n, 'sim_rec': s_rec,
        'hist_unserved': hist_day_unserved, 'sim_unserved': sim_day_unserved,
        'hist_avg': h_avg, 'sim_avg': s_avg,
        'hist_med': h_med, 'sim_med': s_med,
        'hist_p95': h_p95, 'sim_p95': s_p95,
        'hist_sla': h_sla, 'sim_sla': s_sla,
    })

df_comparison = pd.DataFrame(comparison_rows)

# --- Compute delta columns ---
df_comparison['Δ_assigned_%'] = ((df_comparison['sim_n'] - df_comparison['hist_n']).abs() / df_comparison['hist_n'] * 100).round(1)
df_comparison['Δ_rec_%'] = ((df_comparison['sim_rec'] - df_comparison['hist_n']).abs() / df_comparison['hist_n'] * 100).round(1)
df_comparison['Δ_unserved'] = df_comparison['sim_unserved'] - df_comparison['hist_unserved']
df_comparison['Δ_avg_%'] = ((df_comparison['sim_avg'] - df_comparison['hist_avg']).abs() / df_comparison['hist_avg'] * 100).round(1)
df_comparison['Δ_med_%'] = ((df_comparison['sim_med'] - df_comparison['hist_med']).abs() / df_comparison['hist_med'] * 100).round(1)
df_comparison['Δ_p95_%'] = ((df_comparison['sim_p95'] - df_comparison['hist_p95']).abs() / df_comparison['hist_p95'] * 100).round(1)
df_comparison['Δ_sla_pp'] = (df_comparison['sim_sla'] - df_comparison['hist_sla']).round(1)

# --- Build display DataFrame ---
display_rows = []
for _, r in df_comparison.iterrows():
    mark = " ★" if r['day'] == busiest_day else ""
    display_rows.append({
        'Day': r['day'] + mark,
        'Sim Assigned': int(r['sim_n']),
        'Hist Assigned': int(r['hist_n']),
        'Sim Rec': int(r['sim_rec']),
        'Δ Assigned %': r['Δ_assigned_%'],
        'Δ Rec %': r['Δ_rec_%'],
        'Sim Unserved': int(r['sim_unserved']),
        'Hist Unserved': int(r['hist_unserved']),
        'Sim Avg Wait': round(r['sim_avg']),
        'Hist Avg Wait': round(r['hist_avg']),
        'Δ Avg %': r['Δ_avg_%'],
        'Sim Med Wait': round(r['sim_med']),
        'Hist Med Wait': round(r['hist_med']),
        'Δ Med %': r['Δ_med_%'],
        'Sim P95': round(r['sim_p95']),
        'Hist P95': round(r['hist_p95']),
        'Δ P95 %': r['Δ_p95_%'],
        'Sim SLA%': round(r['sim_sla'], 1),
        'Hist SLA%': round(r['hist_sla'], 1),
        'Δ SLA pp': r['Δ_sla_pp'],
    })

# --- Aggregate totals ---
agg_hist_n = int(df_comparison['hist_n'].sum())
agg_sim_n = int(df_comparison['sim_n'].sum())
agg_sim_rec = int(df_comparison['sim_rec'].sum())

# Weighted averages (by number of assignments)
agg_hist_avg = (df_comparison['hist_avg'] * df_comparison['hist_n']).sum() / agg_hist_n
agg_sim_avg = (df_comparison['sim_avg'] * df_comparison['sim_n']).sum() / agg_sim_n

# For aggregated SLA, P95, median: compute from ALL wait times
hist_all_paired = df_calls_qplanner[df_calls_qplanner['recommended'] == True].drop_duplicates(subset='CALL_ID')
hist_all_waits = hist_all_paired['WAIT_TIME_IN_QUEUE_SEC_QTY'].values
sim_all_waits_arr = np.array([a.waiting_time for a in all_assignments])

agg_hist_med = float(np.median(hist_all_waits))
agg_sim_med = float(np.median(sim_all_waits_arr))
agg_hist_p95 = float(np.percentile(hist_all_waits, 95))
agg_sim_p95 = float(np.percentile(sim_all_waits_arr, 95))
agg_hist_sla = (hist_all_waits <= 180).sum() / len(hist_all_waits) * 100
agg_sim_sla = sum(1 for w in sim_all_waits_arr if w <= 180) / len(sim_all_waits_arr) * 100

# Aggregated unserved
agg_hist_unserved = int(df_comparison['hist_unserved'].sum())
agg_sim_unserved = int(df_comparison['sim_unserved'].sum())

# Add TOTAL row
display_rows.append({
    'Day': 'TOTAL',
    'Sim Assigned': agg_sim_n,
    'Hist Assigned': agg_hist_n,
    'Sim Rec': agg_sim_rec,
    'Δ Assigned %': round(abs(agg_sim_n - agg_hist_n) / agg_hist_n * 100, 1),
    'Δ Rec %': round(abs(agg_sim_rec - agg_hist_n) / agg_hist_n * 100, 1),
    'Sim Unserved': agg_sim_unserved,
    'Hist Unserved': agg_hist_unserved,
    'Sim Avg Wait': round(agg_sim_avg),
    'Hist Avg Wait': round(agg_hist_avg),
    'Δ Avg %': round(abs(agg_sim_avg - agg_hist_avg) / agg_hist_avg * 100, 1),
    'Sim Med Wait': round(agg_sim_med),
    'Hist Med Wait': round(agg_hist_med),
    'Δ Med %': round(abs(agg_sim_med - agg_hist_med) / agg_hist_med * 100, 1),
    'Sim P95': round(agg_sim_p95),
    'Hist P95': round(agg_hist_p95),
    'Δ P95 %': round(abs(agg_sim_p95 - agg_hist_p95) / agg_hist_p95 * 100, 1),
    'Sim SLA%': round(agg_sim_sla, 1),
    'Hist SLA%': round(agg_hist_sla, 1),
    'Δ SLA pp': round(agg_sim_sla - agg_hist_sla, 1),
})

df_display = pd.DataFrame(display_rows)

# --- Style: highlight TOTAL row, color-code deltas ---
def _style_comparison(styler):
    """Apply conditional formatting to the comparison table."""
    def _color_delta_pct(val):
        if pd.isna(val): return ''
        v = abs(val)
        if v < 5:   return 'color: #4caf50; font-weight: bold'   # green
        if v < 15:  return 'color: #ff9800; font-weight: bold'   # orange
        return 'color: #f44336; font-weight: bold'                # red

    def _bold_total(row):
        if row['Day'] == 'TOTAL':
            return ['font-weight: bold; border-top: 2px solid #888'] * len(row)
        return [''] * len(row)

    delta_cols = ['Δ Assigned %', 'Δ Rec %', 'Δ Avg %', 'Δ Med %', 'Δ P95 %']
    styler = styler.map(_color_delta_pct, subset=delta_cols)
    styler = styler.map(lambda v: 'color: #4caf50; font-weight: bold' if isinstance(v, (int, float)) and abs(v) < 5 else
                                  'color: #ff9800; font-weight: bold' if isinstance(v, (int, float)) and abs(v) < 15 else
                                  'color: #f44336; font-weight: bold' if isinstance(v, (int, float)) else '',
                        subset=['Δ SLA pp'])
    styler = styler.apply(_bold_total, axis=1)
    styler = styler.format(precision=1)
    styler = styler.set_caption(
        f'Per-Day Comparison — Simulation vs Historical '
        f'(tick={CALIBRATED_TICK}s, latency μ={CALIBRATED_LATENCY}s σ={CALIBRATED_LATENCY_STD}s, '
        f'KAPPA gate: natural FO > 0)')
    return styler

display(_style_comparison(df_display.style))

<a id="validation-plots"></a>
## Cell 15: Validation Plots — Aggregated Distribution Comparison

In [ ]:
# =============================================================================
# CELL 15 — VALIDATION PLOTS: Aggregated distribution + churn outcome comparison
# =============================================================================
# INPUT:
#   - df_calls_qplanner (historical assigned calls with wait times)
#   - df_calls (full operational dataset with RESOL_MOT_DSC)
#   - all_assignments (QP assignments across 21 days)
#   - all_fifo_assignments (FIFO telephony assignments across 21 days)
#   - all_day_states, CALIBRATED_TICK, CALIBRATED_LATENCY, CALIBRATED_LATENCY_STD
#   - VALID_OUTCOMES (from cell 3b)
#   - churn_i_lookup (from cell 7)
#
# DOES:
#   Part A — Wait time validation (2×2 figure):
#     1. Histogram overlay — wait time density (historical vs simulated QP)
#     2. CDF comparison — QP + FIFO cumulative with SLA line at 180s
#     3. QQ plot — quantile-quantile scatter (perfect match = diagonal)
#     4. Metrics table — side-by-side comparison with color-coded deltas
#        NOTE: "Total Served" compares QP-to-QP only.
#              FIFO is non-blocking (counterfactual for churn estimation).
#
#   Part B — Churn outcome estimation ("Not Recovered") — FULL UNIVERSE:
#     Universe = ALL calls that entered the QPlanner queue.
#       - QP assigned:   P(Not Recovered) = churn_ij (model-optimised pairing)
#       - FIFO routed:   P(Not Recovered) = churn_ij (random pairing, ~churn_i)
#         ⚠️ FIFO is non-blocking: operators NOT held busy, used solely for
#            churn estimation of overflow calls (counterfactual layer).
#       - Truly unserved: P(Not Recovered) = churn_i (baseline, no optimisation)
#
#   Part C — Result distribution & wait time by outcome:
#     - Pie chart: QP Assigned / FIFO Counterfactual / Unserved breakdown
#     - Wait time histograms: QP vs FIFO vs Unserved
#
# OUTPUT:
#   - 2×2 wait time validation figure
#   - 1×3 churn outcome figure (full universe)
#   - 2×2 result distribution + wait breakdown
#   - df_all_sim: DataFrame of all QP assignments
# =============================================================================
import matplotlib.pyplot as plt

# Build df_all_sim from all_assignments (used in Part B)
df_all_sim = pd.DataFrame([
    {
        'call_id': a.call_id,
        'customer_id': a.customer_id,
        'operator': a.operator,
        'waiting_time': a.waiting_time,
        'churn_i': a.churn_i,
        'churn_ij': a.churn_ij,
        'churn_reduction': a.churn_reduction,
        'optimization_score': a.optimization_score,
    }
    for a in all_assignments
])

# ═══════════════════════════════════════════════════════════════════════════════
# PART A — WAIT TIME DISTRIBUTION VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

# Historical wait times — DELIVERED calls only (QPlanner-AgentAvailable)
# After delivery failure model, sim all_assignments = delivered calls only,
# so we must compare against historical delivered (not recommended==True).
hist_all_paired = df_calls_qplanner[
    df_calls_qplanner['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable'
].drop_duplicates(subset='CALL_ID')
hist_waits = hist_all_paired['WAIT_TIME_IN_QUEUE_SEC_QTY'].values

# Historical recommendations (for recommendations row)
hist_recommended = df_calls_qplanner[
    df_calls_qplanner['recommended'] == True
].drop_duplicates(subset='CALL_ID')
n_hist_rec = len(hist_recommended)

# Simulated wait times (QP delivered only for fair comparison)
sim_waits = np.array([a.waiting_time for a in all_assignments])

# Simulated recommendations = delivered + failed deliveries
n_sim_rec = len(all_assignments) + total_failed_deliveries

# FIFO wait times (for CDF overlay)
fifo_waits_arr = np.array([a.waiting_time for a in all_fifo_assignments]) if all_fifo_assignments else np.array([])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f'Aggregated Distribution Validation — {len(all_day_states)} QPlanner days '
    f'(tick={CALIBRATED_TICK}s, latency μ={CALIBRATED_LATENCY}s σ={CALIBRATED_LATENCY_STD}s)',
    fontsize=12, fontweight='bold'
)

# 1. Histogram overlay (QP only — apples-to-apples with historical)
ax = axes[0, 0]
bins = np.arange(0, 500, 20)
ax.hist(hist_waits, bins=bins, alpha=0.5, density=True, label=f'Historical (n={len(hist_waits):,})', color='steelblue')
ax.hist(sim_waits, bins=bins, alpha=0.5, density=True, label=f'Sim QP (n={len(sim_waits):,})', color='coral')
ax.axvline(np.mean(hist_waits), color='steelblue', ls='--', lw=1.5, label=f'Hist mean={np.mean(hist_waits):.0f}s')
ax.axvline(np.mean(sim_waits), color='coral', ls='--', lw=1.5, label=f'Sim QP mean={np.mean(sim_waits):.0f}s')
ax.axvline(np.median(hist_waits), color='steelblue', ls=':', lw=1.5, label=f'Hist median={np.median(hist_waits):.0f}s')
ax.axvline(np.median(sim_waits), color='coral', ls=':', lw=1.5, label=f'Sim QP median={np.median(sim_waits):.0f}s')
ax.set_xlabel('Wait Time (s)')
ax.set_ylabel('Density')
ax.set_title('Wait Time Distribution — QP Assigned (All Days)')
ax.legend(fontsize=8)

# 2. CDF comparison (QP + FIFO + Historical)
ax = axes[0, 1]
x_hist = np.sort(hist_waits)
x_sim = np.sort(sim_waits)
ax.plot(x_hist, np.arange(1, len(x_hist)+1) / len(x_hist), label='Historical', color='steelblue', lw=2)
ax.plot(x_sim, np.arange(1, len(x_sim)+1) / len(x_sim), label='Sim QP', color='coral', lw=2)
if len(fifo_waits_arr) > 0:
    x_fifo = np.sort(fifo_waits_arr)
    ax.plot(x_fifo, np.arange(1, len(x_fifo)+1) / len(x_fifo), label=f'Sim FIFO counterfactual (n={len(fifo_waits_arr):,})',
            color='#FF9800', lw=2, ls='--')
ax.axhline(0.5, color='gray', ls=':', alpha=0.5)
ax.axvline(180, color='green', ls='--', alpha=0.5, label='SLA threshold (180s)')
ax.set_xlabel('Wait Time (s)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('CDF Comparison — QP + FIFO (All Days)')
ax.legend(fontsize=9)
ax.set_xlim(0, 700)

# 3. QQ Plot (QP vs Historical)
ax = axes[1, 0]
n_qq = min(len(hist_waits), len(sim_waits), 500)
q_hist = np.percentile(hist_waits, np.linspace(0, 100, n_qq))
q_sim = np.percentile(sim_waits, np.linspace(0, 100, n_qq))
ax.scatter(q_hist, q_sim, s=10, alpha=0.6, color='purple')
max_val = max(q_hist.max(), q_sim.max())
ax.plot([0, max_val], [0, max_val], 'k--', lw=1, label='Perfect match')
ax.set_xlabel('Historical Quantiles (s)')
ax.set_ylabel('Simulated Quantiles (s)')
ax.set_title('Q-Q Plot — QP Assigned (All Days)')
ax.legend()

# 4. Summary metrics comparison
# NOTE: Only QP-vs-QP (delivered) is compared. FIFO is non-blocking (counterfactual only).
ax = axes[1, 1]
ax.axis('off')
hist_n = len(hist_waits)
sim_n = len(sim_waits)
n_fifo = len(fifo_waits_arr)
_total_unserved = sum(len(s.unserved_calls) for s in all_day_states.values())
_total_calls = sim_n + n_fifo + _total_unserved

metrics = {
    'QP Delivered': (hist_n, sim_n),
    'QP Recommendations': (n_hist_rec, n_sim_rec),
    'Failed Deliveries': (n_hist_rec - hist_n, total_failed_deliveries),
    'FIFO Counterfact. ¹': ('—', n_fifo),
    'Truly Unserved': ('—', _total_unserved),
    'QP Svc Rate %': (('—', sim_n / _total_calls * 100) if _total_calls > 0 else ('—', 0)),
    'QP Mean Wait': (np.mean(hist_waits), np.mean(sim_waits)),
    'QP Median Wait': (np.median(hist_waits), np.median(sim_waits)),
    'QP P95': (np.percentile(hist_waits, 95), np.percentile(sim_waits, 95)),
    'QP SLA ≤180s': (sum(hist_waits <= 180)/hist_n*100, sum(sim_waits <= 180)/sim_n*100),
}
if n_fifo > 0:
    metrics['FIFO Median Wait ¹'] = ('—', np.median(fifo_waits_arr))
    metrics['FIFO P95 ¹'] = ('—', np.percentile(fifo_waits_arr, 95))

table_data = []
for name, (h, s) in metrics.items():
    if isinstance(h, str):
        table_data.append([name, h, f'{s:,.0f}' if isinstance(s, (int, float, np.integer, np.floating)) else str(s), '—'])
    else:
        delta = abs(h-s)/h*100 if h != 0 else 0
        fmt = '.0f' if abs(h) > 10 else '.1f'
        table_data.append([name, f'{h:{fmt}}', f'{s:{fmt}}', f'{delta:.1f}%'])

# Add footnote row
table_data.append(['¹ Non-blocking: churn', 'estimation only,', 'ops NOT held busy', ''])

table = ax.table(cellText=table_data,
                 colLabels=['Metric', 'Historical', 'Simulated', 'Δ%'],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.5)
# Color the delta column
for i in range(len(table_data)):
    if table_data[i][3] == '—' or table_data[i][3] == '':
        continue
    delta_val = float(table_data[i][3].rstrip('%'))
    if delta_val < 5:
        table[i+1, 3].set_facecolor('#c8e6c9')  # green
    elif delta_val < 15:
        table[i+1, 3].set_facecolor('#fff9c4')  # yellow
    else:
        table[i+1, 3].set_facecolor('#ffcdd2')  # red
# Style footnote row (last row) with italic-like gray background
_footnote_row = len(table_data)
for j in range(4):
    table[_footnote_row, j].set_facecolor('#f5f5f5')
    table[_footnote_row, j].set_text_props(fontstyle='italic', fontsize=8)
ax.set_title('Metrics Comparison (All Days)', pad=20)

plt.tight_layout()
plt.show()

print(f"   QP delivered (served):      {sim_n:,} (hist AgentAvailable: {hist_n:,}, Δ {abs(sim_n - hist_n) / hist_n * 100:.1f}%)")
print(f"   QP recommendations:         {n_sim_rec:,} (hist recommended: {n_hist_rec:,}, Δ {abs(n_sim_rec - n_hist_rec) / n_hist_rec * 100:.1f}%)")
print(f"   Failed deliveries:          {total_failed_deliveries:,} ({total_failed_deliveries / n_sim_rec * 100:.1f}% of recommendations → FIFO)")
print(f"   FIFO routed (counterfact.): {n_fifo:,} (non-blocking — churn estimation only, ops NOT held busy)")
print(f"   Truly unserved:             {_total_unserved:,}")
print(f"   QP service rate:            {sim_n / _total_calls * 100:.1f}% (QP only — apples-to-apples with historical)")
if n_fifo > 0:
    print(f"   FIFO wait: median={np.median(fifo_waits_arr):.0f}s, P95={np.percentile(fifo_waits_arr, 95):.0f}s")


print(f"\n📝 Simulation summary:")
# PART B — CHURN OUTCOME ESTIMATION: "Not Recovered" — UNIVERSO COMPLETO
# ═══════════════════════════════════════════════════════════════════════════════
# Universe = ALL calls that entered the QPlanner queue.
#   QP assigned:    P(Not Recovered) = churn_ij (model-optimised pairing)
#   FIFO routed:    P(Not Recovered) = churn_ij (matched to random operator)
#     ⚠️ Non-blocking: operators NOT held busy — counterfactual churn estimation
#   Truly unserved: P(Not Recovered) = churn_i  (baseline, no operator)
#
# CALIBRATION: churn_ij values are corrected by CALIBRATED_CHURN_BIAS (additive)
#   to account for EBM model overestimation (+2.88pp). See Cell 32 documentation.
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*65}")
print(f"📊 CHURN OUTCOME ESTIMATION — 'Not Recovered' (universo completo)")
print(f"    (churn_ij corrected by -{CALIBRATED_CHURN_BIAS*100:.2f}pp model bias)")
print(f"{'='*65}")

# --- Build FIFO assignments DataFrame ---
_df_fifo_sim = pd.DataFrame([
    {
        'call_id': a.call_id,
        'churn_i': a.churn_i,
        'churn_ij': a.churn_ij,
        'churn_reduction': a.churn_reduction,
    }
    for a in all_fifo_assignments
]) if all_fifo_assignments else pd.DataFrame(columns=['call_id', 'churn_i', 'churn_ij', 'churn_reduction'])

# --- Build unserved calls DataFrame ---
_unserved_records = []
for _ds, _dstate in sorted(all_day_states.items()):
    for _u in _dstate.unserved_calls:
        _ci = churn_i_lookup.get(_u['call_id'], None)
        _unserved_records.append({
            'call_id': _u['call_id'],
            'churn_i': _ci,
            'churn_ij': _ci,       # no operator → P(churn) = churn_i
            'reason': _u.get('reason', 'unknown'),
        })

_df_unserved = (
    pd.DataFrame(_unserved_records)
    if _unserved_records
    else pd.DataFrame(columns=['call_id', 'churn_i', 'churn_ij', 'reason'])
)

# Join ALL segments to outcomes
_df_unserved = _df_unserved.merge(
    df_calls[['CALL_ID', 'RESOL_MOT_DSC']].drop_duplicates(subset='CALL_ID'),
    left_on='call_id', right_on='CALL_ID', how='left'
)

_df_assigned_sim = df_all_sim.merge(
    df_calls[['CALL_ID', 'RESOL_MOT_DSC']].drop_duplicates(subset='CALL_ID'),
    left_on='call_id', right_on='CALL_ID', how='left'
)

_df_fifo_sim_joined = _df_fifo_sim.merge(
    df_calls[['CALL_ID', 'RESOL_MOT_DSC']].drop_duplicates(subset='CALL_ID'),
    left_on='call_id', right_on='CALL_ID', how='left'
) if not _df_fifo_sim.empty else _df_fifo_sim.assign(CALL_ID=None, RESOL_MOT_DSC=None)

# --- Totals ---
_n_sim_qp = len(_df_assigned_sim)
_n_sim_fifo = len(_df_fifo_sim)
_n_sim_unserved = len(_df_unserved)
_n_sim_total = _n_sim_qp + _n_sim_fifo + _n_sim_unserved

# Filter to "decisor útil" (known outcome)
_qp_du = _df_assigned_sim[_df_assigned_sim['RESOL_MOT_DSC'].isin(VALID_OUTCOMES)]
_fifo_du = _df_fifo_sim_joined[
    (_df_fifo_sim_joined['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
    & (_df_fifo_sim_joined['churn_ij'].notna())
] if not _df_fifo_sim_joined.empty else pd.DataFrame()
_unserved_du = _df_unserved[
    (_df_unserved['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
    & (_df_unserved['churn_i'].notna())
]
_n_qp_du = len(_qp_du)
_n_fifo_du = len(_fifo_du)
_n_unserved_du = len(_unserved_du)
_n_sim_du = _n_qp_du + _n_fifo_du + _n_unserved_du

# E[Not Recovered] = Σ churn_ij_corrected (QP) + Σ churn_ij_corrected (FIFO) + Σ churn_i_corrected (unserved)
# Apply calibration bias correction: churn_ij_corrected = (churn_ij - CALIBRATED_CHURN_BIAS).clip(0, 1)
_e_churn_qp = (_qp_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).sum()
_e_churn_fifo = ((_fifo_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).sum()) if _n_fifo_du > 0 else 0.0
_e_churn_unserved = (_unserved_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).sum()  # churn_i - bias
_e_churn_total = _e_churn_qp + _e_churn_fifo + _e_churn_unserved
_sim_churn_rate = _e_churn_total / _n_sim_du if _n_sim_du > 0 else 0

# Rates per segment (calibrated)
_rate_qp = (_qp_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).mean() if _n_qp_du > 0 else 0
_rate_fifo = (_fifo_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).mean() if _n_fifo_du > 0 else 0
_rate_unserved = (_unserved_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).mean() if _n_unserved_du > 0 else 0

# Actual ground truth for same call_ids
_actual_qp = int((_qp_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum())
_actual_fifo = int((_fifo_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum()) if _n_fifo_du > 0 else 0
_actual_unserved = int((_unserved_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum())
_actual_total = _actual_qp + _actual_fifo + _actual_unserved
_actual_churn_rate = _actual_total / _n_sim_du if _n_sim_du > 0 else 0

# --- Historical: ALL QPlanner calls (not just AgentAvailable) ---
_hist_all = df_calls_qplanner.drop_duplicates(subset='CALL_ID')
_hist_du = _hist_all[_hist_all['RESOL_MOT_DSC'].isin(VALID_OUTCOMES)]
_hist_n_du = len(_hist_du)
_hist_n_churned = int((_hist_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum())
_hist_churn_rate = _hist_n_churned / _hist_n_du if _hist_n_du > 0 else 0

# Historical breakdown: assigned vs not-assigned
_hist_assigned_du = _hist_du[_hist_du['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable']
_hist_notassigned_du = _hist_du[_hist_du['QUEUE_PLANNER_RESULT_DESC'] != 'QPlanner-AgentAvailable']
_hist_assigned_churned = int((_hist_assigned_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum())
_hist_notassigned_churned = int((_hist_notassigned_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum())

# --- Print results ---
print(f"\n  Histórico (todo o universo QPlanner, 'decisor útil'):")
print(f"    Total QP calls:       {len(_hist_all):,}")
print(f"    Decisor útil:         {_hist_n_du:,}")
print(f"      ├─ Assigned (QP-AA):  {len(_hist_assigned_du):,}  →  N Recup: {_hist_assigned_churned:,} "
      f"({_hist_assigned_churned/len(_hist_assigned_du):.1%})")
print(f"      └─ Not assigned:      {len(_hist_notassigned_du):,}  →  N Recup: {_hist_notassigned_churned:,} "
      f"({_hist_notassigned_churned/len(_hist_notassigned_du):.1%})" if len(_hist_notassigned_du) > 0
      else f"      └─ Not assigned:      0")
print(f"    Not Recovered total: {_hist_n_churned:,} ({_hist_churn_rate:.1%})")

print(f"\n  Simulado (universo completo: QP + FIFO counterfact. + unserved):")
print(f"    QP assigned (served):      {_n_sim_qp:,}")
print(f"    FIFO routed (counterfact.):{_n_sim_fifo:,}  ⚠️ non-blocking, churn estimation only")
print(f"    Truly unserved:            {_n_sim_unserved:,}")
print(f"    Total:                     {_n_sim_total:,}")
print(f"    Decisor útil:              {_n_sim_du:,} (QP: {_n_qp_du:,} + FIFO: {_n_fifo_du:,} + Unserved: {_n_unserved_du:,})")
print(f"\n  E[Not Recovered] (modelo):")
print(f"    QP assigned (churn_ij):   {_e_churn_qp:.0f} (taxa: {_rate_qp:.1%} — optimised)")
print(f"    FIFO routed (churn_ij):   {_e_churn_fifo:.0f} (taxa: {_rate_fifo:.1%} — BAU random, counterfact.)")
print(f"    Unserved (churn_i):       {_e_churn_unserved:.0f} (taxa: {_rate_unserved:.1%} — sem operador)")
print(f"    TOTAL:                    {_e_churn_total:.0f} ({_sim_churn_rate:.1%})")

_churn_delta = abs(_sim_churn_rate - _actual_churn_rate) * 100
print(f"\n  Validação cruzada (mesmos call_ids):")
print(f"    Actual N Recup:        {_actual_total:,} ({_actual_churn_rate:.1%})")
print(f"    Modelo ↔ Actual gap:   {_churn_delta:.1f} pp")

_delta_vs_hist = (_sim_churn_rate - _hist_churn_rate) * 100
print(f"\n  Δ Simulado vs Histórico: {_delta_vs_hist:+.1f} pp")

# --- Per-day breakdown ---
_day_comparison = []
for day_str, day_state in sorted(all_day_states.items()):
    day_date = pd.to_datetime(day_str).date()

    # Historical: ALL QPlanner calls for this day, decisor útil
    _h_day = df_calls[
        (pd.to_datetime(df_calls['START_DATE_TIME']).dt.date == day_date) &
        (df_calls['QUEUE_PLANNER_STATUS_DESC'] == 'On') &
        (df_calls['QUEUE_PLANNER_RESULT_DESC'].isin([
            'QPlanner-AgentAvailable', 'QPlanner-NoAgentReturned', 'QPlanner-AgentNotAvailable'
        ])) &
        (df_calls['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
    ].drop_duplicates(subset='CALL_ID')
    _h_n = len(_h_day)
    _h_churned = int((_h_day['RESOL_MOT_DSC'] == 'N RECUPERADO').sum()) if _h_n > 0 else 0
    _h_rate = _h_churned / _h_n if _h_n > 0 else 0

    # Simulated — QP assigned
    _s_qp_df = pd.DataFrame([
        {'call_id': a.call_id, 'churn_ij': a.churn_ij, '_type': 'qp'}
        for a in day_state.assignments
    ]) if day_state.assignments else pd.DataFrame(columns=['call_id', 'churn_ij', '_type'])

    # Simulated — FIFO routed (counterfactual)
    _s_fifo_df = pd.DataFrame([
        {'call_id': a.call_id, 'churn_ij': a.churn_ij, '_type': 'fifo'}
        for a in day_state.fifo_assignments
    ]) if day_state.fifo_assignments else pd.DataFrame(columns=['call_id', 'churn_ij', '_type'])

    # Simulated — UNSERVED
    _s_unserved_df = pd.DataFrame([
        {'call_id': u['call_id'], 'churn_ij': churn_i_lookup.get(u['call_id'], np.nan), '_type': 'unserved'}
        for u in day_state.unserved_calls
    ]) if day_state.unserved_calls else pd.DataFrame(columns=['call_id', 'churn_ij', '_type'])

    # Combine all segments
    _s_all_df = pd.concat([_s_qp_df, _s_fifo_df, _s_unserved_df], ignore_index=True)

    if not _s_all_df.empty:
        _s_all_df = _s_all_df.merge(
            df_calls[['CALL_ID', 'RESOL_MOT_DSC']].drop_duplicates(subset='CALL_ID'),
            left_on='call_id', right_on='CALL_ID', how='left'
        )
        _s_du = _s_all_df[
            (_s_all_df['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
            & (_s_all_df['churn_ij'].notna())
        ]
        _s_n = len(_s_du)
        _s_n_qp = len(_s_du[_s_du['_type'] == 'qp'])
        _s_n_fifo = len(_s_du[_s_du['_type'] == 'fifo'])
        _s_n_unserved = len(_s_du[_s_du['_type'] == 'unserved'])
        _s_expected = (_s_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).sum() if _s_n > 0 else 0
        _s_rate = (_s_du['churn_ij'] - CALIBRATED_CHURN_BIAS).clip(0, 1).mean() if _s_n > 0 else 0
    else:
        _s_n = _s_n_qp = _s_n_fifo = _s_n_unserved = 0
        _s_expected = _s_rate = 0

    _day_comparison.append({
        'day': day_str,
        'hist_du': _h_n,
        'hist_churned': _h_churned,
        'hist_rate': _h_rate,
        'sim_du': _s_n,
        'sim_qp_du': _s_n_qp,
        'sim_fifo_du': _s_n_fifo,
        'sim_unserved_du': _s_n_unserved,
        'sim_expected': _s_expected,
        'sim_rate': _s_rate,
    })

_df_day_churn = pd.DataFrame(_day_comparison)

# --- FIGURE: 3-panel churn validation (full universe) ---
fig2, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5.5),
    gridspec_kw={'width_ratios': [1, 1.3, 1]})
fig2.suptitle(
    "Churn Outcome Estimation — 'Not Recovered' (universo completo: QP + FIFO counterfact. + unserved)",
    fontsize=13, fontweight='bold'
)

# Panel 1: churn probability distribution — QP vs FIFO vs unserved
_bins_churn = np.linspace(0, 1, 51)
ax1.hist(_qp_du['churn_ij'].dropna(), bins=_bins_churn, color='#4CAF50', alpha=0.6,
         edgecolor='white', linewidth=0.5, label=f'QP churn_ij (n={_n_qp_du:,})')
if _n_fifo_du > 0:
    ax1.hist(_fifo_du['churn_ij'].dropna(), bins=_bins_churn, color='#2196F3', alpha=0.5,
             edgecolor='white', linewidth=0.5, label=f'FIFO churn_ij (n={_n_fifo_du:,})')
if _n_unserved_du > 0:
    ax1.hist(_unserved_du['churn_ij'].dropna(), bins=_bins_churn, color='#FF9800', alpha=0.4,
             edgecolor='white', linewidth=0.5, label=f'Unserved churn_i (n={_n_unserved_du:,})')
ax1.axvline(_rate_qp, color='#4CAF50', ls='--', lw=2, label=f'Mean QP = {_rate_qp:.3f}')
if _n_fifo_du > 0:
    ax1.axvline(_rate_fifo, color='#2196F3', ls='--', lw=2, label=f'Mean FIFO = {_rate_fifo:.3f}')
if _n_unserved_du > 0:
    ax1.axvline(_rate_unserved, color='#FF9800', ls='--', lw=2, label=f'Mean unserved = {_rate_unserved:.3f}')
ax1.axvline(_sim_churn_rate, color='#333', ls='-', lw=2, label=f'Total rate = {_sim_churn_rate:.3f}')
ax1.set_xlabel('P(Not Recovered)')
ax1.set_ylabel('Nº chamadas')
ax1.set_title('① Distribuição P(churn)\n(QP: churn_ij | FIFO: churn_ij | Unserved: churn_i)')
ax1.legend(fontsize=7)

# Panel 2: Per-day comparison — bars: historical vs simulated
x_pos = np.arange(len(_df_day_churn))
_bar_width = 0.35

ax2.bar(x_pos - _bar_width/2, _df_day_churn['hist_churned'], _bar_width,
        label='Histórico (Não Recup.)', color='steelblue', alpha=0.8)
ax2.bar(x_pos + _bar_width/2, _df_day_churn['sim_expected'], _bar_width,
        label='Simulado E[Não Recup.]', color='coral', alpha=0.8)
ax2.set_xlabel('Dia')
ax2.set_ylabel('Contagem E[Not Recovered]')
_day_labels = [d.split('-')[-1] for d in _df_day_churn['day']]
ax2.set_xticks(x_pos)
ax2.set_xticklabels(_day_labels, fontsize=8)

ax2_twin = ax2.twinx()
ax2_twin.plot(x_pos, _df_day_churn['hist_rate'] * 100, 'o-', color='steelblue',
              ms=4, lw=1.5, label='Hist %', alpha=0.8)
ax2_twin.plot(x_pos, _df_day_churn['sim_rate'] * 100, 's--', color='coral',
              ms=4, lw=1.5, label='Sim %', alpha=0.8)
ax2_twin.set_ylabel('Taxa Churn (%)')
_lines1, _labels1 = ax2.get_legend_handles_labels()
_lines2, _labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(_lines1 + _lines2, _labels1 + _labels2, fontsize=7, loc='upper left')
ax2.set_title('② Not Recovered por Dia\n(histórico vs simulado — universo completo)')

# Panel 3: Aggregate summary table
ax3.axis('off')
_summary_data = [
    ['Universo (DU)', f'{_hist_n_du:,}', f'{_n_sim_du:,}'],
    ['  ├─ QP Assigned', f'{len(_hist_assigned_du):,}', f'{_n_qp_du:,}'],
    ['  ├─ FIFO Counterfact. ¹', '—', f'{_n_fifo_du:,}'],
    ['  └─ Truly Unserved', f'{len(_hist_notassigned_du):,}', f'{_n_unserved_du:,}'],
    ['', '', ''],
    ['E[Não Recup.] (#)', f'{_hist_n_churned:,}', f'{_e_churn_total:.0f}'],
    ['  ├─ QP (optimised)', f'{_hist_assigned_churned:,}', f'{_e_churn_qp:.0f}'],
    ['  ├─ FIFO (BAU) ¹', '—', f'{_e_churn_fifo:.0f}'],
    ['  └─ Unserved', f'{_hist_notassigned_churned:,}', f'{_e_churn_unserved:.0f}'],
    ['', '', ''],
    ['Taxa Churn (%)', f'{_hist_churn_rate:.1%}', f'{_sim_churn_rate:.1%}'],
    ['Δ vs Histórico', '—', f'{_delta_vs_hist:+.1f} pp'],
    ['', '', ''],
    ['Validação cruzada:', '', ''],
    ['Actual (mesmas calls)', '—', f'{_actual_total:,} ({_actual_churn_rate:.1%})'],
    ['Modelo ↔ Actual', '—', f'{_churn_delta:.1f} pp gap'],
    ['', '', ''],
    ['¹ Non-blocking: churn', 'estimation only,', 'ops NOT held busy'],
]
_tbl = ax3.table(
    cellText=_summary_data,
    colLabels=['Métrica', 'Histórico', 'Simulado'],
    loc='center', cellLoc='center'
)
_tbl.auto_set_font_size(False)
_tbl.set_fontsize(9)
_tbl.scale(1.0, 1.35)
for j in range(3):
    _tbl[0, j].set_facecolor('#e3f2fd')
    _tbl[0, j].set_text_props(fontweight='bold')
# Color delta row
_delta_pp = abs(_delta_vs_hist)
_delta_color = '#c8e6c9' if _delta_pp < 2 else '#fff9c4' if _delta_pp < 5 else '#ffcdd2'
_tbl[12, 2].set_facecolor(_delta_color)
# Color model gap row
_gap_color = '#c8e6c9' if _churn_delta < 2 else '#fff9c4' if _churn_delta < 5 else '#ffcdd2'
_tbl[16, 2].set_facecolor(_gap_color)
# Style footnote row
_fn_row = len(_summary_data)
for j in range(3):
    _tbl[_fn_row, j].set_facecolor('#f5f5f5')
    _tbl[_fn_row, j].set_text_props(fontstyle='italic', fontsize=8)
ax3.set_title('③ Resumo Agregado\n(Histórico vs Simulado — universo completo)', pad=20)

plt.tight_layout()
plt.show()

# --- Print per-day table ---
print(f"\n  Per-day breakdown (Not Recovered — universo completo: QP + FIFO counterfact. + unserved):")
print(f"  {'Day':>12s}  {'Hist DU':>8s}  {'H.Churn':>8s}  {'H.Rate':>8s}  "
      f"{'Sim DU':>8s}  {'(QP+F+U)':>10s}  {'E[Churn]':>8s}  {'S.Rate':>8s}  {'Δ pp':>6s}")
print(f"  {'─'*12}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*8}  {'─'*6}")
for _, r in _df_day_churn.iterrows():
    _d_pp = (r['sim_rate'] - r['hist_rate']) * 100
    _qfu = f"{int(r['sim_qp_du'])}+{int(r['sim_fifo_du'])}+{int(r['sim_unserved_du'])}"
    print(f"  {r['day']:>12s}  {r['hist_du']:>8,}  {r['hist_churned']:>8,}  {r['hist_rate']:>8.1%}  "
          f"{r['sim_du']:>8,}  {_qfu:>10s}  {r['sim_expected']:>8.0f}  {r['sim_rate']:>8.1%}  {_d_pp:>+6.1f}")
print(f"  {'─'*12}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*8}  {'─'*6}")
print(f"  {'TOTAL':>12s}  {_hist_n_du:>8,}  {_hist_n_churned:>8,}  {_hist_churn_rate:>8.1%}  "
      f"{_n_sim_du:>8,}  {f'{_n_qp_du}+{_n_fifo_du}+{_n_unserved_du}':>10s}  {_e_churn_total:>8.0f}  "
      f"{_sim_churn_rate:>8.1%}  {_delta_vs_hist:>+6.1f}")


# PART C — RESULT DISTRIBUTION & WAIT TIME BY OUTCOME (QP vs FIFO vs Unserved)
# ═══════════════════════════════════════════════════════════════════════════════

# --- Compute distributions ---
# Historical: from df_calls_qplanner
_hist_qp = df_calls_qplanner.drop_duplicates(subset='CALL_ID')
_hist_result_counts = _hist_qp['QUEUE_PLANNER_RESULT_DESC'].value_counts()

# Historical wait times by outcome
_hist_assigned_wt = _hist_qp[
    _hist_qp['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable'
]['WAIT_TIME_IN_QUEUE_SEC_QTY'].dropna().values
_hist_unserved_wt = _hist_qp[
    _hist_qp['QUEUE_PLANNER_RESULT_DESC'] != 'QPlanner-AgentAvailable'
]['WAIT_TIME_IN_QUEUE_SEC_QTY'].dropna().values

# Simulated: QP / FIFO / Unserved
_sim_qp_wt = np.array([a.waiting_time for a in all_assignments])
_sim_fifo_wt = np.array([a.waiting_time for a in all_fifo_assignments]) if all_fifo_assignments else np.array([])

_sim_unserved_list = []
for _ds2, _dstate2 in all_day_states.items():
    _sim_unserved_list.extend(_dstate2.unserved_calls)

# Unserved breakdown by reason
_sim_n_fifo_abandoned = sum(1 for u in _sim_unserved_list if u.get('reason') == 'fifo_abandoned')
_sim_n_fifo_eod = sum(1 for u in _sim_unserved_list if u.get('reason') == 'fifo_end_of_day')
_sim_n_gap_cleared = sum(1 for u in _sim_unserved_list if u.get('reason') == 'gap_cleared')
_sim_n_expired_only = sum(1 for u in _sim_unserved_list if u.get('reason') == 'expired')
_sim_unserved_wt = np.array([u['wait_time'] for u in _sim_unserved_list if 'wait_time' in u])

# --- FIGURE 3: 2×2 — Result distribution + wait time breakdown ---
fig3, axes3 = plt.subplots(2, 2, figsize=(16, 11))
fig3.suptitle(
    'QP Result Distribution & Wait Time — Historical vs Simulated\n'
    '(FIFO = non-blocking counterfactual for churn estimation; operators NOT held busy)',
    fontsize=12, fontweight='bold'
)
_sim_unserved_wt = np.array([u['wait_time'] for u in _sim_unserved_list if 'wait_time' in u])
# ── Row 1: Pie charts ────────────────────────────────────────────────────────

# 1a. Historical pie
_pie_colors = {'QPlanner-AgentAvailable': '#4CAF50', 'QPlanner-NoAgentReturned': '#F44336',
               'QPlanner-AgentNotAvailable': '#FF9800'}
_hist_labels = list(_hist_result_counts.index)
_hist_sizes = list(_hist_result_counts.values)
_hist_colors = [_pie_colors.get(l, '#9E9E9E') for l in _hist_labels]

axes3[0, 0].pie(_hist_sizes, labels=_hist_labels, colors=_hist_colors,
                autopct='%1.1f%%', startangle=90, pctdistance=0.6,
                explode=[0.03] * len(_hist_sizes),
                textprops={'fontsize': 9})
axes3[0, 0].set_title(f'Histórico — QP Result Distribution\n(n={sum(_hist_sizes):,} chamadas)')

# 1b. Simulated pie (QP / FIFO counterfactual / Unserved breakdown)
_sim_pie_labels = [
    'QP Assigned (served)',
    'FIFO Counterfact. ¹',
    'FIFO Abandoned',
    'FIFO End-of-Day',
    'Gap Cleared',
    'Expired',
]
_sim_pie_sizes = [len(all_assignments), len(all_fifo_assignments),
                  _sim_n_fifo_abandoned, _sim_n_fifo_eod, _sim_n_gap_cleared, _sim_n_expired_only]
_sim_pie_colors = ['#4CAF50', '#2196F3', '#F44336', '#E91E63', '#FF9800', '#9E9E9E']
# Remove zero slices
_sim_filtered = [(l, s, c) for l, s, c in zip(_sim_pie_labels, _sim_pie_sizes, _sim_pie_colors) if s > 0]
if _sim_filtered:
    _sl, _ss, _sc = zip(*_sim_filtered)
else:
    _sl, _ss, _sc = _sim_pie_labels[:1], [1], _sim_pie_colors[:1]

axes3[0, 1].pie(_ss, labels=_sl, colors=_sc,
                autopct='%1.1f%%', startangle=90, pctdistance=0.6,
                explode=[0.03] * len(_ss),
                textprops={'fontsize': 9})
axes3[0, 1].set_title(
    f'Simulado — Result Distribution\n'
    f'(n={sum(_sim_pie_sizes):,} — ¹ non-blocking, churn estimation only)'
)

# ── Row 2: Wait time histograms ─────────────────────────────────────────────
_wt_bins = np.arange(0, 650, 15)

# 2a. Historical wait times
axes3[1, 0].hist(_hist_assigned_wt, bins=_wt_bins, alpha=0.65, density=True,
                 color='#4CAF50', edgecolor='white', linewidth=0.5,
                 label=f'Assigned (n={len(_hist_assigned_wt):,})')
axes3[1, 0].hist(_hist_unserved_wt, bins=_wt_bins, alpha=0.55, density=True,
                 color='#F44336', edgecolor='white', linewidth=0.5,
                 label=f'Unserved (n={len(_hist_unserved_wt):,})')
axes3[1, 0].axvline(55, color='orange', ls='--', lw=2, alpha=0.8, label='~55 s callback deflection')
axes3[1, 0].axvline(180, color='purple', ls='--', lw=2, alpha=0.8, label='~180 s BAU timeout')
axes3[1, 0].set_xlabel('Wait Time in Queue (seconds)')
axes3[1, 0].set_ylabel('Density')
axes3[1, 0].set_title('Histórico — Wait Time: Assigned vs Unserved')
axes3[1, 0].legend(fontsize=8)

# 2b. Simulated wait times (QP / FIFO / Unserved)
axes3[1, 1].hist(_sim_qp_wt, bins=_wt_bins, alpha=0.65, density=True,
                 color='#4CAF50', edgecolor='white', linewidth=0.5,
                 label=f'QP Assigned (n={len(_sim_qp_wt):,})')
if len(_sim_fifo_wt) > 0:
    axes3[1, 1].hist(_sim_fifo_wt, bins=_wt_bins, alpha=0.55, density=True,
                     color='#2196F3', edgecolor='white', linewidth=0.5,
                     label=f'FIFO Counterfact. (n={len(_sim_fifo_wt):,})')
if len(_sim_unserved_wt) > 0:
    axes3[1, 1].hist(_sim_unserved_wt[_sim_unserved_wt < 650], bins=_wt_bins, alpha=0.4, density=True,
                     color='#F44336', edgecolor='white', linewidth=0.5,
                     label=f'Unserved (n={len(_sim_unserved_wt):,})')
axes3[1, 1].axvline(180, color='purple', ls='--', lw=2, alpha=0.8, label='180 s QP expiry')
axes3[1, 1].axvline(600, color='red', ls='--', lw=2, alpha=0.6, label='600 s FIFO abandon')
axes3[1, 1].set_xlabel('Wait Time in Queue (seconds)')
axes3[1, 1].set_ylabel('Density')
axes3[1, 1].set_title('Simulado — Wait Time: QP vs FIFO counterfact. vs Unserved')
axes3[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary stats
print(f"\n📊 Result Distribution Comparison:")
print(f"  ⚠️  FIFO is non-blocking (counterfactual): operators NOT held busy.")
print(f"      Only QP Assigned are real served calls comparable to historical.\n")
print(f"  {'Category':<30s}  {'Hist':>8s}  {'%':>6s}  {'Sim':>8s}  {'%':>6s}")
print(f"  {'─'*30}  {'─'*8}  {'─'*6}  {'─'*8}  {'─'*6}")
_hist_total = sum(_hist_sizes)
_sim_total = sum(_sim_pie_sizes)
print(f"  {'QP Assigned (served)':<30s}  {_hist_result_counts.get('QPlanner-AgentAvailable', 0):>8,}  "
      f"{_hist_result_counts.get('QPlanner-AgentAvailable', 0)/_hist_total:>6.1%}  "
      f"{len(all_assignments):>8,}  {len(all_assignments)/_sim_total:>6.1%}")
print(f"  {'FIFO Counterfact. ¹':<30s}  {'—':>8s}  {'—':>6s}  "
      f"{len(all_fifo_assignments):>8,}  {len(all_fifo_assignments)/_sim_total:>6.1%}")
_hist_unserved_n = _hist_total - _hist_result_counts.get('QPlanner-AgentAvailable', 0)
_sim_unserved_n = len(_sim_unserved_list)
print(f"  {'Unserved (total)':<30s}  {_hist_unserved_n:>8,}  "
      f"{_hist_unserved_n/_hist_total:>6.1%}  "
      f"{_sim_unserved_n:>8,}  {_sim_unserved_n/_sim_total:>6.1%}")
if _sim_n_fifo_abandoned > 0:
    print(f"    {'→ fifo_abandoned':<28s}  {'':>8s}  {'':>6s}  "
          f"{_sim_n_fifo_abandoned:>8,}  {_sim_n_fifo_abandoned/_sim_total:>6.1%}")
if _sim_n_fifo_eod > 0:
    print(f"    {'→ fifo_end_of_day':<28s}  {'':>8s}  {'':>6s}  "
          f"{_sim_n_fifo_eod:>8,}  {_sim_n_fifo_eod/_sim_total:>6.1%}")
if _sim_n_gap_cleared > 0:
    print(f"    {'→ gap_cleared':<28s}  {'':>8s}  {'':>6s}  "
          f"{_sim_n_gap_cleared:>8,}  {_sim_n_gap_cleared/_sim_total:>6.1%}")
if _sim_n_expired_only > 0:
    print(f"    {'→ expired (no FIFO)':<28s}  {'':>8s}  {'':>6s}  "
          f"{_sim_n_expired_only:>8,}  {_sim_n_expired_only/_sim_total:>6.1%}")
print(f"  {'─'*30}  {'─'*8}  {'─'*6}  {'─'*8}  {'─'*6}")
print(f"  {'TOTAL':<30s}  {_hist_total:>8,}  {'100%':>6s}  {_sim_total:>8,}  {'100%':>6s}")
print(f"\n  ¹ Non-blocking: churn estimation only, operators NOT held busy.")

print(f"\n📊 Wait Time by Outcome:")
print(f"  {'Segment':<25s}  {'Mean':>8s}  {'Median':>8s}  {'P95':>8s}  {'n':>8s}")
print(f"  {'─'*25}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}")
print(f"  {'QP Assigned (served)':<25s}  {np.mean(_sim_qp_wt):>8.0f}  {np.median(_sim_qp_wt):>8.0f}  "
      f"{np.percentile(_sim_qp_wt, 95):>8.0f}  {len(_sim_qp_wt):>8,}")
if len(_sim_fifo_wt) > 0:
    print(f"  {'FIFO Counterfact. ¹':<25s}  {np.mean(_sim_fifo_wt):>8.0f}  {np.median(_sim_fifo_wt):>8.0f}  "
          f"{np.percentile(_sim_fifo_wt, 95):>8.0f}  {len(_sim_fifo_wt):>8,}")
if len(_sim_unserved_wt) > 0:
    print(f"  {'Unserved':<25s}  {np.mean(_sim_unserved_wt):>8.0f}  {np.median(_sim_unserved_wt):>8.0f}  "
          f"{np.percentile(_sim_unserved_wt, 95):>8.0f}  {len(_sim_unserved_wt):>8,}")


<a id="churn-validation"></a>
### Churn Calibration Validation (Post-Simulation)

Gap decomposition and isotonic regression validation — requires `all_day_states` from Cell 13b.
Set `_RUN_CHURN_CALIBRATION = True` (in the churn calibration cell above) to execute.

In [ ]:
# =============================================================================
# CHURN CALIBRATION — Gap Decomposition + Isotonic Regression (post-simulation)
# =============================================================================
# ⚠️ CALIBRATION CELL — Run once for validation of additive correction.
#   Set _RUN_CHURN_CALIBRATION = True (in the churn calibration cell) to run.
#   Compares raw vs calibrated gap: if gap ≈ 0 after correction, the additive
#   bias fix is sufficient and no isotonic regression is needed downstream.
#   Requires: all_day_states from Cell 13b (simulation must have run first).
# =============================================================================
# NOTE: The CALIBRATED_CHURN_BIAS additive correction (used in Cell 36)
# is preferred over isotonic regression for production use because it is
# simpler, interpretable, and the iso_reg object is NOT used downstream.
# =============================================================================

if not _RUN_CHURN_CALIBRATION:
    print("⏭️  Isotonic regression cell skipped (exploratory, not used downstream)")
    print("   Set _RUN_CHURN_CALIBRATION = True in Cell 43 to re-run.")
    print(f"   Production correction uses additive CALIBRATED_CHURN_BIAS = {CALIBRATED_CHURN_BIAS}")

# ─── Isotonic regression code below only runs when _RUN_CHURN_CALIBRATION = True ───
if _RUN_CHURN_CALIBRATION:
    from sklearn.isotonic import IsotonicRegression

    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 1: Fit isotonic regression on historical QP-assigned pairs
    # ═══════════════════════════════════════════════════════════════════════════
    _hist_assigned = df_calls_qplanner[
        df_calls_qplanner['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable'
    ].drop_duplicates(subset='CALL_ID')
    _hist_assigned = _hist_assigned[_hist_assigned['RESOL_MOT_DSC'].isin(VALID_OUTCOMES)].copy()

    _train_records = []
    for _, row in _hist_assigned.iterrows():
        cid = row['CALL_ID']
        op = row['agent_username']
        actual = 1 if row['RESOL_MOT_DSC'] == 'N RECUPERADO' else 0
        pred = full_score_lookup.get((cid, op))
        if pred is not None:
            _train_records.append({'pred': pred, 'actual': actual})

    _df_train = pd.DataFrame(_train_records)
    print(f"Isotonic regression training set: {len(_df_train):,} pairs")
    print(f"  Raw mean pred:   {_df_train['pred'].mean():.4f}")
    print(f"  Actual mean:     {_df_train['actual'].mean():.4f}")
    print(f"  Bias:            {(_df_train['pred'].mean() - _df_train['actual'].mean())*100:+.2f} pp")

    # Fit isotonic regression: maps predicted P(churn) → calibrated P(churn)
    iso_reg = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
    iso_reg.fit(_df_train['pred'].values, _df_train['actual'].values)

    # Verify on training data
    _df_train['calibrated'] = iso_reg.predict(_df_train['pred'].values)
    print(f"\n  After isotonic calibration (on training set):")
    print(f"    Calibrated mean: {_df_train['calibrated'].mean():.4f}")
    print(f"    Residual bias:   {(_df_train['calibrated'].mean() - _df_train['actual'].mean())*100:+.2f} pp")

    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 2: Apply calibration to ALL sim assignments (QP + FIFO + unserved)
    # ═══════════════════════════════════════════════════════════════════════════
    _sim_records_full = []
    for _ds, _dstate in sorted(all_day_states.items()):
        for a in _dstate.assignments:
            _sim_records_full.append({
                'call_id': a.call_id, 'churn_ij_raw': a.churn_ij,
                'segment': 'QP', 'day': _ds
            })
        for a in _dstate.fifo_assignments:
            _sim_records_full.append({
                'call_id': a.call_id, 'churn_ij_raw': a.churn_ij,
                'segment': 'FIFO', 'day': _ds
            })
        for u in _dstate.unserved_calls:
            ci = churn_i_lookup.get(u['call_id'])
            if ci is not None:
                _sim_records_full.append({
                    'call_id': u['call_id'], 'churn_ij_raw': ci,
                    'segment': 'Unserved', 'day': _ds
                })

    _df_sim_full = pd.DataFrame(_sim_records_full)

    # Join with actual outcomes
    _df_sim_full = _df_sim_full.merge(
        df_calls[['CALL_ID', 'RESOL_MOT_DSC']].drop_duplicates(subset='CALL_ID'),
        left_on='call_id', right_on='CALL_ID', how='left'
    )

    # Filter to decisor útil
    _df_sim_du = _df_sim_full[
        (_df_sim_full['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
        & (_df_sim_full['churn_ij_raw'].notna())
    ].copy()

    # Apply isotonic calibration
    _df_sim_du['churn_ij_calibrated'] = iso_reg.predict(_df_sim_du['churn_ij_raw'].values)
    _df_sim_du['actual'] = (_df_sim_du['RESOL_MOT_DSC'] == 'N RECUPERADO').astype(int)

    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 3: Compare raw vs calibrated vs historical
    # ═══════════════════════════════════════════════════════════════════════════
    _hist_all = df_calls_qplanner.drop_duplicates(subset='CALL_ID')
    _hist_du = _hist_all[_hist_all['RESOL_MOT_DSC'].isin(VALID_OUTCOMES)]
    _hist_churned = int((_hist_du['RESOL_MOT_DSC'] == 'N RECUPERADO').sum())
    _hist_rate = _hist_churned / len(_hist_du)

    _n_total = len(_df_sim_du)
    _raw_total = _df_sim_du['churn_ij_raw'].sum()
    _cal_total = _df_sim_du['churn_ij_calibrated'].sum()
    _actual_total = _df_sim_du['actual'].sum()

    _raw_rate = _df_sim_du['churn_ij_raw'].mean()
    _cal_rate = _df_sim_du['churn_ij_calibrated'].mean()
    _actual_rate_sim = _df_sim_du['actual'].mean()

    print(f"\n{'='*80}")
    print(f"📊 CHURN CALIBRATION RESULTS — Isotonic Regression")
    print(f"{'='*80}")
    print(f"\n  Full sim universe (DU only, n={_n_total:,}):")
    print(f"  {'':30s} {'Rate':>8s}  {'E[churn]':>10s}  {'Gap vs Hist':>12s}")
    print(f"  {'─'*65}")
    print(f"  {'Historical (actual)':30s} {_hist_rate*100:7.2f}%  {_hist_churned:>10,}  {'(reference)':>12s}")
    print(f"  {'Sim actual (same calls)':30s} {_actual_rate_sim*100:7.2f}%  {_actual_total:>10,}  {(_actual_rate_sim - _hist_rate)*100:+7.2f} pp")
    print(f"  {'─'*65}")
    print(f"  {'Sim RAW (model pred)':30s} {_raw_rate*100:7.2f}%  {_raw_total:>10,.0f}  {(_raw_rate - _hist_rate)*100:+7.2f} pp")
    print(f"  {'Sim CALIBRATED (isotonic)':30s} {_cal_rate*100:7.2f}%  {_cal_total:>10,.0f}  {(_cal_rate - _hist_rate)*100:+7.2f} pp")

    # Per-segment breakdown
    print(f"\n  Per-segment breakdown:")
    print(f"  {'Segment':12s} {'n':>6s}  {'Raw':>8s}  {'Calibrated':>10s}  {'Actual':>8s}  {'Bias Raw':>9s}  {'Bias Cal':>9s}")
    print(f"  {'─'*70}")
    for seg in ['QP', 'FIFO', 'Unserved']:
        _seg = _df_sim_du[_df_sim_du['segment'] == seg]
        if len(_seg) == 0:
            continue
        _seg_raw = _seg['churn_ij_raw'].mean()
        _seg_cal = _seg['churn_ij_calibrated'].mean()
        _seg_act = _seg['actual'].mean()
        _bias_raw = (_seg_raw - _seg_act) * 100
        _bias_cal = (_seg_cal - _seg_act) * 100
        print(f"  {seg:12s} {len(_seg):>6,}  {_seg_raw*100:7.2f}%  {_seg_cal*100:9.2f}%  {_seg_act*100:7.2f}%  {_bias_raw:+8.2f}pp  {_bias_cal:+8.2f}pp")

    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 4: Per-day comparison with calibrated values
    # ═══════════════════════════════════════════════════════════════════════════
    _day_cal = _df_sim_du.groupby('day').agg(
        n=('actual', 'count'),
        raw_churn=('churn_ij_raw', 'sum'),
        cal_churn=('churn_ij_calibrated', 'sum'),
        actual_churn=('actual', 'sum'),
        raw_rate=('churn_ij_raw', 'mean'),
        cal_rate=('churn_ij_calibrated', 'mean'),
        actual_rate=('actual', 'mean'),
    ).reset_index()

    # Historical per-day
    _hist_day_rates = []
    for day_str in sorted(all_day_states.keys()):
        day_date = pd.to_datetime(day_str).date()
        _h_day = df_calls[
            (pd.to_datetime(df_calls['START_DATE_TIME']).dt.date == day_date) &
            (df_calls['QUEUE_PLANNER_STATUS_DESC'] == 'On') &
            (df_calls['QUEUE_PLANNER_RESULT_DESC'].isin([
                'QPlanner-AgentAvailable', 'QPlanner-NoAgentReturned', 'QPlanner-AgentNotAvailable'
            ])) &
            (df_calls['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
        ].drop_duplicates(subset='CALL_ID')
        _h_n = len(_h_day)
        _h_churned = int((_h_day['RESOL_MOT_DSC'] == 'N RECUPERADO').sum()) if _h_n > 0 else 0
        _hist_day_rates.append({'day': day_str, 'hist_rate': _h_churned / _h_n if _h_n > 0 else 0, 'hist_n': _h_n})
    _df_hist_day = pd.DataFrame(_hist_day_rates)

    _day_cal = _day_cal.merge(_df_hist_day, on='day', how='left')

    # ═══════════════════════════════════════════════════════════════════════════
    # STEP 5: Visualization
    # ═══════════════════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    fig.suptitle('Churn Calibration Fix — Isotonic Regression', fontsize=13, fontweight='bold')

    # Panel 1: Calibration function
    ax1 = axes[0]
    _x_range = np.linspace(0, 1, 500)
    _y_cal = iso_reg.predict(_x_range)
    ax1.plot(_x_range, _y_cal, 'b-', linewidth=2.5, label='Isotonic calibration')
    ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Identity (no correction)')
    ax1.fill_between(_x_range, _x_range, _y_cal, alpha=0.15, color='blue')
    ax1.set_xlabel('Raw Model P(churn)')
    ax1.set_ylabel('Calibrated P(churn)')
    ax1.set_title('Isotonic Calibration Function')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)

    # Panel 2: Bar chart — raw vs calibrated vs historical (overall)
    ax2 = axes[1]
    _categories = ['Historical\n(actual)', 'Sim Actual\n(same calls)', 'Sim RAW\n(model)', 'Sim CALIBRATED\n(isotonic)']
    _values = [_hist_rate * 100, _actual_rate_sim * 100, _raw_rate * 100, _cal_rate * 100]
    _colors = ['#4CAF50', '#FF9800', '#F44336', '#2196F3']
    bars = ax2.bar(_categories, _values, color=_colors, alpha=0.8, edgecolor='white', linewidth=1.5)
    ax2.axhline(y=_hist_rate * 100, color='#4CAF50', ls='--', lw=1.5, alpha=0.7)
    for bar, val in zip(bars, _values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax2.set_ylabel('Churn Rate (%)')
    ax2.set_title('Overall Churn Rate Comparison')
    ax2.set_ylim(0, max(_values) * 1.15)
    ax2.grid(True, alpha=0.3, axis='y')

    # Panel 3: Per-day churn rates — historical vs raw vs calibrated
    ax3 = axes[2]
    _x = np.arange(len(_day_cal))
    ax3.plot(_x, _day_cal['hist_rate'] * 100, 'o-', color='#4CAF50', lw=2, ms=5, label='Historical', zorder=3)
    ax3.plot(_x, _day_cal['raw_rate'] * 100, 's-', color='#F44336', lw=1.5, ms=4, alpha=0.7, label='Sim raw')
    ax3.plot(_x, _day_cal['cal_rate'] * 100, '^-', color='#2196F3', lw=2, ms=5, label='Sim calibrated', zorder=2)
    ax3.set_xticks(_x[::3])
    ax3.set_xticklabels([pd.to_datetime(d).strftime('%m-%d') for d in _day_cal['day'].iloc[::3]],
                         rotation=45, fontsize=8)
    ax3.set_xlabel('Day')
    ax3.set_ylabel('Churn Rate (%)')
    ax3.set_title('Per-Day Churn: Historical vs Raw vs Calibrated')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary
    _gap_raw = (_raw_rate - _hist_rate) * 100
    _gap_cal = (_cal_rate - _hist_rate) * 100
    print(f"\n  ✅ SUMMARY:")
    print(f"     Raw gap:        {_gap_raw:+.2f} pp")
    print(f"     Calibrated gap: {_gap_cal:+.2f} pp")
    print(f"     Improvement:    {abs(_gap_raw) - abs(_gap_cal):.2f} pp reduction")
    print(f"     Residual is mix effect (sim assigns to inherently different calls)")

<a id="sim-results-viz"></a>
## Cell 16: Visualization — Simulation Results (4 Panels)

In [ ]:
# =============================================================================
# CELL 16 — VISUALIZATION: Simulation results (all 21 days, 4 panels)
# =============================================================================
# INPUT:
#   - df_all_sim: DataFrame of QP assignments (from cell 15)
#   - all_assignments: list of Assignment objects across all 21 days (from cell 13)
#   - all_day_states: dict day_str → SimulationState (from cell 13)
#   - CALIBRATED_TICK, CALIBRATED_LATENCY, CALIBRATED_LATENCY_STD
#
# DOES:
#   1. Extends df_all_sim with tick/timestamp columns (built in cell 15)
#   2. Creates a 2×2 figure with 4 panels:
#      - Waiting time histogram — distribution with mean line
#      - Waiting time over simulation period — scatter of wait vs timestamp
#      - Churn reduction distribution — histogram of (churn_i - churn_ij)
#      - FO score distribution — histogram of optimization scores
#
# OUTPUT:
#   - df_all_sim: enriched with tick + timestamp columns
#   - 4-panel matplotlib figure (displayed inline)
# =============================================================================

# Extend df_all_sim with tick/timestamp (needed for temporal scatter plot)
if 'tick' not in df_all_sim.columns:
    _tick_ts = pd.DataFrame([
        {'call_id': a.call_id, 'operator': a.operator, 'tick': a.tick, 'timestamp': a.timestamp}
        for a in all_assignments
    ])
    df_all_sim = df_all_sim.merge(_tick_ts, on=['call_id', 'operator'], how='left')

print(f"df_all_sim: {len(df_all_sim):,} assignments across {len(all_day_states)} days")

if not df_all_sim.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(
        f"Simulation Results — Baseline FO (all {len(all_day_states)} days, "
        f"n={len(df_all_sim):,})",
        fontsize=14, fontweight='bold'
    )

    # 1. Waiting time distribution
    axes[0, 0].hist(df_all_sim['waiting_time'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0, 0].axvline(df_all_sim['waiting_time'].mean(), color='red', linestyle='--',
                        label=f"mean={df_all_sim['waiting_time'].mean():.0f}s")
    axes[0, 0].axvline(df_all_sim['waiting_time'].median(), color='orange', linestyle=':',
                        label=f"median={df_all_sim['waiting_time'].median():.0f}s")
    axes[0, 0].set_xlabel('Waiting Time (s)')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Waiting Time Distribution')
    axes[0, 0].legend()

    # 2. Waiting time over simulation period
    axes[0, 1].scatter(df_all_sim['timestamp'], df_all_sim['waiting_time'], s=2, alpha=0.3, color='steelblue')
    axes[0, 1].set_xlabel('Date/Time')
    axes[0, 1].set_ylabel('Waiting Time (s)')
    axes[0, 1].set_title('Waiting Time Over Simulation Period')
    axes[0, 1].tick_params(axis='x', rotation=45)

    # 3. Churn reduction distribution
    axes[1, 0].hist(df_all_sim['churn_reduction'], bins=40, color='seagreen', edgecolor='white', alpha=0.8)
    axes[1, 0].axvline(df_all_sim['churn_reduction'].mean(), color='red', linestyle='--',
                        label=f"mean={df_all_sim['churn_reduction'].mean():.4f}")
    axes[1, 0].set_xlabel('Churn Reduction (churn_i - churn_ij)')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Churn Reduction per Assignment')
    axes[1, 0].legend()

    # 4. Optimization score distribution
    axes[1, 1].hist(df_all_sim['optimization_score'], bins=40, color='coral', edgecolor='white', alpha=0.8)
    axes[1, 1].set_xlabel('Optimization Score')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('FO Score Distribution')

    plt.tight_layout()
    plt.show()